# 4.2.1 Construct Minimal Leakage-Safe Forecasting Features

Notebook 4.1.2 established **what is forecast, when each forecast is issued, which future horizons are evaluated, which Taxi Zone × Metric sequences participate, and what historical information each modeling track is allowed to use**. This notebook now turns that experiment contract into the first forecasting predictor layer.

The scope is intentionally narrow. We begin with information from the target series' own history and deterministic calendar structure: recent sequence lags, comparable-period history, rolling summaries, seasonal/trend structure, and cyclical representations. Broader multimodal, spatial, weather, cluster, and other contextual predictors belong to later feature layers.

Features are constructed at the **forecast origin**, not separately for the 1-, 2-, and 5-step targets. At a given origin the forecaster knows the same historical information regardless of which future horizon is being predicted, so one origin-level feature vector can later be joined to all applicable horizon-specific targets from 4.1.2.

Any feature learned from mobility outcomes must respect the historical window of its modeling track. The Post-CP Primary cannot reach backward before January 5, 2025; the Full-History Regime-Aware track may use the longer history available before each origin; and the Pre-CP Reference must remain completely isolated from actual post-CP outcomes when it is ultimately carried forward as the no-CP counterfactual reference.

This distinction becomes especially important for decomposition and seasonality. **3.3.1 gives us a useful procedure, not forecasting answers that can simply be copied forward.** Its candidate-decomposition framework will be rebuilt here using only leakage-safe training windows. No retrospective winning seasonal models, fitted components, seasonal strengths, or residual values are reused.

**By the end of this notebook, the output will be a leakage-safe origin-level temporal forecasting feature layer together with metadata describing how each feature was constructed and whether it can be used directly or must be generated recursively during post-CP counterfactual application.**

---

## Notebook Setup

📝 **Intermediate-cache policy.** Several operations in this notebook work over millions of ordered observations, and the decomposition diagnostics require repeated STL/MSTL fits. Expensive blocks therefore write reusable artifacts to `pipeline_data/4.2.1.intermediate_tables/`. Leave a rebuild toggle `False` to reuse completed work; switch only the affected toggle to `True` when that feature family genuinely needs to be regenerated.

In [1]:
# 4.2.1.0 — Install required forecasting dependencies

%pip install -q -U statsmodels


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 4.2.1.0A — Define imports, globals, toggles, and paths

from pathlib import Path
import gc
import math
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL, MSTL
from tqdm.auto import tqdm

# `project_branding` in this environment provides the palette constants but
# does not expose `apply_branding`.  Keep the existing notebook API available
# with a small local compatibility implementation.
from project_branding import BRAND_COLORS, BRAND_COLOR_SEQUENCE

try:
    from project_branding import apply_branding
except ImportError:
    def apply_branding(fig, *args, **kwargs):
        """Apply the available project palette without requiring a helper export.

        The function is intentionally compatible with calls that pass extra
        positional or keyword arguments to the project helper.  It mutates
        and returns the Plotly figure, matching the usual Plotly helper
        convention used by the notebook.
        """
        if fig is None:
            return fig

        try:
            fig.update_layout(colorway=list(BRAND_COLOR_SEQUENCE))
        except Exception:
            pass

        # Preserve any explicitly supplied layout while providing a clean,
        # white plotting surface consistent with the project figures.
        try:
            fig.update_layout(
                plot_bgcolor="white",
                paper_bgcolor="white",
                font=dict(color="#222222"),
            )
            fig.update_xaxes(showgrid=True, gridcolor="#E5E7EB", zeroline=False)
            fig.update_yaxes(showgrid=True, gridcolor="#E5E7EB", zeroline=False)
        except Exception:
            pass
        return fig

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


# ---------------------------------------------------------------------
# Canonical forecasting contract
# ---------------------------------------------------------------------

CP_START_DATE = pd.Timestamp("2025-01-05")

FORECAST_TARGETS = [
    "taxi_trip_count", "taxi_avg_trip_speed",
    "fhvhv_trip_count", "fhvhv_avg_trip_speed",
    "subway_ridership",
]

FORECAST_HORIZONS = [1, 2, 5]
DAYPART_ORDER = ["overnight", "am_peak", "midday", "pm_peak", "evening"]

STEPS_PER_DAY = 5
STEPS_PER_WEEK = 35


# ---------------------------------------------------------------------
# Temporal-history feature settings
# ---------------------------------------------------------------------

RECENT_SEQUENCE_LAGS = [1, 2, 3, 4]
SAME_DAYPART_DAY_LAGS = [1, 2, 7]
SAME_DAYPART_ROLLING_WINDOWS = [7, 28]
ROLLING_MIN_OBS_SHARE = 0.80


# ---------------------------------------------------------------------
# Leakage-safe decomposition diagnostic settings
#
# These define which candidate structures are TESTED. They do not select
# any candidate for production. Selection happens only after HITL review.
# ---------------------------------------------------------------------

SEASONAL_CANDIDATES = pd.DataFrame([
    {"candidate_model": "none",       "primary_period": np.nan, "secondary_period": np.nan, "min_training_days": 42,  "complexity_units": 0},
    {"candidate_model": "single_7",   "primary_period": 7,      "secondary_period": np.nan, "min_training_days": 42,  "complexity_units": 1},
    {"candidate_model": "single_30",  "primary_period": 30,     "secondary_period": np.nan, "min_training_days": 90,  "complexity_units": 1},
    {"candidate_model": "single_365", "primary_period": 365,    "secondary_period": np.nan, "min_training_days": 731, "complexity_units": 1},
    {"candidate_model": "dual_7_365", "primary_period": 7,      "secondary_period": 365,    "min_training_days": 731, "complexity_units": 2},
])

DIAGNOSTIC_MIN_OBS_SHARE = 0.80
DIAGNOSTIC_SERIES_PER_METRIC_DAYPART = 6
DIAGNOSTIC_RANDOM_STATE = 696


# ---------------------------------------------------------------------
# Reader-facing diagnostic controls
# ---------------------------------------------------------------------

PLOT_TRACK = "post_cp_primary"
PLOT_METRIC = "taxi_trip_count"
PLOT_DAYPART = "pm_peak"
PLOT_SEASONAL_CANDIDATE = "single_7"

PLOT_RECENT_POINTS = 100
PLOT_COMPARABLE_DAYS = 120
PLOT_DECOMPOSITION_DAYS = 180
PLOT_WIDTH = 920
PLOT_HEIGHT = 620

RUN_VISUALS = True


# ---------------------------------------------------------------------
# Rebuild toggles
# ---------------------------------------------------------------------

REBUILD_ORIGIN_REGISTRY = False
REBUILD_TRACK_HISTORY = False
REBUILD_ORIGIN_BASE = False
REBUILD_RECENT_FEATURES = False
REBUILD_COMPARABLE_FEATURES = False

REBUILD_SEASONAL_SUPPORT = False
REBUILD_SEASONAL_SAMPLE = False
REBUILD_SEASONAL_DIAGNOSTICS = False

REBUILD_WEEKLY_SELECTION = False
REBUILD_WEEKLY_REFRESH_CALENDAR = False
REBUILD_WEEKLY_RUNTIME_BENCHMARK = False
REBUILD_ANNUAL_FOURIER_FEATURES = False

WRITE_INTERMEDIATES = True


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PIPELINE_DIR = Path("pipeline_data")
INPUT_411_DIR = PIPELINE_DIR / "4.1.1.final_tables"
INPUT_412_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.1.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.2.1.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_SEQUENCE_PATH = INPUT_411_DIR / "forecasting_target_sequence_panel.parquet"
ROLLING_ORIGIN_PATH = INPUT_412_DIR / "rolling_origin_registry.parquet"
TRACK_CONTRACT_PATH = INPUT_412_DIR / "forecasting_track_contract_registry.parquet"
PRE_CP_REFIT_CONTRACT_PATH = INPUT_412_DIR / "pre_cp_refit_application_contract.parquet"

NATIVE_REGISTRY_PATHS = {
    "post_cp_primary": INPUT_412_DIR / "post_cp_primary_sequence_registry.parquet",
    "full_history_regime_aware": INPUT_412_DIR / "full_history_sequence_registry.parquet",
    "pre_cp_reference": INPUT_412_DIR / "pre_cp_reference_sequence_registry.parquet",
}
POST_CP_COMMON_COMPARISON_REGISTRY_PATH = INPUT_412_DIR / "post_cp_common_comparison_sequence_registry.parquet"
CROSS_REGIME_COMPARISON_REGISTRY_PATH = INPUT_412_DIR / "cross_regime_comparison_sequence_registry.parquet"

# ---------------------------------------------------------------------
# Reader-facing labels
# ---------------------------------------------------------------------

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
    "pre_cp_reference": "Pre-CP Reference",
}

METRIC_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway ridership",
}

/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Define shared cache, schema, timing, and diagnostic helpers.** These helpers keep later production blocks compact while enforcing the forecasting contracts already established upstream.

In [3]:
# 4.2.1.0B — Define shared notebook helpers

def require_columns(df, required, name):
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def load_or_build(path, rebuild, builder):
    """Load a valid cached Parquet artifact or rebuild it safely."""

    path = Path(path)

    # Reuse the cache only if it exists and can actually be read.
    if path.exists() and not rebuild:
        try:
            if path.stat().st_size == 0:
                raise ValueError("cached file is 0 bytes")

            return pd.read_parquet(path)

        except Exception as exc:
            print(f"Invalid cache detected for {path.name}: {exc}")
            print("Rebuilding the artifact instead.")
            path.unlink(missing_ok=True)

    df = builder()

    # Write through a temporary Parquet file first so an interrupted write
    # cannot leave a zero-byte artifact masquerading as a valid cache.
    if WRITE_INTERMEDIATES:
        temp_path = path.with_suffix(".tmp.parquet")
        temp_path.unlink(missing_ok=True)

        df.to_parquet(temp_path, index=False)

        if not temp_path.exists() or temp_path.stat().st_size == 0:
            raise IOError(f"Failed to create a valid temporary Parquet file for {path.name}.")

        temp_path.replace(path)

    return df

def track_path(stem, track):
    return INTERMEDIATE_DIR / f"{stem}__{track}.parquet"


def first_validation_origin_id(track):
    rows = rolling_origin_registry_df.loc[
        rolling_origin_registry_df["modeling_track"].eq(track)
        & rolling_origin_registry_df["stage"].eq("rolling_validation")
    ]
    if rows.empty:
        raise ValueError(f"No rolling-validation rows found for {track}.")
    return int(rows["observation_sequence_id"].min())


def training_history_start(track):
    row = forecasting_track_contract_registry_df.loc[
        forecasting_track_contract_registry_df["modeling_track"].eq(track)
    ]
    if len(row) != 1:
        raise ValueError(f"Expected one track-contract row for {track}.")
    return pd.Timestamp(row.iloc[0]["training_history_start_date"])


def component_strength(component, residual):
    component = np.asarray(component, dtype=float)
    residual = np.asarray(residual, dtype=float)
    denominator = np.nanvar(component + residual)
    if not np.isfinite(denominator) or denominator <= 0:
        return np.nan
    return float(np.clip(1 - np.nanvar(residual) / denominator, 0, 1))


def robust_mad(values):
    values = pd.Series(values, dtype=float)
    return float((values - values.median()).abs().median())


def select_training_safe_sequence(history_df, track, metric, daypart=None):
    cutoff = first_validation_origin_id(track)

    eligible = history_df.loc[
        history_df["metric"].eq(metric)
        & history_df["observation_sequence_id"].le(cutoff)
    ].copy()

    if daypart is not None:
        eligible = eligible.loc[eligible["daypart"].eq(daypart)]

    support = (
        eligible.groupby(["forecast_sequence_id", "taxi_zone_id", "zone"], as_index=False)
        .agg(possible_rows=("observed_value", "size"), observed_rows=("observed_value", "count"))
    )
    support["observed_share"] = support["observed_rows"] / support["possible_rows"]

    support = support.sort_values(
        ["observed_share", "observed_rows", "forecast_sequence_id"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    if support.empty:
        raise ValueError(f"No representative sequence available for {track} / {metric} / {daypart}.")

    return support.loc[0, "forecast_sequence_id"]

**Load the authoritative forecasting sequence and experiment contracts.** Only the completed 4.1.1 and 4.1.2 outputs are loaded; no deprecated temporal feature outputs or retrospectively fitted decomposition components enter this notebook.

In [4]:
# 4.2.1.0C — Load and validate authoritative forecasting inputs

SEQUENCE_COLUMNS = [
    "taxi_zone_id", "zone", "borough", "canonical_location_id",
    "date", "daypart", "daypart_order", "temporal_bucket", "temporal_bucket_order",
    "observation_sequence_id", "pre_post_cp", "metric", "observed_value", "forecast_sequence_id",
]

forecast_sequence_df = pd.read_parquet(FORECAST_SEQUENCE_PATH, columns=SEQUENCE_COLUMNS)
rolling_origin_registry_df = pd.read_parquet(ROLLING_ORIGIN_PATH)
forecasting_track_contract_registry_df = pd.read_parquet(TRACK_CONTRACT_PATH)
pre_cp_refit_application_contract_df = pd.read_parquet(PRE_CP_REFIT_CONTRACT_PATH)

require_columns(forecast_sequence_df, SEQUENCE_COLUMNS, "forecasting_target_sequence_panel")
require_columns(
    rolling_origin_registry_df,
    ["modeling_track", "stage", "date", "observation_sequence_id", "target_date", "target_observation_sequence_id", "horizon"],
    "rolling_origin_registry",
)
require_columns(
    forecasting_track_contract_registry_df,
    ["modeling_track", "training_history_start_date", "rolling_validation_target_start_date", "rolling_validation_target_end_date"],
    "forecasting_track_contract_registry",
)
require_columns(
    pre_cp_refit_application_contract_df,
    ["modeling_track", "refit_only_start_date", "final_refit_end_date", "final_refit_end_observation_sequence_id"],
    "pre_cp_refit_application_contract",
)

forecast_sequence_df["date"] = pd.to_datetime(forecast_sequence_df["date"]).dt.normalize()
rolling_origin_registry_df["date"] = pd.to_datetime(rolling_origin_registry_df["date"]).dt.normalize()
rolling_origin_registry_df["target_date"] = pd.to_datetime(rolling_origin_registry_df["target_date"]).dt.normalize()

for col in ["training_history_start_date", "rolling_validation_target_start_date", "rolling_validation_target_end_date"]:
    forecasting_track_contract_registry_df[col] = pd.to_datetime(forecasting_track_contract_registry_df[col]).dt.normalize()

for col in ["refit_only_start_date", "final_refit_end_date"]:
    pre_cp_refit_application_contract_df[col] = pd.to_datetime(pre_cp_refit_application_contract_df[col]).dt.normalize()

native_registry_df_by_track = {}
for track, path in NATIVE_REGISTRY_PATHS.items():
    native_registry_df_by_track[track] = pd.read_parquet(path)
    require_columns(native_registry_df_by_track[track], ["forecast_sequence_id", "metric"], path.name)

# Preserve the authoritative comparison cohorts defined upstream in 4.1.2.
# These registries do not redefine native forecasting eligibility; they record
# which already-valid Forecast Sequences support apples-to-apples comparisons.
post_cp_common_comparison_sequence_registry_df = pd.read_parquet(
    POST_CP_COMMON_COMPARISON_REGISTRY_PATH
)
cross_regime_comparison_sequence_registry_df = pd.read_parquet(
    CROSS_REGIME_COMPARISON_REGISTRY_PATH
)

require_columns(
    post_cp_common_comparison_sequence_registry_df,
    ["forecast_sequence_id"],
    POST_CP_COMMON_COMPARISON_REGISTRY_PATH.name,
)
require_columns(
    cross_regime_comparison_sequence_registry_df,
    ["forecast_sequence_id"],
    CROSS_REGIME_COMPARISON_REGISTRY_PATH.name,
)

post_cp_common_comparison_sequence_ids = set(
    post_cp_common_comparison_sequence_registry_df["forecast_sequence_id"]
)
cross_regime_comparison_sequence_ids = set(
    cross_regime_comparison_sequence_registry_df["forecast_sequence_id"]
)

sequence_calendar_df = (
    forecast_sequence_df[
        ["observation_sequence_id", "date", "daypart", "daypart_order",
         "temporal_bucket", "temporal_bucket_order", "pre_post_cp"]
    ]
    .drop_duplicates()
    .sort_values("observation_sequence_id")
    .reset_index(drop=True)
)

if sequence_calendar_df["observation_sequence_id"].duplicated().any():
    raise ValueError("Canonical observation_sequence_id does not map uniquely to the sequence calendar.")

input_inventory_df = pd.DataFrame([
    {"input": "Forecast sequence panel", "rows": len(forecast_sequence_df), "sequences": forecast_sequence_df["forecast_sequence_id"].nunique()},
    {"input": "Rolling-origin registry", "rows": len(rolling_origin_registry_df), "sequences": np.nan},
    {"input": "Post-CP Primary native registry", "rows": len(native_registry_df_by_track["post_cp_primary"]), "sequences": native_registry_df_by_track["post_cp_primary"]["forecast_sequence_id"].nunique()},
    {"input": "Full-History native registry", "rows": len(native_registry_df_by_track["full_history_regime_aware"]), "sequences": native_registry_df_by_track["full_history_regime_aware"]["forecast_sequence_id"].nunique()},
    {"input": "Pre-CP Reference native registry", "rows": len(native_registry_df_by_track["pre_cp_reference"]), "sequences": native_registry_df_by_track["pre_cp_reference"]["forecast_sequence_id"].nunique()},
    {"input": "Post-CP common comparison registry", "rows": len(post_cp_common_comparison_sequence_registry_df), "sequences": post_cp_common_comparison_sequence_registry_df["forecast_sequence_id"].nunique()},
    {"input": "Cross-regime comparison registry", "rows": len(cross_regime_comparison_sequence_registry_df), "sequences": cross_regime_comparison_sequence_registry_df["forecast_sequence_id"].nunique()},
])

display(input_inventory_df)

,input,rows,sequences
0,Forecast sequence panel,7797950,1315.0
1,Rolling-origin registry,35496,NaN
2,Post-CP Primary native registry,1108,1108.0
3,Full-History native registry,1051,1051.0
4,Pre-CP Reference native registry,1039,1039.0
5,Post-CP common comparison registry,1042,1042.0
6,Cross-regime comparison registry,1029,1029.0


**Findings.** The forecasting inputs loaded at the expected scales: the canonical sequence panel contains 7,797,950 rows across 1,315 forecast sequences, and the rolling-origin registry contains 35,496 scheduled origin/horizon rows. The three native modeling populations contain 1,108 Post-CP Primary sequences, 1,051 Full-History Regime-Aware sequences, and 1,039 Pre-CP Reference sequences, confirming that feature construction begins from the same modeling populations established upstream.

---

# 4.2.1.1 Establish the Leakage-Safe Temporal Feature Contract

Every feature in this notebook is anchored to a **forecast origin**. The 4.1.2 rolling registry is horizon-specific because one origin can predict several future targets, but the information available at that origin does not change according to whether we are predicting one, two, or five steps ahead.

The canonical feature grain is therefore:

**Modeling Track × Forecast Sequence × Origin Observation Sequence ID**

The three future targets will be joined back later through the 4.1.2 origin × horizon registry.

Each modeling track also receives only its legitimate historical source. In particular, the Post-CP Primary is physically prevented from reaching backward into the pre-CP period. This makes leakage prevention part of the data construction itself rather than something downstream model code must remember to enforce.

Simple calendar fields are also added here. Daypart, temporal bucket, sequence position, weekday/weekend status, month, and elapsed time are deterministic at the forecast origin, so they do not require a separate feature-engineering chapter.



**4.2.1.1A — Construct the canonical feature-origin registry.**

The forecasting feature layer needs one authoritative list of observation states at which predictors must exist. Ordinary rolling-validation origins come directly from the frozen 4.1.2 schedule.

The **Pre-CP Reference** also requires a final-refit origin set. Eligibility for this set is defined by the **January 1–4, 2025 target window**: each legal target is shifted backward by the frozen h=1, h=2, and h=5 forecast horizons to recover every observation state needed to construct its predictors. The final January 4 Evening state is retained as the last legal observed state before recursive no-congestion-pricing forecasting begins.

If a required final-refit origin already belongs to the ordinary rolling schedule, its scheduled-backtest role remains authoritative. This keeps model selection fixed through **December 31, 2024** while ensuring that the feature surface contains every legal observed state needed for the final production refit through **January 4, 2025**.

In [5]:
# 4.2.1.1A — Construct the canonical feature-origin registry

FEATURE_ORIGIN_PATH = INTERMEDIATE_DIR / "forecast_feature_origin_registry.parquet"

def build_feature_origin_registry():
    scheduled = rolling_origin_registry_df[["modeling_track", "observation_sequence_id"]].drop_duplicates().copy()
    scheduled["origin_role"] = "scheduled_backtest"

    pre_cp_contract = pre_cp_refit_application_contract_df.loc[
        pre_cp_refit_application_contract_df["modeling_track"].eq("pre_cp_reference")
    ].iloc[0]

    # Final-refit membership is defined by target date. Shift the Jan. 1–4
    # targets backward by each frozen horizon to recover every required origin.
    refit_targets = sequence_calendar_df.loc[
        sequence_calendar_df["date"].between(
            pre_cp_contract["refit_only_start_date"],
            pre_cp_contract["final_refit_end_date"],
            inclusive="both",
        ),
        ["observation_sequence_id"],
    ].rename(columns={
        "observation_sequence_id": "target_observation_sequence_id"
    })

    refit_ids = pd.concat(
        [
            refit_targets.assign(
                observation_sequence_id=(
                    refit_targets["target_observation_sequence_id"] - horizon
                )
            )[["observation_sequence_id"]]
            for horizon in FORECAST_HORIZONS
        ],
        ignore_index=True,
    )

    # Keep Jan. 4 Evening as the last legal observed state before recursion.
    refit_ids = pd.concat(
        [
            refit_ids,
            pd.DataFrame({
                "observation_sequence_id": [
                    int(refit_targets["target_observation_sequence_id"].max())
                ]
            }),
        ],
        ignore_index=True,
    ).drop_duplicates()

    refit_ids["modeling_track"] = "pre_cp_reference"
    refit_ids["origin_role"] = "pre_cp_refit_only"

    origins = pd.concat([scheduled, refit_ids], ignore_index=True)
    origins["role_priority"] = origins["origin_role"].eq("pre_cp_refit_only").astype(int)

    origins = (
        origins.sort_values(["modeling_track", "observation_sequence_id", "role_priority"])
        .drop_duplicates(["modeling_track", "observation_sequence_id"], keep="first")
        .drop(columns="role_priority")
        .merge(sequence_calendar_df, on="observation_sequence_id", how="left", validate="many_to_one")
        .sort_values(["modeling_track", "observation_sequence_id"])
        .reset_index(drop=True)
    )

    if origins.duplicated(["modeling_track", "observation_sequence_id"]).any():
        raise ValueError("Duplicate track × feature-origin rows detected.")

    return origins

feature_origin_registry_df = load_or_build(
    FEATURE_ORIGIN_PATH, REBUILD_ORIGIN_REGISTRY, build_feature_origin_registry
)

feature_origin_summary_df = (
    feature_origin_registry_df.groupby(["modeling_track", "origin_role"], as_index=False)
    .agg(origin_count=("observation_sequence_id", "nunique"), first_origin=("date", "min"), last_origin=("date", "max"))
)
feature_origin_summary_df["modeling_track"] = feature_origin_summary_df["modeling_track"].map(TRACK_LABELS)

display(feature_origin_summary_df)

,modeling_track,origin_role,origin_count,first_origin,last_origin
0,Full-History Regime-Aware,scheduled_backtest,5929,2023-01-01,2026-03-31
1,Post-CP Primary,scheduled_backtest,2254,2025-01-05,2026-03-31
2,Pre-CP Reference,pre_cp_refit_only,21,2024-12-31,2025-01-04
3,Pre-CP Reference,scheduled_backtest,3654,2023-01-01,2024-12-31


**Findings.** Collapsing the horizon-specific backtest schedule produces one reusable feature origin per track and sequence position while preserving each track’s intended calendar. Full-History has 5,929 scheduled origins from January 1, 2023 through March 31, 2026; Post-CP Primary has 2,254 scheduled origins from January 5, 2025 through March 31, 2026; and Pre-CP Reference preserves 3,654 scheduled backtest origins through December 31, 2024 plus 20 refit-only positions covering January 1–4, 2025.

---

**Build each track's permitted historical source.** Lagged and rolling features will be calculated only after the canonical panel has been restricted to that track's native sequence population and legitimate historical start date.

In [6]:
# 4.2.1.1B — Build track-specific leakage-safe historical sources

TRACK_HISTORY_COLUMNS = [
    "taxi_zone_id", "zone", "borough", "metric", "forecast_sequence_id",
    "observation_sequence_id", "date", "daypart", "daypart_order",
    "temporal_bucket", "temporal_bucket_order", "pre_post_cp", "observed_value",
]

def build_track_history(track):
    sequence_ids = set(native_registry_df_by_track[track]["forecast_sequence_id"])
    start_date = training_history_start(track)
    last_origin_id = int(
        feature_origin_registry_df.loc[
            feature_origin_registry_df["modeling_track"].eq(track), "observation_sequence_id"
        ].max()
    )

    history = forecast_sequence_df.loc[
        forecast_sequence_df["forecast_sequence_id"].isin(sequence_ids)
        & forecast_sequence_df["date"].ge(start_date)
        & forecast_sequence_df["observation_sequence_id"].le(last_origin_id),
        TRACK_HISTORY_COLUMNS,
    ].copy()

    history.insert(0, "modeling_track", track)
    history = history.sort_values(["forecast_sequence_id", "observation_sequence_id"]).reset_index(drop=True)

    if history.duplicated(["forecast_sequence_id", "observation_sequence_id"]).any():
        raise ValueError(f"Duplicate sequence positions detected for {track}.")

    return history

track_history_summary = []

for track in TRACK_LABELS:
    history_df = load_or_build(
        track_path("track_history_source", track),
        REBUILD_TRACK_HISTORY,
        lambda track=track: build_track_history(track),
    )

    track_history_summary.append({
        "modeling_track": TRACK_LABELS[track],
        "rows": len(history_df),
        "sequences": history_df["forecast_sequence_id"].nunique(),
        "first_date": history_df["date"].min(),
        "last_date": history_df["date"].max(),
        "observed_share": history_df["observed_value"].notna().mean(),
    })

    del history_df
    gc.collect()

track_history_summary_df = pd.DataFrame(track_history_summary)
display(track_history_summary_df.style.format({"observed_share": "{:.3f}"}))

,modeling_track,rows,sequences,first_date,last_date,observed_share
0,Post-CP Primary,2497432,1108,2025-01-05 00:00:00,2026-03-31 00:00:00,0.994
1,Full-History Regime-Aware,6231379,1051,2023-01-01 00:00:00,2026-03-31 00:00:00,0.994
2,Pre-CP Reference,3818325,1039,2023-01-01 00:00:00,2025-01-04 00:00:00,0.995


**Findings.** The track-specific history tables enforce the intended information boundaries directly in the data. Post-CP Primary begins on January 5, 2025 and contains 2,497,432 rows across 1,108 sequences; Full-History reaches back to January 1, 2023 with 6,231,379 rows across 1,051 sequences; and Pre-CP Reference stops on January 4, 2025 with 3,818,325 rows across 1,039 sequences. Observed-value coverage remains consistently high across all three histories, so the track restrictions preserve nearly all usable target-series information while preventing access to disallowed periods.

---

**Create the origin backbone and deterministic calendar context.** This is the base table to which every temporal feature family will eventually join.

In [7]:
# 4.2.1.1C — Materialize the origin-level feature backbone

STUDY_START_DATE = sequence_calendar_df["date"].min()

def build_origin_base(track):
    history = pd.read_parquet(track_path("track_history_source", track))
    origin_roles = feature_origin_registry_df.loc[
        feature_origin_registry_df["modeling_track"].eq(track),
        ["observation_sequence_id", "origin_role"],
    ]

    base = (
        history.loc[history["observation_sequence_id"].isin(set(origin_roles["observation_sequence_id"]))]
        .merge(origin_roles, on="observation_sequence_id", how="left", validate="many_to_one")
        .rename(columns={
            "date": "origin_date",
            "daypart": "origin_daypart",
            "daypart_order": "origin_daypart_order",
            "temporal_bucket": "origin_temporal_bucket",
            "temporal_bucket_order": "origin_temporal_bucket_order",
            "pre_post_cp": "origin_pre_post_cp",
            "observed_value": "origin_value",
        })
    )

    base["origin_weekday_index"] = base["origin_date"].dt.weekday.astype("int8")
    base["origin_is_weekend"] = base["origin_weekday_index"].ge(5).astype("int8")
    base["origin_month"] = base["origin_date"].dt.month.astype("int8")
    base["origin_day_of_year"] = base["origin_date"].dt.dayofyear.astype("int16")
    base["origin_elapsed_days"] = (base["origin_date"] - STUDY_START_DATE).dt.days.astype("int16")

    return base.sort_values(["forecast_sequence_id", "observation_sequence_id"]).reset_index(drop=True)

origin_base_summary = []

for track in TRACK_LABELS:
    base_df = load_or_build(
        track_path("origin_feature_base", track),
        REBUILD_ORIGIN_BASE,
        lambda track=track: build_origin_base(track),
    )

    duplicate_rows = int(base_df.duplicated(["forecast_sequence_id", "observation_sequence_id"]).sum())
    origin_base_summary.append({
        "modeling_track": TRACK_LABELS[track],
        "origin_rows": len(base_df),
        "sequences": base_df["forecast_sequence_id"].nunique(),
        "duplicate_rows": duplicate_rows,
        "origin_value_available": base_df["origin_value"].notna().mean(),
    })

    if duplicate_rows:
        raise ValueError(f"Duplicate origin feature rows detected for {track}.")

    del base_df
    gc.collect()

origin_base_summary_df = pd.DataFrame(origin_base_summary)
display(origin_base_summary_df.style.format({"origin_value_available": "{:.1%}"}))

,modeling_track,origin_rows,sequences,duplicate_rows,origin_value_available
0,Post-CP Primary,2497432,1108,0,99.4%
1,Full-History Regime-Aware,6231379,1051,0,99.4%
2,Pre-CP Reference,3818325,1039,0,99.5%


**Findings.** The origin backbone is unique at the Forecast Sequence × Origin Observation Sequence ID grain for all three tracks. Post-CP Primary contains 2,497,432 origin rows across 1,108 sequences, Full-History contains 6,231,379 across 1,051 sequences, and Pre-CP Reference contains 3,817,286 across 1,039 sequences; all three report zero duplicate rows. Origin values are available for 99.4%, 99.4%, and 99.5% of rows, respectively, leaving only a small amount of source-series missingness while deterministic calendar context remains defined for every origin.

### Section Summary

Section 4.2.1.1 converted the forecasting experiment contract into a leakage-safe feature backbone. Horizon-specific schedules were reduced to reusable forecast origins, each modeling track was restricted to its legal historical window, and the resulting origin tables are duplicate-free with approximately 99.4%–99.5% observed-value availability. Calendar and sequence context can now travel with every origin without changing the information boundary.

---

# 4.2.1.2 Construct Recent Sequence History Features

The first predictors describe the mobility state immediately preceding each forecast origin. The canonical Taxi Zone × Metric sequence moves continuously through the five ordered dayparts, so these short lags capture recent system state without assuming any seasonal structure.

A one-step lag means the immediately preceding canonical daypart, a two-step lag means two dayparts earlier, and so on. Missing observations remain explicit positions in the sequence; the code never jumps backward to the previous non-missing value.

The current origin value is already preserved in the feature backbone and is legitimately known when forecasting future positions. During the later no-CP counterfactual application, however, target-history values after January 5 will have to be supplied recursively from earlier counterfactual forecasts rather than from actual observed post-CP mobility.

---

**Construct one- through four-step recent sequence history.** Validate each lag against the canonical observation-position arithmetic before retaining origin rows.

In [8]:
# 4.2.1.2A — Construct recent sequence-history features

def build_recent_features(track):
    """
    Build leakage-safe short-lag features from one modeling track's legal history.

    Input grain
    -----------
    Taxi Zone × Metric × canonical observation_sequence_id.

    Output grain
    ------------
    Forecast Sequence × registered forecast-origin observation_sequence_id for
    the requested modeling track.

    Method
    ------
    The track-specific history table already enforces the legal historical
    boundary for the requested modeling track. Within each Forecast Sequence,
    rows are ordered by canonical observation_sequence_id and shifted by 1–4
    positions.

    Because missing canonical observations remain as rows in the sequence,
    `shift(k)` means exactly k canonical daypart positions earlier rather than
    "the kth previous non-missing observation."

    Leakage safeguards
    ------------------
    Lag source IDs are checked against canonical sequence arithmetic so a future
    ordering/filtering change cannot silently convert these into misaligned lags.
    Features are calculated over the complete legal track history before the
    result is restricted to registered forecast origins, preserving all legitimate
    pre-origin context.

    Counterfactual note
    -------------------
    These are observed-history features. They are directly usable for ordinary
    rolling forecasts and Pre-CP model development. Later no-CP counterfactual
    application after the final legal Pre-CP observation will require recursive
    or otherwise forecastable history rather than actual post-CP outcomes.

    Parameters
    ----------
    track : str
        Modeling-track identifier used to load the legal track-specific history
        and identify the registered forecast origins.

    Returns
    -------
    pandas.DataFrame
        Recent-history features at the canonical forecast-origin grain.
    """
    history = pd.read_parquet(track_path("track_history_source", track))

    # Canonical sequence order is the forecasting clock:
    # Overnight → AM Peak → Midday → PM Peak → Evening → next-day Overnight.
    history = history.sort_values(
        ["forecast_sequence_id", "observation_sequence_id"]
    ).reset_index(drop=True)
    grouped = history.groupby("forecast_sequence_id", sort=False)

    # Preserve exact sequence-position semantics for every lag. The source-ID
    # validation protects against accidentally skipping missing canonical rows.
    for lag in RECENT_SEQUENCE_LAGS:
        history[f"seq_lag_{lag}"] = grouped["observed_value"].shift(lag)

        source_id = grouped["observation_sequence_id"].shift(lag)
        valid = source_id.notna()

        if not (
            history.loc[valid, "observation_sequence_id"] - source_id.loc[valid]
        ).eq(lag).all():
            raise ValueError(
                f"{track}: seq_lag_{lag} failed canonical sequence-gap validation."
            )

    # Current origin value minus the immediately preceding canonical daypart.
    # The origin value itself is known when an ordinary future forecast is issued.
    history["seq_delta_1"] = history["observed_value"] - history["seq_lag_1"]

    # Construct history first, then project onto actual registered origins so
    # earlier legal observations remain available when generating each feature.
    origin_ids = set(
        feature_origin_registry_df.loc[
            feature_origin_registry_df["modeling_track"].eq(track),
            "observation_sequence_id",
        ]
    )

    columns = [
        "modeling_track", "forecast_sequence_id", "metric", "observation_sequence_id",
        "seq_lag_1", "seq_lag_2", "seq_lag_3", "seq_lag_4", "seq_delta_1",
    ]

    return (
        history.loc[history["observation_sequence_id"].isin(origin_ids), columns]
        .sort_values(["forecast_sequence_id", "observation_sequence_id"])
        .reset_index(drop=True)
    )


# Cache one recent-history feature table per modeling track.
recent_summaries = []

for track in TRACK_LABELS:
    recent_df = load_or_build(
        track_path("recent_sequence_features", track),
        REBUILD_RECENT_FEATURES,
        lambda track=track: build_recent_features(track),
    )

    # Availability is descriptive QA only. It does not redefine the native
    # forecasting cohort or discard series with incomplete historical features.
    report_cols = ["seq_lag_1", "seq_lag_2", "seq_lag_3", "seq_lag_4", "seq_delta_1"]

    summary = (
        recent_df.groupby("metric", as_index=False)
        .agg(**{
            f"{col}_available": (col, lambda s: s.notna().mean())
            for col in report_cols
        })
    )
    summary.insert(0, "modeling_track", track)
    recent_summaries.append(summary)

recent_feature_coverage_df = pd.concat(recent_summaries, ignore_index=True)

# Reader-facing labels are applied only to the display copy.
recent_feature_coverage_display_df = recent_feature_coverage_df.copy()
recent_feature_coverage_display_df["modeling_track"] = (
    recent_feature_coverage_display_df["modeling_track"].map(TRACK_LABELS)
)
recent_feature_coverage_display_df["metric"] = (
    recent_feature_coverage_display_df["metric"].map(METRIC_LABELS)
)

display(
    recent_feature_coverage_display_df.style.format({
        col: "{:.3f}"
        for col in recent_feature_coverage_display_df.columns
        if col.endswith("_available")
    })
)

,modeling_track,metric,seq_lag_1_available,seq_lag_2_available,seq_lag_3_available,seq_lag_4_available,seq_delta_1_available
0,Post-CP Primary,FHVHV speed,0.998,0.998,0.997,0.997,0.997
1,Post-CP Primary,FHVHV trips,0.998,0.998,0.997,0.997,0.997
2,Post-CP Primary,Subway ridership,0.996,0.996,0.996,0.996,0.995
3,Post-CP Primary,Taxi speed,0.975,0.975,0.974,0.974,0.954
4,Post-CP Primary,Taxi trips,0.998,0.998,0.997,0.997,0.997
5,Full-History Regime-Aware,FHVHV speed,0.998,0.997,0.997,0.997,0.996
6,Full-History Regime-Aware,FHVHV trips,0.998,0.998,0.997,0.997,0.996
7,Full-History Regime-Aware,Subway ridership,0.989,0.989,0.989,0.989,0.987
8,Full-History Regime-Aware,Taxi speed,0.972,0.972,0.971,0.971,0.949
9,Full-History Regime-Aware,Taxi trips,0.998,0.998,0.997,0.997,0.996


**Findings.** The one- through four-step sequence lags are available for the overwhelming majority of forecast origins across the track and metric combinations, with only small declines as lag depth increases. For example, Post-CP FHVHV speed retains 99.8% availability at one and two steps and 99.7% at three and four steps, while its one-step change feature is also available for 99.7% of origins. The remaining gaps therefore reflect source-series missingness and normal start-of-history edge effects rather than a breakdown in sequence alignment.

---

**Visualize what a one-step sequence lag actually means.** The representative sequence is selected using only information available before validation begins.

In [9]:
# 4.2.1.2B — Visualize observed origin values against the immediate sequence lag

if RUN_VISUALS:
    history = pd.read_parquet(track_path("track_history_source", PLOT_TRACK))
    sequence_id = select_training_safe_sequence(history, PLOT_TRACK, PLOT_METRIC)

    base = pd.read_parquet(track_path("origin_feature_base", PLOT_TRACK))
    recent = pd.read_parquet(track_path("recent_sequence_features", PLOT_TRACK))
    cutoff = first_validation_origin_id(PLOT_TRACK)

    plot_df = (
        base.loc[
            base["forecast_sequence_id"].eq(sequence_id)
            & base["observation_sequence_id"].le(cutoff)
        ]
        .merge(
            recent[["forecast_sequence_id", "observation_sequence_id", "seq_lag_1"]],
            on=["forecast_sequence_id", "observation_sequence_id"],
            how="left",
            validate="one_to_one",
        )
        .sort_values("observation_sequence_id")
        .tail(PLOT_RECENT_POINTS)
    )

    plot_df["origin_label"] = plot_df["origin_date"].dt.strftime("%b %d") + " · " + plot_df["origin_daypart"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df["origin_label"], y=plot_df["origin_value"], mode="lines",
        name="Observed at origin", line=dict(color=BRAND_COLORS["dark_teal"], width=2.5)
    ))
    fig.add_trace(go.Scatter(
        x=plot_df["origin_label"], y=plot_df["seq_lag_1"], mode="lines",
        name="Previous sequence step", line=dict(color=BRAND_COLORS["terracotta"], width=2)
    ))

    fig.update_layout(
        title=f"<b>Recent Sequence History Alignment</b><br><sup>{TRACK_LABELS[PLOT_TRACK]} · {METRIC_LABELS[PLOT_METRIC]} · {plot_df['zone'].iloc[0]}</sup>",
        width=PLOT_WIDTH, height=520, margin=dict(l=60, r=30, t=90, b=110),
        xaxis_title="Forecast origin", yaxis_title=METRIC_LABELS[PLOT_METRIC],
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_xaxes(tickangle=-55, nticks=18)
    fig.show()

**Findings.** In the representative Post-CP Taxi-trip series for Baisley Park, the one-step lag tracks the same short-run movement as the observed series while remaining displaced by one canonical sequence position. The visualization therefore confirms the intended alignment: recent-history predictors preserve the immediately preceding mobility state without borrowing information from the future target.

### 4.2.1.2 Summary

Recent own-history features are now available at the forecast-origin grain without changing the backtest schedule. The four sequence lags provide progressively deeper short-run memory and the one-step change adds a compact movement signal, while the representative alignment check confirms that these predictors follow the ordered sequence rather than looking ahead.

---

# 4.2.1.3 Construct Comparable-Period and Longer-History Features

Recent adjacent dayparts describe the immediate mobility state, but many useful comparisons are more naturally made against the **same daypart on earlier calendar days**.

Because every Taxi Zone × Metric sequence contains exactly one observation for each of the five dayparts per calendar day, the prior observation within the same daypart is exactly five canonical sequence steps earlier. Seven prior observations within that daypart are exactly one week earlier.

This lets us add interpretable comparable-period history without first estimating a seasonal model:

- previous day's same daypart;
- two days earlier at the same daypart;
- previous week's same daypart;
- short and medium rolling level;
- short and medium rolling volatility;
- an exponentially weighted recent level.

The rolling summaries are deliberately shifted one same-daypart observation before calculation. The current origin value remains separately available, while the rolling features represent **history strictly before that origin**.

---

**Construct same-daypart lags and shifted rolling histories.**

In [10]:
# 4.2.1.3A — Construct comparable-period and longer-history features

def build_comparable_features(track):
    """
    Build leakage-safe same-daypart history features for one modeling track.

    The source history is ordered at the Taxi Zone × Metric × Daypart daily-series
    grain, where each row represents one recurring daypart within a forecast sequence.
    Features are constructed only from values that occur strictly before the current
    forecast origin.

    This block creates:
    - same-daypart lags at 1, 2, and 7 days;
    - a 1-day same-daypart change;
    - prior-only 7- and 28-day rolling mean, standard deviation, and observed-share
      summaries;
    - a prior-only 7-day exponentially weighted mean.

    Canonical sequence-gap checks verify that each same-daypart lag points to the
    expected earlier sequence position, using `STEPS_PER_DAY` to translate day lags
    back to canonical observation-sequence distance.

    Rolling and exponentially weighted summaries are shifted by one comparable
    observation before aggregation, so the current origin value is never included
    in its own historical feature window.

    The returned dataframe is restricted to registered forecast origins for the
    requested modeling track and contains one row per Forecast Sequence × Origin
    Observation Sequence ID with the comparable-period feature columns attached.

    Parameters
    ----------
    track : str
        Modeling-track identifier used to load the legal track-specific history
        and restrict the result to that track's registered forecast origins.

    Returns
    -------
    pandas.DataFrame
        Leakage-safe comparable-period features at the forecast-origin grain.
    """
    history = pd.read_parquet(track_path("track_history_source", track))
    history = history.sort_values(["forecast_sequence_id", "daypart", "observation_sequence_id"]).reset_index(drop=True)

    keys = ["forecast_sequence_id", "daypart"]
    grouped = history.groupby(keys, sort=False, observed=True)

    for day_lag in SAME_DAYPART_DAY_LAGS:
        col = f"same_daypart_lag_{day_lag}d"
        history[col] = grouped["observed_value"].shift(day_lag)

        source_id = grouped["observation_sequence_id"].shift(day_lag)
        valid = source_id.notna()
        expected_gap = day_lag * STEPS_PER_DAY

        if not (history.loc[valid, "observation_sequence_id"] - source_id.loc[valid]).eq(expected_gap).all():
            raise ValueError(f"{track}: {col} failed canonical sequence-gap validation.")

    history["same_daypart_delta_1d"] = history["observed_value"] - history["same_daypart_lag_1d"]

    # Shift first so rolling summaries contain only comparable periods strictly before the origin.
    history["_prior_same_daypart_value"] = grouped["observed_value"].shift(1)
    history["_prior_same_daypart_observed"] = history["_prior_same_daypart_value"].notna().astype(float)

    prior_group = history.groupby(keys, sort=False, observed=True)["_prior_same_daypart_value"]
    observed_group = history.groupby(keys, sort=False, observed=True)["_prior_same_daypart_observed"]

    for window in SAME_DAYPART_ROLLING_WINDOWS:
        min_periods = math.ceil(window * ROLLING_MIN_OBS_SHARE)

        history[f"same_daypart_roll_mean_{window}d"] = prior_group.transform(
            lambda s, w=window, m=min_periods: s.rolling(w, min_periods=m).mean()
        )
        history[f"same_daypart_roll_std_{window}d"] = prior_group.transform(
            lambda s, w=window, m=min_periods: s.rolling(w, min_periods=m).std()
        )
        history[f"same_daypart_roll_obs_share_{window}d"] = observed_group.transform(
            lambda s, w=window: s.rolling(w, min_periods=1).mean()
        )

    history["same_daypart_ewm_mean_7d"] = prior_group.transform(
        lambda s: s.ewm(span=7, adjust=False, min_periods=math.ceil(7 * ROLLING_MIN_OBS_SHARE)).mean()
    )

    origin_ids = set(
        feature_origin_registry_df.loc[
            feature_origin_registry_df["modeling_track"].eq(track), "observation_sequence_id"
        ]
    )

    feature_cols = [
        "same_daypart_lag_1d", "same_daypart_lag_2d", "same_daypart_lag_7d", "same_daypart_delta_1d",
        "same_daypart_roll_mean_7d", "same_daypart_roll_std_7d", "same_daypart_roll_obs_share_7d",
        "same_daypart_ewm_mean_7d", "same_daypart_roll_mean_28d", "same_daypart_roll_std_28d",
        "same_daypart_roll_obs_share_28d",
    ]

    keep = ["modeling_track", "forecast_sequence_id", "metric", "observation_sequence_id", *feature_cols]

    return (
        history.loc[history["observation_sequence_id"].isin(origin_ids), keep]
        .sort_values(["forecast_sequence_id", "observation_sequence_id"])
        .reset_index(drop=True)
    )

comparable_summaries = []

for track in TRACK_LABELS:
    comparable_df = load_or_build(
        track_path("comparable_history_features", track),
        REBUILD_COMPARABLE_FEATURES,
        lambda track=track: build_comparable_features(track),
    )

    report_cols = [
        "same_daypart_lag_1d", "same_daypart_lag_7d",
        "same_daypart_roll_mean_7d", "same_daypart_roll_std_7d",
        "same_daypart_roll_mean_28d", "same_daypart_roll_std_28d",
        "same_daypart_ewm_mean_7d",
    ]

    summary = (
        comparable_df.groupby("metric", as_index=False)
        .agg(**{f"{col}_available": (col, lambda s: s.notna().mean()) for col in report_cols})
    )
    summary.insert(0, "modeling_track", TRACK_LABELS[track])
    comparable_summaries.append(summary)

comparable_feature_availability_df = pd.concat(comparable_summaries, ignore_index=True)
availability_cols = [c for c in comparable_feature_availability_df if c.endswith("_available")]

display(comparable_feature_availability_df.style.format({c: "{:.1%}" for c in availability_cols}))

,modeling_track,metric,same_daypart_lag_1d_available,same_daypart_lag_7d_available,same_daypart_roll_mean_7d_available,same_daypart_roll_std_7d_available,same_daypart_roll_mean_28d_available,same_daypart_roll_std_28d_available,same_daypart_ewm_mean_7d_available
0,Post-CP Primary,fhvhv_avg_trip_speed,99.6%,98.3%,98.4%,98.4%,94.7%,94.7%,98.7%
1,Post-CP Primary,fhvhv_trip_count,99.6%,98.3%,98.5%,98.5%,94.8%,94.8%,98.7%
2,Post-CP Primary,subway_ridership,99.6%,98.3%,98.6%,98.6%,94.9%,94.9%,98.7%
3,Post-CP Primary,taxi_avg_trip_speed,97.3%,96.1%,94.9%,94.9%,91.9%,91.9%,98.4%
4,Post-CP Primary,taxi_trip_count,99.6%,98.3%,98.5%,98.5%,94.8%,94.8%,98.7%
5,Full-History Regime-Aware,fhvhv_avg_trip_speed,99.7%,99.2%,99.1%,99.1%,97.7%,97.7%,99.5%
6,Full-History Regime-Aware,fhvhv_trip_count,99.7%,99.2%,99.1%,99.1%,97.7%,97.7%,99.5%
7,Full-History Regime-Aware,subway_ridership,98.9%,98.4%,98.4%,98.4%,97.1%,97.1%,99.5%
8,Full-History Regime-Aware,taxi_avg_trip_speed,97.1%,96.6%,94.6%,94.6%,93.3%,93.3%,99.4%
9,Full-History Regime-Aware,taxi_trip_count,99.7%,99.2%,99.1%,99.1%,97.7%,97.7%,99.5%


**Findings.** Comparable-period features remain broadly available, but support falls in the expected order as the amount of required history grows. One-day and seven-day same-daypart lags retain the strongest coverage, while the shifted 28-day rolling summaries lose more rows near the beginning of each legal history window and for metrics with more missing observations. For example, Post-CP FHVHV speed retains 99.6% availability for the prior-day comparison and 98.3% for the prior-week comparison, with the longer rolling features introducing the expected additional warm-up requirement.

---

**Visualize the leakage-safe rolling and exponentially weighted histories.**

In [11]:
# 4.2.1.3B — Visualize comparable-period smoothing

if RUN_VISUALS:
    history = pd.read_parquet(track_path("track_history_source", PLOT_TRACK))
    sequence_id = select_training_safe_sequence(history, PLOT_TRACK, PLOT_METRIC, PLOT_DAYPART)

    base = pd.read_parquet(track_path("origin_feature_base", PLOT_TRACK))
    comparable = pd.read_parquet(track_path("comparable_history_features", PLOT_TRACK))
    cutoff = first_validation_origin_id(PLOT_TRACK)

    plot_df = (
        base.loc[
            base["forecast_sequence_id"].eq(sequence_id)
            & base["origin_daypart"].eq(PLOT_DAYPART)
            & base["observation_sequence_id"].le(cutoff)
        ]
        .merge(
            comparable,
            on=["modeling_track", "forecast_sequence_id", "metric", "observation_sequence_id"],
            how="left",
            validate="one_to_one",
        )
        .sort_values("observation_sequence_id")
        .tail(PLOT_COMPARABLE_DAYS)
    )

    fig = go.Figure()

    specs = [
        ("origin_value", "Observed", BRAND_COLORS["dark_teal"], None, 2.2),
        ("same_daypart_roll_mean_7d", "Prior 7-day rolling mean", BRAND_COLORS["terracotta"], None, 2),
        ("same_daypart_ewm_mean_7d", "Prior 7-day EMA", BRAND_COLORS["seafoam"], None, 2),
        ("same_daypart_roll_mean_28d", "Prior 28-day rolling mean", "#555555", "dot", 1.7),
    ]

    for col, label, color, dash, width in specs:
        fig.add_trace(go.Scatter(
            x=plot_df["origin_date"], y=plot_df[col], mode="lines", name=label,
            line=dict(color=color, dash=dash, width=width),
        ))

    fig.update_layout(
        title=f"<b>Leakage-Safe Comparable-Period Smoothing</b><br><sup>{TRACK_LABELS[PLOT_TRACK]} · {METRIC_LABELS[PLOT_METRIC]} · {plot_df['zone'].iloc[0]} · {PLOT_DAYPART}</sup>",
        width=PLOT_WIDTH, height=540, margin=dict(l=60, r=30, t=90, b=60),
        xaxis_title="Origin date", yaxis_title=METRIC_LABELS[PLOT_METRIC],
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.show()

**Findings.** For Post-CP Primary Taxi trips in Baisley Park during the PM peak, the shorter comparable-period summaries respond more quickly to local changes while the 28-day rolling level provides a visibly smoother medium-run reference. Retaining both time scales therefore gives later models distinct signals for recent same-daypart conditions and a steadier local baseline, with the rolling values constructed strictly from observations preceding the forecast origin.

---

**Visualize short- versus medium-window comparable-period volatility.**

In [12]:
# 4.2.1.3C — Visualize comparable-period rolling volatility

if RUN_VISUALS:
    volatility_df = plot_df[
        ["origin_date", "same_daypart_roll_std_7d", "same_daypart_roll_std_28d"]
    ].copy()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=volatility_df["origin_date"], y=volatility_df["same_daypart_roll_std_7d"],
        mode="lines", name="Prior 7-day volatility",
        line=dict(color=BRAND_COLORS["terracotta"], width=2),
    ))
    fig.add_trace(go.Scatter(
        x=volatility_df["origin_date"], y=volatility_df["same_daypart_roll_std_28d"],
        mode="lines", name="Prior 28-day volatility",
        line=dict(color=BRAND_COLORS["dark_teal"], width=2),
    ))

    fig.update_layout(
        title=f"<b>Comparable-Period Mobility Volatility</b><br><sup>{TRACK_LABELS[PLOT_TRACK]} · {METRIC_LABELS[PLOT_METRIC]} · {plot_df['zone'].iloc[0]} · {PLOT_DAYPART}</sup>",
        width=PLOT_WIDTH, height=500, margin=dict(l=60, r=30, t=90, b=60),
        xaxis_title="Origin date", yaxis_title="Rolling standard deviation",
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.show()

**Findings.** In the same Baisley Park PM-peak series, the 7-day rolling standard deviation responds more sharply to short-lived changes in variability, while the 28-day measure moves more smoothly. The two windows therefore capture distinct short- and medium-run volatility states rather than simply duplicating the same information at different scales.

### 4.2.1.3 Summary

**Summary.** Same-daypart history adds a second temporal view alongside the immediate sequence lags: prior-day and prior-week comparisons preserve recurring time-of-day behavior, while shifted 7- and 28-day level and volatility summaries separate short-run movement from a steadier local baseline. The longer windows naturally have somewhat lower availability, but the executed outputs retain sufficient support to carry both time scales forward for later model testing.

---

# 4.2.1.4 Diagnose Leakage-Safe Seasonal Decomposition Structure

The target model still forecasts **one ordered Taxi Zone × Metric sequence**, but seasonality is easier to learn if we temporarily look at each daypart separately.

For example, one Taxi Zone × Metric forecasting sequence looks conceptually like:

`Monday Overnight → Monday AM Peak → Monday Midday → Monday PM Peak → Monday Evening → Tuesday Overnight → ...`

For decomposition, we temporarily pull out five regular daily subseries:

`Overnight: Monday → Tuesday → Wednesday → ...`  
`AM Peak: Monday → Tuesday → Wednesday → ...`  
`Midday: Monday → Tuesday → Wednesday → ...`  
`PM Peak: Monday → Tuesday → Wednesday → ...`  
`Evening: Monday → Tuesday → Wednesday → ...`

That lets STL ask a natural question such as **“Does this Monday PM Peak resemble previous Monday PM Peaks?”** A weekly cycle is simply seven daily observations rather than 35 interleaved sequence positions.

Crucially, this does **not** create five forecasting models. Each fitted component still keeps its Date and Daypart, so it can be merged straight back onto the original `observation_sequence_id`. After stitching, the forecasting model again sees one ordered Taxi Zone × Metric sequence.

Put differently:

> **Daypart is the grain at which we learn the seasonal process; Taxi Zone × Metric remains the grain at which we forecast mobility.**

The five daypart decompositions are therefore **five seasonal lenses for one forecasting sequence**.

### What is—and is not—carried forward from 3.3.1

3.3.1 established a useful methodological pattern: evaluate a compact family containing no seasonality, weekly STL, monthly-ish STL, annual STL, and dual weekly + annual MSTL; compare seasonal/trend strength and residual behavior; then choose a suitable decomposition at the individual Taxi Zone × Metric × Daypart series level.

We reuse that **procedure**, but none of its retrospectively learned results. Every candidate below is fitted again using only the history available before the first rolling-validation forecast for its modeling track. We also stop before applying 3.3.1's retrospective strength cutoffs or complexity penalty. The leakage-safe diagnostics come first; the forecasting production rule is a HITL decision afterward.

Candidate analysis may interpolate missing values **inside the already isolated initial-training window** so STL/MSTL can operate on a regular daily grid. Those interpolated values are diagnostic-only and are never exported as forecasting predictors. Production features later in this notebook will require genuinely causal construction.

---

**Freeze the training-only seasonal-diagnostic windows.** The cutoff is the information state available at the earliest rolling-validation origin for each track; no validation target outcome has yet been observed.

<img src="image-20260917-214602.png" width="" align="" />

In [13]:
# 4.2.1.4A — Freeze leakage-safe seasonal-diagnostic training windows

# Seasonal diagnosis and model-structure selection must stop at the information
# state available when rolling validation is about to begin. The cutoff is the
# first VALIDATION ORIGIN, not the first validation target date: this notebook
# is governed by what information is known when a forecast is issued.
training_window_rows = []

for track in TRACK_LABELS:
    cutoff_id = first_validation_origin_id(track)

    # Translate the canonical cutoff position back to its date/daypart so the
    # training-window contract remains inspectable by humans as well as by code.
    cutoff_row = sequence_calendar_df.loc[
        sequence_calendar_df["observation_sequence_id"].eq(cutoff_id)
    ].iloc[0]

    # History begins at the track-specific legal start established in 4.1.2.
    # In particular, Post-CP Primary cannot reach backward into Pre-CP mobility.
    start_date = training_history_start(track)

    training_window_rows.append({
        "modeling_track": track,
        "training_history_start_date": start_date,
        "first_validation_origin_sequence_id": cutoff_id,
        "first_validation_origin_date": cutoff_row["date"],
        "first_validation_origin_daypart": cutoff_row["daypart"],

        # This is the available CALENDAR span, not the number of observed values.
        # Per-series observed coverage is measured separately in 4.2.1.4B.
        "available_calendar_days": int((cutoff_row["date"] - start_date).days + 1),
    })

# One row per modeling track. This registry records the frozen legal window used
# for seasonal diagnosis; it contains no fitted decomposition components itself.
seasonal_training_window_registry_df = pd.DataFrame(training_window_rows)

# Keep stable machine-facing track IDs in the registry and map friendly labels
# only in the reader-facing display object.
seasonal_training_window_display_df = seasonal_training_window_registry_df.copy()
seasonal_training_window_display_df["modeling_track"] = (
    seasonal_training_window_display_df["modeling_track"].map(TRACK_LABELS)
)

display(seasonal_training_window_display_df)

,modeling_track,training_history_start_date,first_validation_origin_sequence_id,first_validation_origin_date,first_validation_origin_daypart,available_calendar_days
0,Post-CP Primary,2025-01-05,4121,2025-04-04,overnight,90
1,Full-History Regime-Aware,2023-01-01,4121,2025-04-04,overnight,825
2,Pre-CP Reference,2023-01-01,1821,2023-12-31,overnight,365


**Findings.** The leakage-safe seasonal-diagnostic training windows differ substantially across modeling tracks. Post-CP Primary has 90 calendar days of legal training history, from January 5 through April 4, 2025; Full-History Regime-Aware has 825 days, from January 1, 2023 through April 4, 2025; and Pre-CP Reference has 365 days, covering January 1 through December 31, 2023. In all three cases, the first validation origin falls in the overnight daypart. These frozen windows establish exactly how much pre-validation history is available for evaluating candidate seasonal structures without using validation outcomes.

---

**Build the training-only Taxi Zone × Metric × Daypart support registry and diagnostic sample.** Sampling keeps the initial scouting run practical without using any validation or holdout observations to choose representative series.

In [14]:
# 4.2.1.4B — Build leakage-safe daypart-series support and diagnostic samples

SEASONAL_SUPPORT_PATH = INTERMEDIATE_DIR / "seasonal_training_series_support.parquet"
SEASONAL_SAMPLE_PATH = INTERMEDIATE_DIR / "seasonal_diagnostic_series_sample.parquet"

def build_seasonal_support():
    """
    Build training-only support metadata for every native daypart series.

    Grain
    -----
    One output row per Modeling Track × Forecast Sequence × Daypart, equivalent
    to Taxi Zone × Metric × Daypart within a modeling track.

    Leakage rule
    ------------
    Each track's history is truncated at its first rolling-validation origin
    before any support statistic is calculated. Validation targets and holdout
    outcomes therefore cannot influence whether a seasonal candidate appears to
    have adequate training history.

    Support measures
    ----------------
    For each daily Daypart lens, the function records the calendar days that
    could have been observed, the days with a non-null mobility value, and the
    resulting observed share. `daypart_series_id` provides a stable identifier
    for the temporary daily seasonal series.

    These support fields are diagnostic metadata. They do not by themselves
    remove the underlying Forecast Sequence from the approved forecasting
    population.

    Returns
    -------
    pandas.DataFrame
        Training-only Daypart-series support registry across all modeling tracks.
    """
    frames = []

    for track in TRACK_LABELS:
        history = pd.read_parquet(track_path("track_history_source", track))
        cutoff = first_validation_origin_id(track)
        training = history.loc[history["observation_sequence_id"].le(cutoff)].copy()

        support = (
            training.groupby(
                ["forecast_sequence_id", "metric", "taxi_zone_id", "zone", "borough", "daypart"],
                as_index=False,
                observed=True,
            )
            .agg(
                possible_days=("date", "size"),
                observed_days=("observed_value", "count"),
                first_date=("date", "min"),
                last_date=("date", "max"),
            )
        )

        support["observed_share"] = support["observed_days"] / support["possible_days"]
        support["modeling_track"] = track
        support["daypart_series_id"] = support["forecast_sequence_id"].astype(str) + "__" + support["daypart"].astype(str)
        frames.append(support)

        del history, training
        gc.collect()

    return pd.concat(frames, ignore_index=True)


def build_seasonal_sample():
    """
    Draw a reproducible diagnostic sample from the training-only support registry.

    Sampling is stratified across Modeling Track × Metric × Daypart so the
    exploratory decomposition scan represents all target metrics and all five
    daily seasonal lenses without fitting every native series.

    The sampling decision uses support metadata only. No candidate-fit score,
    validation performance, holdout outcome, or later production-selection result
    can influence which series enters this diagnostic sample.

    This sample is used only for 4.2.1.4 seasonal scouting. Production weekly-STL
    assignment is subsequently evaluated across the full native eligible
    Daypart-series population in 4.2.1.5B.

    Returns
    -------
    pandas.DataFrame
        Reproducible set of Daypart-series support rows selected for diagnostics.
    """
    eligible = seasonal_training_series_support_df.loc[
        seasonal_training_series_support_df["observed_share"].ge(DIAGNOSTIC_MIN_OBS_SHARE)
        & seasonal_training_series_support_df["possible_days"].ge(SEASONAL_CANDIDATES["min_training_days"].min())
    ].copy()

    sampled = []

    for (track, metric, daypart), group in eligible.groupby(["modeling_track", "metric", "daypart"], sort=True):
        n = min(DIAGNOSTIC_SERIES_PER_METRIC_DAYPART, len(group))
        sampled.append(group.sample(n=n, random_state=DIAGNOSTIC_RANDOM_STATE).copy())

    return pd.concat(sampled, ignore_index=True).sort_values(
        ["modeling_track", "metric", "daypart", "forecast_sequence_id"]
    ).reset_index(drop=True)


seasonal_training_series_support_df = load_or_build(
    SEASONAL_SUPPORT_PATH, REBUILD_SEASONAL_SUPPORT, build_seasonal_support
)

seasonal_diagnostic_series_sample_df = load_or_build(
    SEASONAL_SAMPLE_PATH, REBUILD_SEASONAL_SAMPLE, build_seasonal_sample
)

seasonal_sample_summary_df = (
    seasonal_diagnostic_series_sample_df.groupby(["modeling_track", "metric"], as_index=False)
    .agg(
        sampled_daypart_series=("daypart_series_id", "nunique"),
        represented_dayparts=("daypart", "nunique"),
        median_training_coverage=("observed_share", "median"),
        minimum_training_coverage=("observed_share", "min"),
    )
)

seasonal_sample_summary_df["modeling_track"] = seasonal_sample_summary_df["modeling_track"].map(TRACK_LABELS)
seasonal_sample_summary_df["metric"] = seasonal_sample_summary_df["metric"].map(METRIC_LABELS)

display(
    seasonal_sample_summary_df.style.format({
        "median_training_coverage": "{:.3f}",
        "minimum_training_coverage": "{:.3f}",
    })
)

,modeling_track,metric,sampled_daypart_series,represented_dayparts,median_training_coverage,minimum_training_coverage
0,Full-History Regime-Aware,FHVHV speed,30,5,1.000,1.000
1,Full-History Regime-Aware,FHVHV trips,30,5,1.000,1.000
2,Full-History Regime-Aware,Subway ridership,30,5,1.000,0.899
3,Full-History Regime-Aware,Taxi speed,30,5,1.000,0.844
4,Full-History Regime-Aware,Taxi trips,30,5,1.000,1.000
5,Post-CP Primary,FHVHV speed,30,5,1.000,1.000
6,Post-CP Primary,FHVHV trips,30,5,1.000,1.000
7,Post-CP Primary,Subway ridership,30,5,1.000,1.000
8,Post-CP Primary,Taxi speed,30,5,1.000,0.809
9,Post-CP Primary,Taxi trips,30,5,1.000,1.000


**Findings.** The diagnostic sample is balanced across the seasonal-analysis grain: each Modeling Track × Metric combination contains 30 sampled Taxi Zone × Metric × Daypart series spanning all five dayparts. Training coverage is also high across the sampled series, so the subsequent candidate comparison is based on well-supported daily histories rather than sparse examples. This gives each metric and modeling track comparable daypart representation before STL/MSTL candidates are evaluated.

---

📝 **Why the next block is cached.** This is the first intentionally expensive scouting block in 4.2.1. Each sampled Taxi Zone × Metric × Daypart training series is placed on a regular daily grid and evaluated against every candidate decomposition it has enough history to support. The output contains diagnostic scores only; it does not yet declare a winning production model.

**Fit the leakage-safe candidate decomposition family.** Compare no-seasonality, weekly, monthly-ish, annual, and dual weekly + annual structures using seasonal strength, trend strength, and several residual-dispersion measures.

In [15]:
# 4.2.1.4C — Evaluate training-only STL/MSTL candidate decompositions

SEASONAL_DIAGNOSTICS_PATH = INTERMEDIATE_DIR / "seasonal_candidate_fit_diagnostics.parquet"


def prepare_daily_training_series(series_df):
    """
    Regularize one training-only Daypart series onto a daily calendar.

    The caller must already have isolated legal training history for one
    Forecast Sequence × Daypart. This helper sorts by date, creates the regular
    daily grid required by STL/MSTL, and handles internal missing observations
    only within that already-frozen training window.

    Any interpolation performed here exists solely to make decomposition
    diagnostics estimable. Interpolated mobility values are not exported as
    forecasting predictors and cannot extend beyond the legal training cutoff.

    Returns
    -------
    pandas.Series
        Daily float-valued training series indexed by date.
    """
    ts = (
        series_df[["date", "observed_value"]]
        .sort_values("date")
        .set_index("date")["observed_value"]
        .asfreq("D")
    )

    if ts.dropna().empty:
        return ts

    # Safe for model-selection diagnostics because every interpolated point lies
    # inside the already isolated initial-training window.
    return ts.interpolate(method="time", limit_direction="both")


def fit_seasonality_candidate(observed, candidate):
    """
    Fit one diagnostic seasonal candidate to an already-isolated training series.

    Supported candidates include the no-seasonality diagnostic baseline,
    single-period STL alternatives, and the dual weekly + annual diagnostic
    structure defined by the candidate registry.

    Every fit operates only on the supplied training series. This helper does
    not choose a production winner, apply validation results, or reuse fitted
    components from the earlier retrospective decomposition workflow.

    The returned components are used to measure training-window seasonal/trend
    strength and residual behavior. Longer-period candidates remain diagnostic
    evidence unless a later HITL production policy explicitly approves them.

    Returns
    -------
    dict
        Candidate decomposition components needed by the diagnostic summaries.
    """
    observed = pd.Series(observed, dtype=float)

    if candidate == "none":
        trend = observed.rolling(7, center=True, min_periods=1).median()
        seasonal = pd.Series(0.0, index=observed.index)
        residual = observed - trend

    elif candidate.startswith("single_"):
        period = int(candidate.split("_")[1])
        fit = STL(observed, period=period, robust=True).fit()
        trend = pd.Series(fit.trend, index=observed.index)
        seasonal = pd.Series(fit.seasonal, index=observed.index)
        residual = pd.Series(fit.resid, index=observed.index)

    elif candidate == "dual_7_365":
        fit = MSTL(observed, periods=(7, 365), stl_kwargs={"robust": True}).fit()
        seasonal_array = np.asarray(fit.seasonal, dtype=float)
        seasonal = pd.Series(
            seasonal_array.sum(axis=1) if seasonal_array.ndim == 2 else seasonal_array,
            index=observed.index,
        )
        trend = pd.Series(fit.trend, index=observed.index)
        residual = pd.Series(fit.resid, index=observed.index)

    else:
        raise ValueError(f"Unsupported decomposition candidate: {candidate}")

    return {
        "trend": trend,
        "seasonal": seasonal,
        "expected": trend + seasonal,
        "residual": residual,
    }


def build_seasonal_diagnostics():
    """
    Fit one diagnostic seasonal candidate to an already-isolated training series.

    Supported candidates include the no-seasonality diagnostic baseline,
    single-period STL alternatives, and the dual weekly + annual diagnostic
    structure defined by the candidate registry.

    Every fit operates only on the supplied training series. This helper does
    not choose a production winner, apply validation results, or reuse fitted
    components from the earlier retrospective decomposition workflow.

    The returned components are used to measure training-window seasonal/trend
    strength and residual behavior. Longer-period candidates remain diagnostic
    evidence unless a later HITL production policy explicitly approves them.

    Returns
    -------
    dict
        Candidate decomposition components needed by the diagnostic summaries.
    """
    records = []

    for track in TRACK_LABELS:
        print(f"Evaluating seasonal candidates: {TRACK_LABELS[track]}")

        history = pd.read_parquet(track_path("track_history_source", track))
        cutoff = first_validation_origin_id(track)

        sample = seasonal_diagnostic_series_sample_df.loc[
            seasonal_diagnostic_series_sample_df["modeling_track"].eq(track)
        ].copy()

        sample_sequence_ids = set(sample["forecast_sequence_id"])
        training = history.loc[
            history["forecast_sequence_id"].isin(sample_sequence_ids)
            & history["observation_sequence_id"].le(cutoff)
        ].copy()

        grouped = training.groupby(["forecast_sequence_id", "daypart"], sort=False, observed=True)

        for row in tqdm(sample.itertuples(index=False), total=len(sample), desc=TRACK_LABELS[track]):
            series_df = grouped.get_group((row.forecast_sequence_id, row.daypart))
            observed = prepare_daily_training_series(series_df)

            raw_var = float(np.nanvar(observed))
            raw_mad = robust_mad(observed)

            for candidate in SEASONAL_CANDIDATES.itertuples(index=False):
                base_record = {
                    "modeling_track": track,
                    "metric": row.metric,
                    "forecast_sequence_id": row.forecast_sequence_id,
                    "taxi_zone_id": row.taxi_zone_id,
                    "zone": row.zone,
                    "borough": row.borough,
                    "daypart": row.daypart,
                    "daypart_series_id": row.daypart_series_id,
                    "candidate_model": candidate.candidate_model,
                    "primary_period": candidate.primary_period,
                    "secondary_period": candidate.secondary_period,
                    "training_days": len(observed),
                    "observed_share": row.observed_share,
                }

                if len(observed) < candidate.min_training_days:
                    records.append({
                        **base_record,
                        "fit_status": "insufficient_history",
                        "seasonal_strength": np.nan,
                        "trend_strength": np.nan,
                        "residual_variance_ratio": np.nan,
                        "residual_std": np.nan,
                        "residual_mad_ratio": np.nan,
                        "residual_iqr": np.nan,
                    })
                    continue

                try:
                    components = fit_seasonality_candidate(observed, candidate.candidate_model)
                    residual = components["residual"]

                    resid_var = float(np.nanvar(residual))
                    resid_mad = robust_mad(residual)
                    resid_iqr = float(residual.quantile(0.75) - residual.quantile(0.25))

                    records.append({
                        **base_record,
                        "fit_status": "fit_ok",
                        "seasonal_strength": 0.0 if candidate.candidate_model == "none"
                            else component_strength(components["seasonal"], residual),
                        "trend_strength": component_strength(components["trend"], residual),
                        "residual_variance_ratio": resid_var / raw_var if raw_var > 0 else np.nan,
                        "residual_std": float(np.nanstd(residual)),
                        "residual_mad_ratio": resid_mad / raw_mad if raw_mad > 0 else np.nan,
                        "residual_iqr": resid_iqr,
                    })

                except Exception as exc:
                    records.append({
                        **base_record,
                        "fit_status": f"fit_failed: {type(exc).__name__}",
                        "seasonal_strength": np.nan,
                        "trend_strength": np.nan,
                        "residual_variance_ratio": np.nan,
                        "residual_std": np.nan,
                        "residual_mad_ratio": np.nan,
                        "residual_iqr": np.nan,
                    })

        del history, training
        gc.collect()

    return pd.DataFrame(records)


seasonal_candidate_fit_diagnostics_df = load_or_build(
    SEASONAL_DIAGNOSTICS_PATH, REBUILD_SEASONAL_DIAGNOSTICS, build_seasonal_diagnostics
)

fit_ok_df = seasonal_candidate_fit_diagnostics_df.loc[
    seasonal_candidate_fit_diagnostics_df["fit_status"].eq("fit_ok")
].copy()

seasonal_candidate_summary_df = (
    fit_ok_df.groupby(["modeling_track", "metric", "candidate_model"], as_index=False)
    .agg(
        fitted_series=("daypart_series_id", "nunique"),
        median_seasonal_strength=("seasonal_strength", "median"),
        median_trend_strength=("trend_strength", "median"),
        median_residual_variance_ratio=("residual_variance_ratio", "median"),
        median_residual_mad_ratio=("residual_mad_ratio", "median"),
    )
    .sort_values(["modeling_track", "metric", "candidate_model"])
    .reset_index(drop=True)
)

seasonal_candidate_summary_df["modeling_track"] = seasonal_candidate_summary_df["modeling_track"].map(TRACK_LABELS)
seasonal_candidate_summary_df["metric"] = seasonal_candidate_summary_df["metric"].map(METRIC_LABELS)

display(
    seasonal_candidate_summary_df.style.format({
        "median_seasonal_strength": "{:.3f}",
        "median_trend_strength": "{:.3f}",
        "median_residual_variance_ratio": "{:.3f}",
        "median_residual_mad_ratio": "{:.3f}",
    })
)

,modeling_track,metric,candidate_model,fitted_series,median_seasonal_strength,median_trend_strength,median_residual_variance_ratio,median_residual_mad_ratio
0,Full-History Regime-Aware,FHVHV speed,dual_7_365,30,0.999,0.867,0.001,0.014
1,Full-History Regime-Aware,FHVHV speed,none,30,0.000,0.166,0.834,0.759
2,Full-History Regime-Aware,FHVHV speed,single_30,30,0.091,0.115,0.794,0.696
3,Full-History Regime-Aware,FHVHV speed,single_365,30,0.999,0.907,0.001,0.017
4,Full-History Regime-Aware,FHVHV speed,single_7,30,0.568,0.336,0.354,0.297
5,Full-History Regime-Aware,FHVHV trips,dual_7_365,30,0.999,0.963,0.001,0.010
6,Full-History Regime-Aware,FHVHV trips,none,30,0.000,0.153,0.847,0.750
7,Full-History Regime-Aware,FHVHV trips,single_30,30,0.046,0.117,0.862,0.679
8,Full-History Regime-Aware,FHVHV trips,single_365,30,0.999,0.961,0.001,0.013
9,Full-History Regime-Aware,FHVHV trips,single_7,30,0.721,0.405,0.228,0.216


**Findings.** The training-only candidate results show that seasonal-model eligibility differs materially by modeling track. Full-History Regime-Aware has enough pre-validation history to evaluate the full candidate family, including annual and dual weekly-plus-annual structures, while Pre-CP Reference is limited to the no-seasonality, 7-day, and 30-day candidates. Post-CP Primary is more constrained still: the no-seasonality and 7-day candidates can be evaluated broadly, while the 30-day candidate reaches only the subset of sampled series with the full required training length. The diagnostic scores also show meaningful differences among candidates—for example, Pre-CP Taxi trips have median seasonal strength of 0.157 and residual-variance ratio of 0.816 under the 7-day candidate, compared with seasonal strength of 0.000 and residual-variance ratio of 1.000 under the 30-day candidate. These are candidate-fit diagnostics only; this block does not select a production decomposition rule.

---

**Compare candidate behavior across dayparts.** This visualization preserves the Taxi Zone × Metric × Daypart diagnostic grain so we can see whether the same candidate behaves consistently throughout the day or whether seasonality is materially daypart-specific.

In [16]:
# 4.2.1.4D — Visualize training-safe seasonal evidence by daypart

if RUN_VISUALS:
    candidate_daypart_summary_df = (
        fit_ok_df.loc[
            fit_ok_df["modeling_track"].eq(PLOT_TRACK)
            & fit_ok_df["metric"].eq(PLOT_METRIC)
        ]
        .groupby(["daypart", "candidate_model"], as_index=False)
        .agg(
            median_seasonal_strength=("seasonal_strength", "median"),
            median_residual_variance_ratio=("residual_variance_ratio", "median"),
        )
    )

    candidate_daypart_summary_df["daypart"] = pd.Categorical(
        candidate_daypart_summary_df["daypart"], categories=DAYPART_ORDER, ordered=True
    )
    candidate_daypart_summary_df = candidate_daypart_summary_df.sort_values(["daypart", "candidate_model"])

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Median seasonal strength", "Median residual variance / raw variance"],
        horizontal_spacing=0.12,
    )

    candidates = candidate_daypart_summary_df["candidate_model"].drop_duplicates().tolist()

    for i, candidate in enumerate(candidates):
        plot_candidate = candidate_daypart_summary_df.loc[
            candidate_daypart_summary_df["candidate_model"].eq(candidate)
        ]
        color = BRAND_COLOR_SEQUENCE[i % len(BRAND_COLOR_SEQUENCE)]

        fig.add_trace(
            go.Bar(
                x=plot_candidate["daypart"], y=plot_candidate["median_seasonal_strength"],
                name=candidate, marker_color=color, legendgroup=candidate,
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Bar(
                x=plot_candidate["daypart"], y=plot_candidate["median_residual_variance_ratio"],
                name=candidate, marker_color=color, legendgroup=candidate, showlegend=False,
            ),
            row=1, col=2,
        )

    fig.update_layout(
        title=f"<b>Leakage-Safe Seasonal Candidate Comparison</b><br><sup>{TRACK_LABELS[PLOT_TRACK]} · {METRIC_LABELS[PLOT_METRIC]}</sup>",
        width=PLOT_WIDTH, height=540, barmode="group",
        margin=dict(l=60, r=30, t=100, b=80),
        legend_title_text="Candidate",
        plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_yaxes(title_text="Seasonal strength", row=1, col=1)
    fig.update_yaxes(title_text="Residual variance ratio", row=1, col=2)
    fig.show()

**Findings.** For Post-CP Primary Taxi trips, the daypart comparison shows that the 7-day candidate can be evaluated across the full daily cycle, whereas the 30-day candidate appears only where the short initial training window provides enough history. The chart also shows that seasonal strength and residual-variance reduction are not identical across dayparts, supporting the decision to preserve Taxi Zone × Metric × Daypart as the grain for learning seasonal structure rather than assuming one identical decomposition behavior throughout the day.

---

### A concrete split-and-stitch example

The next visualization deliberately stays at one daypart while decomposition is being learned. That is only the temporary analytical view.

If we fit PM Peak separately, its output still contains the date **and the fact that it is PM Peak**. Later, that component joins back to the PM Peak row for the corresponding `observation_sequence_id`. Overnight, AM Peak, Midday, and Evening do the same thing.

So a final forecasting sequence might again look like:

`Overnight component → AM Peak component → Midday component → PM Peak component → Evening component → next Overnight ...`

Those values may have come from five separate daily decomposition fits, but once stitched back together they are simply consecutive features attached to the same **Taxi Zone × Metric forecasting sequence**.

This is why we can allow seasonality to differ by daypart without changing the actual forecasting grain.

**Inspect one training-safe decomposition directly.** The candidate is controlled by `PLOT_SEASONAL_CANDIDATE`; changing the toggle reruns only this lightweight visualization, not the cached candidate scan.

In [17]:
# 4.2.1.4E — Visualize one training-safe decomposition candidate

if RUN_VISUALS:
    history = pd.read_parquet(track_path("track_history_source", PLOT_TRACK))
    sequence_id = select_training_safe_sequence(history, PLOT_TRACK, PLOT_METRIC, PLOT_DAYPART)
    cutoff = first_validation_origin_id(PLOT_TRACK)

    example_df = history.loc[
        history["forecast_sequence_id"].eq(sequence_id)
        & history["daypart"].eq(PLOT_DAYPART)
        & history["observation_sequence_id"].le(cutoff),
        ["date", "observed_value", "zone"],
    ].sort_values("date")

    observed = prepare_daily_training_series(example_df)
    candidate_spec = SEASONAL_CANDIDATES.loc[
        SEASONAL_CANDIDATES["candidate_model"].eq(PLOT_SEASONAL_CANDIDATE)
    ].iloc[0]

    if len(observed) < candidate_spec["min_training_days"]:
        raise ValueError(
            f"{PLOT_SEASONAL_CANDIDATE} requires at least {candidate_spec['min_training_days']} training days; "
            f"this leakage-safe window contains {len(observed)}."
        )

    components = fit_seasonality_candidate(observed, PLOT_SEASONAL_CANDIDATE)

    decomposition_df = pd.DataFrame({
        "date": observed.index,
        "observed": observed.to_numpy(),
        "expected": components["expected"].to_numpy(),
        "trend": components["trend"].to_numpy(),
        "seasonal": components["seasonal"].to_numpy(),
        "residual": components["residual"].to_numpy(),
    }).tail(PLOT_DECOMPOSITION_DAYS)

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=["Observed vs expected", "Trend", "Seasonal component", "Residual"],
    )

    fig.add_trace(go.Scatter(
        x=decomposition_df["date"], y=decomposition_df["observed"], mode="lines",
        name="Observed", line=dict(color=BRAND_COLORS["dark_teal"], width=2)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=decomposition_df["date"], y=decomposition_df["expected"], mode="lines",
        name="Expected", line=dict(color=BRAND_COLORS["terracotta"], width=2)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=decomposition_df["date"], y=decomposition_df["trend"], mode="lines",
        name="Trend", line=dict(color=BRAND_COLORS["dark_teal"], width=1.8), showlegend=False
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=decomposition_df["date"], y=decomposition_df["seasonal"], mode="lines",
        name="Seasonal", line=dict(color=BRAND_COLORS["seafoam"], width=1.8), showlegend=False
    ), row=3, col=1)

    fig.add_trace(go.Scatter(
        x=decomposition_df["date"], y=decomposition_df["residual"], mode="lines",
        name="Residual", line=dict(color=BRAND_COLORS["terracotta"], width=1.5), showlegend=False
    ), row=4, col=1)

    fig.add_hline(y=0, line_dash="dot", line_color="#888888", row=4, col=1)

    fig.update_layout(
        title=f"<b>Training-Safe Seasonal Decomposition</b><br><sup>{TRACK_LABELS[PLOT_TRACK]} · {METRIC_LABELS[PLOT_METRIC]} · {example_df['zone'].iloc[0]} · {PLOT_DAYPART} · {PLOT_SEASONAL_CANDIDATE}</sup>",
        width=PLOT_WIDTH, height=660, margin=dict(l=65, r=30, t=100, b=50),
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.show()

**Findings.** The representative Post-CP Primary Taxi-trip series for Baisley Park PM peak shows how the selected single_7 candidate separates the training-window observations into expected, trend, weekly seasonal, and residual components. The expected series follows the broader movement in the observed values, while the seasonal and residual panels separate repeating short-cycle structure from departures left unexplained by the fitted trend and seasonality. This is a visual inspection of one training-safe candidate fit, not evidence that single_7 should automatically be selected for every series or modeling track.

Visualize a deliberately strong weekly\-seasonal example\. Rather than choosing a geography in advance, this diagnostic uses the training\-only results from 4\.2\.1\.4C to identify the Post\-CP Primary sampled series with the highest fitted 7\-day seasonal strength\. It then visualizes that same leakage\-safe decomposition so we can see what a strong weekly signal looks like in one of the actual forecasting targets\. This is an illustrative extreme from the diagnostic sample, not a production\-selection rule\.

In [18]:
# 4.2.1.4F — Visualize a strongly weekly-seasonal forecasting series

if RUN_VISUALS:
    # Rank only successfully fitted 7-day STL candidates from the Post-CP
    # Primary diagnostic sample. This reuses 4C's training-only evidence;
    # no new candidate search and no validation-period outcomes are introduced.
    strong_weekly_candidates = (
        seasonal_candidate_fit_diagnostics_df.loc[
            seasonal_candidate_fit_diagnostics_df["modeling_track"].eq("post_cp_primary")
            & seasonal_candidate_fit_diagnostics_df["candidate_model"].eq("single_7")
            & seasonal_candidate_fit_diagnostics_df["fit_status"].eq("fit_ok")
            & seasonal_candidate_fit_diagnostics_df["seasonal_strength"].notna()
        ]
        .sort_values(["seasonal_strength", "residual_variance_ratio"], ascending=[False, True])
        .reset_index(drop=True)
    )

    if strong_weekly_candidates.empty:
        raise ValueError("No fitted Post-CP Primary single_7 diagnostic series are available.")

    # Use the strongest weekly-seasonal series already evaluated in 4C.
    selected = strong_weekly_candidates.iloc[0]
    selected_sequence_id = selected["forecast_sequence_id"]
    selected_metric = selected["metric"]
    selected_zone = selected["zone"]
    selected_daypart = selected["daypart"]
    selected_strength = float(selected["seasonal_strength"])
    selected_residual_ratio = float(selected["residual_variance_ratio"])

    history = pd.read_parquet(track_path("track_history_source", "post_cp_primary"))
    cutoff = first_validation_origin_id("post_cp_primary")

    # Reconstruct exactly the same Taxi Zone × Metric × Daypart daily series
    # using only observations available through the first validation origin.
    example_df = history.loc[
        history["forecast_sequence_id"].eq(selected_sequence_id)
        & history["daypart"].eq(selected_daypart)
        & history["observation_sequence_id"].le(cutoff),
        ["date", "observed_value"],
    ].sort_values("date")

    observed = prepare_daily_training_series(example_df)
    components = fit_seasonality_candidate(observed, "single_7")

    decomposition_df = pd.DataFrame({
        "date": observed.index,
        "observed": observed.to_numpy(),
        "expected": components["expected"].to_numpy(),
        "trend": components["trend"].to_numpy(),
        "seasonal": components["seasonal"].to_numpy(),
        "residual": components["residual"].to_numpy(),
    }).tail(PLOT_DECOMPOSITION_DAYS)

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=["Observed vs expected", "Trend", "7-day seasonal component", "Residual"],
    )

    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["observed"],
                   mode="lines", name="Observed",
                   line=dict(color=BRAND_COLORS["dark_teal"], width=2.2)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["expected"],
                   mode="lines", name="Expected",
                   line=dict(color=BRAND_COLORS["terracotta"], width=2.0)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["trend"],
                   mode="lines", name="Trend",
                   line=dict(color=BRAND_COLORS["dark_teal"], width=2.0),
                   showlegend=False),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["seasonal"],
                   mode="lines", name="Seasonal",
                   line=dict(color=BRAND_COLORS["seafoam"], width=2.0),
                   showlegend=False),
        row=3, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["residual"],
                   mode="lines", name="Residual",
                   line=dict(color=BRAND_COLORS["terracotta"], width=1.8),
                   showlegend=False),
        row=4, col=1,
    )

    fig.add_hline(y=0, row=4, col=1, line_dash="dot", line_color="#888888")

    metric_label = METRIC_LABELS.get(selected_metric, selected_metric)

    fig.update_layout(
        title=(
            "<b>Strong Weekly Seasonality Example</b><br>"
            f"<sup>Post-CP Primary · {metric_label} · {selected_zone} · {selected_daypart} · "
            f"seasonal strength {selected_strength:.3f} · residual variance ratio {selected_residual_ratio:.3f}</sup>"
        ),
        width=PLOT_WIDTH, height=660,
        margin=dict(l=65, r=30, t=100, b=50),
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )

    fig.show()

Findings\. The strongest sampled weekly\-seasonality example is Post\-CP Primary FHVHV trips in Greenwich Village South during the overnight daypart, with seasonal strength 0\.974 and a residual\-variance ratio of 0\.026\. The fitted expected series closely tracks the observed values, while the 7\-day seasonal component captures a pronounced and highly repeatable weekly cycle with large recurring peaks and troughs\. The trend varies much more gradually, and the residuals remain near zero for most dates aside from a small number of isolated deviations\. This provides a clear example of a series where weekly seasonality explains most of the short\-run structure and a 7\-day decomposition is substantively meaningful rather than merely fitting noise\.

Inspect a series where annual structure adds the most beyond weekly seasonality\. This diagnostic pairs the training\-only 7\-day and 365\-day fits for the same Full\-History Taxi Zone × Metric × Daypart series, then selects the sampled series where the annual candidate produces the largest additional reduction in residual variance\. The full training window is shown so that repeated annual behavior can be judged visually\. This is diagnostic evidence only; it does not promote annual STL into the production feature set\.

In [19]:
# 4.2.1.4G — Visualize a strong incremental annual-seasonality example

if RUN_VISUALS:
    # Compare annual and weekly fits only where both candidates were successfully
    # evaluated on the exact same Full-History Taxi Zone × Metric × Daypart series.
    annual = seasonal_candidate_fit_diagnostics_df.loc[
        seasonal_candidate_fit_diagnostics_df["modeling_track"].eq("full_history_regime_aware")
        & seasonal_candidate_fit_diagnostics_df["candidate_model"].eq("single_365")
        & seasonal_candidate_fit_diagnostics_df["fit_status"].eq("fit_ok")
    ].copy()

    weekly = seasonal_candidate_fit_diagnostics_df.loc[
        seasonal_candidate_fit_diagnostics_df["modeling_track"].eq("full_history_regime_aware")
        & seasonal_candidate_fit_diagnostics_df["candidate_model"].eq("single_7")
        & seasonal_candidate_fit_diagnostics_df["fit_status"].eq("fit_ok"),
        ["forecast_sequence_id", "daypart", "seasonal_strength", "residual_variance_ratio"],
    ].rename(columns={
        "seasonal_strength": "weekly_seasonal_strength",
        "residual_variance_ratio": "weekly_residual_variance_ratio",
    })

    paired = annual.merge(weekly, on=["forecast_sequence_id", "daypart"], how="inner", validate="one_to_one")
    paired["annual_variance_gain"] = paired["weekly_residual_variance_ratio"] - paired["residual_variance_ratio"]
    paired = paired.dropna(subset=["annual_variance_gain", "seasonal_strength"]).sort_values(
        ["annual_variance_gain", "seasonal_strength"], ascending=[False, False]
    ).reset_index(drop=True)

    if paired.empty:
        raise ValueError("No Full-History series has successful paired single_7 and single_365 diagnostics.")

    selected = paired.iloc[0]
    selected_sequence_id = selected["forecast_sequence_id"]
    selected_metric = selected["metric"]
    selected_zone = selected["zone"]
    selected_daypart = selected["daypart"]

    history = pd.read_parquet(track_path("track_history_source", "full_history_regime_aware"))
    cutoff = first_validation_origin_id("full_history_regime_aware")

    example_df = history.loc[
        history["forecast_sequence_id"].eq(selected_sequence_id)
        & history["daypart"].eq(selected_daypart)
        & history["observation_sequence_id"].le(cutoff),
        ["date", "observed_value"],
    ].sort_values("date")

    observed = prepare_daily_training_series(example_df)
    components = fit_seasonality_candidate(observed, "single_365")

    # Keep the entire leakage-safe training window: an annual diagnostic is not
    # interpretable from a short tail because we need to see repeated yearly behavior.
    decomposition_df = pd.DataFrame({
        "date": observed.index,
        "observed": observed.to_numpy(),
        "expected": components["expected"].to_numpy(),
        "trend": components["trend"].to_numpy(),
        "seasonal": components["seasonal"].to_numpy(),
        "residual": components["residual"].to_numpy(),
    })

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=["Observed vs expected", "Trend", "365-day seasonal component", "Residual"],
    )

    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["observed"], mode="lines",
                   name="Observed", line=dict(color=BRAND_COLORS["dark_teal"], width=1.8)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["expected"], mode="lines",
                   name="Expected", line=dict(color=BRAND_COLORS["terracotta"], width=1.8)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["trend"], mode="lines",
                   name="Trend", line=dict(color=BRAND_COLORS["dark_teal"], width=1.8), showlegend=False),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["seasonal"], mode="lines",
                   name="Annual seasonal", line=dict(color=BRAND_COLORS["seafoam"], width=1.8), showlegend=False),
        row=3, col=1,
    )
    fig.add_trace(
        go.Scatter(x=decomposition_df["date"], y=decomposition_df["residual"], mode="lines",
                   name="Residual", line=dict(color=BRAND_COLORS["terracotta"], width=1.5), showlegend=False),
        row=4, col=1,
    )

    fig.add_hline(y=0, row=4, col=1, line_dash="dot", line_color="#888888")

    metric_label = METRIC_LABELS.get(selected_metric, selected_metric)

    fig.update_layout(
        title=(
            "<b>Incremental Annual Seasonality Example</b><br>"
            f"<sup>Full-History Regime-Aware · {metric_label} · {selected_zone} · {selected_daypart} · "
            f"annual strength {selected['seasonal_strength']:.3f} · "
            f"weekly residual ratio {selected['weekly_residual_variance_ratio']:.3f} → "
            f"annual {selected['residual_variance_ratio']:.3f}</sup>"
        ),
        width=PLOT_WIDTH, height=660,
        margin=dict(l=65, r=30, t=100, b=50),
        legend_title_text="", plot_bgcolor="white", paper_bgcolor="white",
    )

    fig.show()

Findings\. The strongest apparent incremental annual\-seasonality example occurs for Full\-History Taxi trips in South Beach/Dongan Hills during the evening, where the 365\-day candidate reports seasonal strength of 0\.998 and reduces the residual\-variance ratio from 1\.331 under the weekly candidate to 0\.002\. Visual inspection, however, shows that this near\-perfect fit is driven by a very sparse series whose annual seasonal component closely reproduces individual low\-count spikes rather than revealing a clearly repeating yearly mobility cycle\. With only a little more than two annual cycles available in the leakage\-safe training window, the result demonstrates that extremely high annual seasonal strength and in\-sample variance reduction can reflect an overly flexible decomposition rather than robust forecastable annual structure\.

> 🔶 Decision — Seasonal Decomposition Policy

The training\-safe diagnostics above were reviewed before production seasonal features were constructed\. The resulting policy deliberately keeps the forecasting feature set simpler than the retrospective decomposition framework used in 3\.3\.1\.

🔹 Series\-level rule\. Seasonal structure will continue to be learned at the Taxi Zone × Metric × Daypart level, because the diagnostics show meaningful variation among individual mobility series\. Production selection is limited to single\_7 or none rather than allowing each series to choose among the full diagnostic candidate family\.

🔹 Weekly\-seasonality qualification\. A series receives single\_7 when the fit succeeds, at least 42 training days are available, observed coverage is at least 80%, and training\-window seasonal strength is at least 0\.35\. Otherwise, no STL\-derived seasonal feature is used\. Residual\-dispersion measures remain diagnostic checks rather than additional hard selection thresholds, and no complexity penalty is needed for this two\-option rule\.

🔹 Longer seasonal structures\. The 30\-day, 365\-day, and dual 7\+365 candidates remain diagnostic only\. In particular, visual inspection showed that extremely strong annual decomposition scores can arise when a flexible 365\-day component closely reproduces sparse historical observations rather than revealing a stable repeating annual pattern\. Longer known calendar cycles will therefore be represented separately using deterministic Fourier features\.

🔹 Rolling\-update policy\. The single\_7 versus none assignment is selected from each track's initial legal training window and then held fixed\. Qualifying weekly decompositions are refreshed monthly using only expanding history available before the refresh point\. This avoids the cost of refitting at every forecast origin while still allowing the estimated weekly pattern to evolve\.

🔹 Pre\-CP counterfactual safeguard\. Pre\-CP Reference may refresh its seasonal representation only while observations remain legally pre\-CP\. After the final refit using data through January 4, 2025, its seasonal representation is frozen so that actual post\-CP mobility can never enter the no\-CP counterfactual feature history\.

Decision outcome: production decomposition will therefore use training\-selected weekly STL where weekly seasonality is sufficiently supported, otherwise no STL decomposition, with longer calendar periodicity handled by Fourier features\.

### Section Summary

The seasonal diagnostics established a forecasting-safe way to learn recurring structure without changing the canonical forecasting grain. **Daypart is the grain at which we learn the seasonal process; Taxi Zone × Metric remains the grain at which we forecast mobility.** Training-only candidate fits showed that strong weekly structure can be substantively useful, while highly flexible longer-period decompositions can look deceptively strong in-sample. The resolved production policy therefore keeps only a guarded 7-day STL option, falls back to no STL when weekly structure is not convincing, freezes each series' model assignment after the initial legal training window, and delegates longer calendar cycles to deterministic Fourier features.

## 4.2.1.5 Prepare Production Seasonal and Cyclical Features

The diagnostic phase answered **which seasonal structure we are willing to trust**. This section turns that decision into an executable production plan before we launch the more expensive rolling STL build.

There are four jobs here.

First, apply the approved `single_7` versus `none` rule to **every native Taxi Zone × Metric × Daypart series**, not merely the diagnostic sample. Second, define the legal monthly refresh points at which an enabled weekly fit may learn from newly available history. Third, benchmark that weekly-only workload using late-history fits so we know what the full build will cost before committing to it. Finally, construct the annual Fourier basis, which is deterministic from the calendar and therefore does not require a fitted mobility model at all.

The STL assignment remains a property of the Daypart-level seasonal lens, while the eventual forecasting features will be stitched back to canonical Taxi Zone × Metric sequence positions.

By the next HITL gate, we will have four concrete objects:

- `weekly_stl_selection_registry_df` — the frozen `single_7` / `none` assignment for every native daypart series;
- `weekly_stl_refresh_calendar_df` — the legal monthly refit schedule for each modeling track;
- `weekly_stl_runtime_benchmark_df` — measured weekly-STL refresh costs on a controlled late-history sample;
- `annual_fourier_calendar_df` — the future-known annual cyclical representation for every canonical sequence position.

We will stop there before materializing the full monthly STL feature surface.

**Lock the production seasonal and cyclical policy.** These constants translate the resolved HITL decision into executable rules. Weekly STL remains selective and training-fitted; annual Fourier structure is deterministic and requires no mobility-outcome fitting.

In [20]:
# 4.2.1.5A — Lock the production seasonal and cyclical policy

# Weekly STL is intentionally narrow: the initial legal training window decides
# whether a Taxi Zone × Metric × Daypart series uses weekly STL at all.
WEEKLY_STL_PERIOD = 7
WEEKLY_STL_MIN_TRAINING_DAYS = 42
WEEKLY_STL_MIN_OBS_SHARE = 0.80
WEEKLY_STL_STRENGTH_THRESHOLD = 0.35

# Once assigned, the model class stays fixed. Qualifying weekly fits may refresh
# approximately monthly using expanding legal history.
WEEKLY_STL_REFRESH_MONTHS = 1
WEEKLY_STL_BENCHMARK_SERIES_PER_METRIC_DAYPART = 2
WEEKLY_STL_BENCHMARK_RANDOM_STATE = 696

# Longer calendar structure is represented deterministically rather than with a
# fitted 365-day STL model. One sine/cosine pair keeps this feature family minimal.
ANNUAL_FOURIER_PERIOD_DAYS = 365.2425

WEEKLY_SELECTION_PATH = INTERMEDIATE_DIR / "weekly_stl_selection_registry.parquet"
WEEKLY_REFRESH_CALENDAR_PATH = INTERMEDIATE_DIR / "weekly_stl_refresh_calendar.parquet"
WEEKLY_RUNTIME_BENCHMARK_PATH = INTERMEDIATE_DIR / "weekly_stl_refresh_runtime_benchmark.parquet"
ANNUAL_FOURIER_PATH = INTERMEDIATE_DIR / "annual_fourier_calendar.parquet"

# The Pre-CP Reference track inherits its final legal fit boundary directly from
# the 4.1.2 refit contract rather than duplicating those dates by hand.
pre_cp_refit_row = pre_cp_refit_application_contract_df.loc[
    pre_cp_refit_application_contract_df["modeling_track"].eq("pre_cp_reference")
].iloc[0]

PRE_CP_REFIT_ONLY_START_DATE = pd.Timestamp(pre_cp_refit_row["refit_only_start_date"])
PRE_CP_SEASONAL_FREEZE_DATE = pd.Timestamp(pre_cp_refit_row["final_refit_end_date"])
PRE_CP_SEASONAL_FREEZE_ID = int(pre_cp_refit_row["final_refit_end_observation_sequence_id"])

**Assign the optional weekly-STL feature across the already-approved forecasting population.** This block starts from each modeling track's native eligible Forecast Sequences and asks one narrower question for each Taxi Zone × Metric × Daypart lens: should this valid forecasting series receive a fitted 7-day STL seasonal feature?

To answer that question without leakage, the block uses only history available at the track's first rolling-validation origin. A daypart series receives single_7 when at least 42 calendar days of legal training history are available, observed coverage is at least 80%, the 7-day STL fit succeeds, and weekly seasonal strength is at least 0.35. If those conditions are not met, it receives none — but it remains fully eligible for forecasting and continues with the other leakage-safe features.

The single_7 versus none assignment is frozen after this initial training-only decision. Later monthly refreshes may update the fitted weekly pattern for a single_7 series, but they do not reopen forecasting eligibility or switch the seasonal-model class.

The output is weekly_stl_selection_registry_df, with one frozen weekly-STL assignment for every native eligible Taxi Zone × Metric × Daypart series. The registry also preserves whether each Forecast Sequence belongs to the shared Post-CP comparison cohort or the 1,029-sequence cross-regime comparison cohort defined in 4.1.2.

In [21]:
# 4.2.1.5B — Assign optional weekly STL across native eligible series

def build_weekly_stl_selection_registry():
    """
    Assign frozen `single_7` versus `none` seasonal models to native daypart series.

    Selection grain
    ---------------
    Modeling Track × Forecast Sequence × Daypart.

    Forecasting population
    ----------------------
    The function starts from the native forecasting population already approved
    for each modeling track. Weekly-STL selection is an optional feature decision;
    assigning `none` does not make a Forecast Sequence ineligible for forecasting.

    Leakage rule
    ------------
    Support and weekly seasonal strength are evaluated only through the track's
    first rolling-validation origin. Validation and holdout outcomes therefore
    cannot influence the frozen seasonal-model class.

    Production rule
    ---------------
    `single_7` is enabled only when:
    - legal training history contains at least WEEKLY_STL_MIN_TRAINING_DAYS;
    - observed training share meets WEEKLY_STL_MIN_OBS_SHARE;
    - the 7-day STL fit succeeds; and
    - weekly seasonal strength meets WEEKLY_STL_STRENGTH_THRESHOLD.

    Otherwise the series receives `none`. Residual-variance diagnostics are
    retained for QA but do not add another production threshold.

    Frozen assignment
    -----------------
    Later refreshes may update the fitted weekly seasonal representation for a
    `single_7` series, but they cannot switch the series between `single_7` and
    `none`.

    Provenance
    ----------
    Shared-comparison and cross-regime cohort flags are carried forward as
    metadata only; those restrictive comparison cohorts do not define native
    forecasting eligibility.

    Returns
    -------
    pandas.DataFrame
        One frozen weekly-seasonality assignment per native Daypart series.
    """
    records = []

    for track in TRACK_LABELS:
        cutoff_id = first_validation_origin_id(track)
        cutoff_date = sequence_calendar_df.loc[
            sequence_calendar_df["observation_sequence_id"].eq(cutoff_id), "date"
        ].iloc[0]

        support = seasonal_training_series_support_df.loc[
            seasonal_training_series_support_df["modeling_track"].eq(track)
        ].copy()

        # Rebuild exactly the initial-training information state used for model
        # selection. Validation and holdout outcomes cannot enter these fits.
        history = pd.read_parquet(track_path("track_history_source", track))
        training = history.loc[history["observation_sequence_id"].le(cutoff_id)].copy()
        grouped = training.groupby(["forecast_sequence_id", "daypart"], observed=True, sort=False)

        for row in tqdm(support.itertuples(index=False), total=len(support),
                        desc=f"{TRACK_LABELS[track]} weekly STL selection"):
            base = {
                "modeling_track": track, "forecast_sequence_id": row.forecast_sequence_id,
                "daypart_series_id": row.daypart_series_id, "taxi_zone_id": row.taxi_zone_id,
                "zone": row.zone, "borough": row.borough, "metric": row.metric, "daypart": row.daypart,
                "selection_cutoff_date": cutoff_date,
                "selection_cutoff_observation_sequence_id": cutoff_id,
                "training_possible_days": int(row.possible_days),
                "training_observed_days": int(row.observed_days),
                "training_observed_share": float(row.observed_share),
                "in_post_cp_common_comparison_cohort": row.forecast_sequence_id in post_cp_common_comparison_sequence_ids,
                "in_cross_regime_comparison_cohort": row.forecast_sequence_id in cross_regime_comparison_sequence_ids,
            }

            # Failing either support rule means STL is never attempted. "none"
            # is a deliberate production assignment rather than a missing result.
            if row.possible_days < WEEKLY_STL_MIN_TRAINING_DAYS:
                records.append({
                    **base, "weekly_fit_status": "not_attempted_insufficient_history",
                    "selection_status": "insufficient_history", "weekly_seasonal_strength": np.nan,
                    "weekly_residual_variance_ratio": np.nan, "weekly_fit_seconds": np.nan,
                    "selected_seasonal_model": "none", "weekly_stl_enabled": False,
                })
                continue

            if pd.isna(row.observed_share) or row.observed_share < WEEKLY_STL_MIN_OBS_SHARE:
                records.append({
                    **base, "weekly_fit_status": "not_attempted_low_coverage",
                    "selection_status": "low_coverage", "weekly_seasonal_strength": np.nan,
                    "weekly_residual_variance_ratio": np.nan, "weekly_fit_seconds": np.nan,
                    "selected_seasonal_model": "none", "weekly_stl_enabled": False,
                })
                continue

            key = (row.forecast_sequence_id, row.daypart)
            if key not in grouped.groups:
                raise KeyError(f"Training history is missing supported daypart series {key} for {track}.")

            # Interpolation remains confined to the already-isolated legal
            # initial-training window. It supports fitting/selection only and is
            # never exported as a forecasting predictor.
            series_df = grouped.get_group(key)
            observed = prepare_daily_training_series(series_df)
            fit_start = perf_counter()

            try:
                components = fit_seasonality_candidate(observed, "single_7")
                fit_seconds = perf_counter() - fit_start
                residual = components["residual"]

                weekly_strength = float(component_strength(components["seasonal"], residual))
                raw_variance = float(np.nanvar(observed.to_numpy(dtype=float)))
                residual_variance = float(np.nanvar(residual.to_numpy(dtype=float)))
                residual_ratio = residual_variance / raw_variance if raw_variance > 0 else np.nan

                enabled = bool(
                    np.isfinite(weekly_strength)
                    and weekly_strength >= WEEKLY_STL_STRENGTH_THRESHOLD
                )

                records.append({
                    **base, "weekly_fit_status": "fit_ok",
                    "selection_status": (
                        "weekly_seasonality_enabled"
                        if enabled else "weak_or_undefined_weekly_seasonality"
                    ),
                    "weekly_seasonal_strength": weekly_strength,
                    "weekly_residual_variance_ratio": residual_ratio,
                    "weekly_fit_seconds": fit_seconds,
                    "selected_seasonal_model": "single_7" if enabled else "none",
                    "weekly_stl_enabled": enabled,
                })

            except Exception as exc:
                fit_seconds = perf_counter() - fit_start
                records.append({
                    **base, "weekly_fit_status": f"fit_failed: {type(exc).__name__}",
                    "selection_status": "weekly_fit_failed", "weekly_seasonal_strength": np.nan,
                    "weekly_residual_variance_ratio": np.nan, "weekly_fit_seconds": fit_seconds,
                    "selected_seasonal_model": "none", "weekly_stl_enabled": False,
                })

        del history, training, grouped
        gc.collect()

    registry = (
        pd.DataFrame.from_records(records)
        .sort_values(["modeling_track", "metric", "daypart", "forecast_sequence_id"])
        .reset_index(drop=True)
    )

    # The production registry must cover the full support registry exactly once
    # and may contain only the two HITL-approved model states.
    if len(registry) != len(seasonal_training_series_support_df):
        raise ValueError("Weekly STL selection registry does not cover the full seasonal-support registry.")
    if registry.duplicated(["modeling_track", "daypart_series_id"]).any():
        raise ValueError("Duplicate modeling-track × daypart-series selections detected.")
    if not set(registry["selected_seasonal_model"].unique()).issubset({"single_7", "none"}):
        raise ValueError("Unexpected seasonal model found after weekly-only selection.")

    invalid_enabled = registry["weekly_stl_enabled"] & (
        registry["weekly_fit_status"].ne("fit_ok")
        | registry["training_possible_days"].lt(WEEKLY_STL_MIN_TRAINING_DAYS)
        | registry["training_observed_share"].lt(WEEKLY_STL_MIN_OBS_SHARE)
        | registry["weekly_seasonal_strength"].lt(WEEKLY_STL_STRENGTH_THRESHOLD)
    )
    if invalid_enabled.any():
        raise ValueError("At least one weekly-STL assignment violates the resolved production rule.")

    return registry


weekly_stl_selection_registry_df = load_or_build(
    WEEKLY_SELECTION_PATH, REBUILD_WEEKLY_SELECTION, build_weekly_stl_selection_registry
)

weekly_selection_summary_df = (
    weekly_stl_selection_registry_df.groupby(["modeling_track", "metric"], as_index=False)
    .agg(
        daypart_series=("daypart_series_id", "nunique"),
        single_7_series=("weekly_stl_enabled", "sum"),
        fit_failure_series=("weekly_fit_status", lambda s: int(s.str.startswith("fit_failed").sum())),
        median_weekly_strength=("weekly_seasonal_strength", "median"),
        median_fit_seconds=("weekly_fit_seconds", "median"),
        total_initial_fit_seconds=("weekly_fit_seconds", "sum"),
    )
)

weekly_selection_summary_df["none_series"] = (
    weekly_selection_summary_df["daypart_series"] - weekly_selection_summary_df["single_7_series"]
)
weekly_selection_summary_df["single_7_share"] = (
    weekly_selection_summary_df["single_7_series"] / weekly_selection_summary_df["daypart_series"]
)

weekly_selection_display_df = weekly_selection_summary_df[
    ["modeling_track", "metric", "daypart_series", "single_7_series", "none_series",
     "single_7_share", "fit_failure_series", "median_weekly_strength",
     "median_fit_seconds", "total_initial_fit_seconds"]
].copy()

weekly_selection_display_df["modeling_track"] = weekly_selection_display_df["modeling_track"].map(TRACK_LABELS)
weekly_selection_display_df["metric"] = weekly_selection_display_df["metric"].map(METRIC_LABELS)

display(
    weekly_selection_display_df.style.format(
        {
            "single_7_share": "{:.3f}", "median_weekly_strength": "{:.3f}",
            "median_fit_seconds": "{:.3f}", "total_initial_fit_seconds": "{:.3f}",
        },
        na_rep="—",
    )
)

,modeling_track,metric,daypart_series,single_7_series,none_series,single_7_share,fit_failure_series,median_weekly_strength,median_fit_seconds,total_initial_fit_seconds
0,Full-History Regime-Aware,FHVHV speed,1285,1050,235,0.817,0,0.557,0.034,43.527
1,Full-History Regime-Aware,FHVHV trips,1290,1208,82,0.936,0,0.699,0.034,43.727
2,Full-History Regime-Aware,Subway ridership,775,692,83,0.893,0,0.757,0.034,26.270
3,Full-History Regime-Aware,Taxi speed,615,224,391,0.364,0,0.279,0.034,19.506
4,Full-History Regime-Aware,Taxi trips,1290,540,750,0.419,0,0.303,0.034,43.616
5,Post-CP Primary,FHVHV speed,1280,1031,249,0.805,0,0.704,0.004,5.571
6,Post-CP Primary,FHVHV trips,1285,1233,52,0.960,0,0.832,0.004,5.592
7,Post-CP Primary,Subway ridership,745,720,25,0.966,0,0.890,0.004,3.249
8,Post-CP Primary,Taxi speed,945,500,445,0.529,0,0.402,0.004,3.812
9,Post-CP Primary,Taxi trips,1285,753,532,0.586,0,0.422,0.004,5.594


**Findings.** Weekly-STL activation varies substantially across the displayed Track × Metric groups rather than being applied uniformly. In Post-CP Primary, 1,233 of 1,285 FHVHV-trip daypart series (96.0%) and 720 of 745 Subway-ridership series (96.6%) receive single_7, compared with 753 of 1,285 Taxi-trip series (58.6%). Taxi-trip activation is lower in the longer-history tracks, at 540 of 1,290 series (41.9%) for Full-History Regime-Aware and 484 of 1,290 (37.5%) for Pre-CP Reference. The displayed groups also report no weekly-STL fit failures, so the difference between single_7 and none is being driven by the resolved support-and-strength rule rather than failed model fits.

**Visualize where weekly STL is actually activated.** The aggregate table above tells us how often the weekly rule fires by track and metric. This heatmap keeps the Daypart dimension visible so we can check whether activation is broadly distributed or concentrated in particular parts of the daily mobility cycle.

Each cell is the share of native Taxi Zone × Metric × Daypart series assigned `single_7`. A low share is not a failure: the production policy intentionally prefers `none` over forcing a weak decomposition.

In [22]:
# 4.2.1.5C — Visualize where weekly STL is actually activated

if RUN_VISUALS:
    weekly_daypart_activation_df = (
        weekly_stl_selection_registry_df.groupby(["modeling_track", "metric", "daypart"], as_index=False)
        .agg(daypart_series=("daypart_series_id", "nunique"), single_7_series=("weekly_stl_enabled", "sum"))
    )
    weekly_daypart_activation_df["single_7_share"] = (
        weekly_daypart_activation_df["single_7_series"] / weekly_daypart_activation_df["daypart_series"]
    )

    plot_df = weekly_daypart_activation_df.copy()
    plot_df["track_label"] = plot_df["modeling_track"].map(TRACK_LABELS)
    plot_df["metric_label"] = plot_df["metric"].map(METRIC_LABELS)
    plot_df["row_label"] = plot_df["track_label"] + " · " + plot_df["metric_label"]
    plot_df["daypart_label"] = (
        plot_df["daypart"].str.replace("_", " ", regex=False).str.title()
        .str.replace("Am ", "AM ", regex=False).str.replace("Pm ", "PM ", regex=False)
    )

    row_order = [
        f"{TRACK_LABELS[track]} · {METRIC_LABELS[metric]}"
        for track in TRACK_LABELS for metric in FORECAST_TARGETS
    ]
    daypart_labels = [
        d.replace("_", " ").title().replace("Am ", "AM ").replace("Pm ", "PM ")
        for d in DAYPART_ORDER
    ]

    matrix = (
        plot_df.pivot(index="row_label", columns="daypart_label", values="single_7_share")
        .reindex(index=row_order, columns=daypart_labels)
    )
    annotation_text = [
        [f"{value:.0%}" if pd.notna(value) else "—" for value in row]
        for row in matrix.to_numpy()
    ]

    fig = go.Figure(
        go.Heatmap(
            z=matrix.to_numpy(), x=daypart_labels, y=row_order,
            text=annotation_text, texttemplate="%{text}",
            colorscale=[
                [0.00, BRAND_COLORS["ice"]],
                [0.50, BRAND_COLORS["seafoam"]],
                [1.00, BRAND_COLORS["dark_teal"]],
            ],
            zmin=0, zmax=1,
            colorbar={"title": "single_7<br>share", "tickformat": ".0%"},
            hovertemplate=(
                "Series group: %{y}<br>Daypart: %{x}<br>"
                "Weekly STL share: %{z:.1%}<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=(
            "<b>Weekly STL Activation by Modeling Track and Daypart</b><br>"
            "<sup>Share of native Taxi Zone × Metric × Daypart series assigned single_7</sup>"
        ),
        width=920, height=650, margin={"l": 235, "r": 80, "t": 95, "b": 70},
        plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_xaxes(title_text="Daypart")
    fig.update_yaxes(title_text=None, autorange="reversed")
    fig.show()

**Findings.** The weekly-STL activation rate is materially daypart-specific for several metric-track combinations. Post-CP Taxi speed ranges from 21% of overnight series to 80% in both the AM and PM peaks, while Post-CP Taxi trips range from 48% at Midday to 68% in the Evening. By contrast, FHVHV trips are consistently high across dayparts in all three tracks: 94%–97% for Post-CP Primary, 91%–95% for Full-History Regime-Aware, and 90%–95% for Pre-CP Reference. Subway ridership is also frequently selected, including 94%–100% across Post-CP dayparts. The heatmap therefore shows that the resolved weekly signal is neither uniformly present across metrics nor constant across the five dayparts.

Show how many eligible series actually receive weekly STL\. The table in 5B reports the exact selection counts, while the heatmap in 5C shows how activation varies across Dayparts\. This view provides the higher\-level count comparison: for each Modeling Track × Metric group, how many native Taxi Zone × Metric × Daypart series receive the optional single\_7 weekly\-STL feature and how many continue with none\.

Each horizontal bar represents the full eligible Daypart\-series population for one Track × Metric group\. The two stacked segments partition that population into single\_7 and none; neither category changes forecasting eligibility\. Selection was already frozen in 5B from the initial legal training window, so this block only visualizes those existing assignments and performs no refitting or additional feature selection\.

In [23]:
# 4.2.1.5D — Visualize weekly-STL selection counts by Track × Metric

if RUN_VISUALS:
    # Reuse the frozen selection summary produced in 5B. The chart operates only
    # on already-resolved single_7 / none assignments; it does not rerun STL or
    # change forecasting eligibility.
    weekly_selection_count_plot_df = weekly_selection_summary_df[
        ["modeling_track", "metric", "daypart_series", "single_7_series", "none_series", "single_7_share"]
    ].copy()

    weekly_selection_count_plot_df["track_label"] = (
        weekly_selection_count_plot_df["modeling_track"].map(TRACK_LABELS)
    )
    weekly_selection_count_plot_df["metric_label"] = (
        weekly_selection_count_plot_df["metric"].map(METRIC_LABELS)
    )
    weekly_selection_count_plot_df["group_label"] = (
        weekly_selection_count_plot_df["track_label"]
        + " · "
        + weekly_selection_count_plot_df["metric_label"]
    )

    # Preserve the notebook's established track and metric order rather than
    # allowing Plotly to reorder groups alphabetically.
    plot_order = [
        f"{TRACK_LABELS[track]} · {METRIC_LABELS[metric]}"
        for track in TRACK_LABELS
        for metric in FORECAST_TARGETS
    ]
    weekly_selection_count_plot_df["group_label"] = pd.Categorical(
        weekly_selection_count_plot_df["group_label"],
        categories=plot_order,
        ordered=True,
    )
    weekly_selection_count_plot_df = (
        weekly_selection_count_plot_df
        .sort_values("group_label")
        .reset_index(drop=True)
    )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            y=weekly_selection_count_plot_df["group_label"],
            x=weekly_selection_count_plot_df["single_7_series"],
            name="Weekly STL enabled",
            orientation="h",
            marker_color=BRAND_COLORS["dark_teal"],
            customdata=np.column_stack([
                weekly_selection_count_plot_df["daypart_series"],
                weekly_selection_count_plot_df["single_7_share"],
            ]),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Weekly STL enabled: %{x:,}<br>"
                "Eligible series: %{customdata[0]:,}<br>"
                "Enabled share: %{customdata[1]:.1%}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Bar(
            y=weekly_selection_count_plot_df["group_label"],
            x=weekly_selection_count_plot_df["none_series"],
            name="No weekly STL",
            orientation="h",
            marker_color=BRAND_COLORS["seafoam"],
            customdata=np.column_stack([
                weekly_selection_count_plot_df["daypart_series"],
                1.0 - weekly_selection_count_plot_df["single_7_share"],
            ]),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "No weekly STL: %{x:,}<br>"
                "Eligible series: %{customdata[0]:,}<br>"
                "No-STL share: %{customdata[1]:.1%}"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        barmode="stack",
        title=(
            "<b>Weekly STL Selection by Modeling Track and Metric</b><br>"
            "<sup>Count of eligible Daypart series assigned single_7 versus none</sup>"
        ),
        xaxis_title="Eligible Taxi Zone × Metric × Daypart series",
        yaxis_title=None,
        width=920,
        height=650,
        margin={"l": 235, "r": 40, "t": 95, "b": 70},
        legend_title_text="Production assignment",
        hovermode="y unified",
    )

    fig.update_yaxes(autorange="reversed")

    fig = apply_branding(fig)
    fig.show()

Findings\. The count view makes clear that weekly STL is useful for some mobility “packets” far more often than for others\. FHVHV trips stand out: in all three modeling tracks, the large majority of eligible Taxi Zone × Daypart series receive the weekly\-STL feature\. Subway ridership also shows a high activation rate, although its total eligible\-series population is smaller\. Taxi trips are much more mixed, with sizeable groups both receiving and not receiving weekly STL, while Taxi speed is the least consistently selected—especially in the Full\-History and Pre\-CP tracks\. The chart therefore reinforces that the 7\-day decomposition is being applied selectively rather than automatically: each Track × Metric group keeps weekly STL only for the individual series where the initial legal training history showed a sufficiently strong and well\-supported weekly pattern\.

**Build the monthly legal-history refresh calendar.** The forecasting process still advances through **every forecast origin and every daypart**. Overnight, AM Peak, Midday, PM Peak, and Evening forecasts continue to roll forward at the normal forecasting cadence, and ordinary history-based features such as lags, rolling means, and EWM update at each origin.

The **monthly cadence applies only to the fitted weekly STL pattern**. Once a series has qualified for weekly STL, we periodically refit its Monday–Sunday seasonal pattern using all legally available history up to that point. We call each of these allowed refits a **refresh point**. Between refresh points, forecasts continue normally at every daypart while reusing the most recently fitted weekly template.

The first STL refresh state is the weekly pattern available at that track’s **first rolling-validation origin**. After that, the pattern is refreshed about once per month, using the first scheduled forecast origin on or after each monthly anchor date. At every refresh, the STL fit may use only observations that were already available at that exact origin.

Anchoring the schedule to the actual first validation origin avoids unnecessary duplicate fits around calendar boundaries. For example, if the first legal STL fit occurs near the end of a month, we do not immediately refit it again simply because the calendar has moved into the next month.

The Pre-CP Reference has one additional special refresh: a final legal refit through **January 4, 2025**. After congestion pricing begins, that weekly pattern is frozen and reused for the later no-congestion-pricing counterfactual rather than being updated with post-CP observations.

In [24]:
# 4.2.1.5E — Build the monthly legal-history refresh calendar

def build_weekly_stl_refresh_calendar():
    """
    Build the legal monthly refresh calendar for selected weekly-STL features.

    The first fit state for each modeling track is anchored to that track's first
    rolling-validation ORIGIN: the information state available when validation
    forecasting begins.

    Subsequent monthly anchors are measured forward from that initial state.
    Because an arbitrary calendar anniversary may not itself be a registered
    forecasting origin, each anchor is mapped to the first scheduled origin on
    or after the anchor date.

    This calendar controls WHEN an already-selected `single_7` representation may
    learn from newly available history. It does not reopen the frozen seasonal
    model assignment.

    Pre-CP safeguard
    ----------------
    Pre-CP Reference receives only legally Pre-CP regular refreshes plus one
    explicit final fit state through January 4, 2025. That final state is frozen
    for later no-CP counterfactual use so observed post-CP mobility can never
    enter its fitted seasonal representation.

    Returns
    -------
    pandas.DataFrame
        One row per Modeling Track × legal weekly-STL refresh state.
    """
    records = []

    for track in TRACK_LABELS:
        cutoff_id = first_validation_origin_id(track)
        cutoff_date = sequence_calendar_df.loc[
            sequence_calendar_df["observation_sequence_id"].eq(cutoff_id), "date"
        ].iloc[0]

        # Only scheduled forecasting origins can trigger ordinary refreshes.
        # Pre-CP Reference stops before the Jan 1-4 final-refit-only window.
        regular = feature_origin_registry_df.loc[
            feature_origin_registry_df["modeling_track"].eq(track)
            & feature_origin_registry_df["origin_role"].eq("scheduled_backtest")
            & feature_origin_registry_df["observation_sequence_id"].ge(cutoff_id)
        ].sort_values("observation_sequence_id").copy()

        if track == "pre_cp_reference":
            regular = regular.loc[regular["date"].lt(PRE_CP_REFIT_ONLY_START_DATE)].copy()

        if regular.empty:
            raise ValueError(f"No legal scheduled refresh origins found for {track}.")

        # Fit once at the first validation information state, then refresh on a
        # one-month cadence anchored to that date.
        records.append({
            "modeling_track": track, "refresh_role": "initial_validation_fit",
            "refresh_anchor_date": cutoff_date, "refresh_date": cutoff_date,
            "refresh_observation_sequence_id": cutoff_id,
        })

        last_regular_date = regular["date"].max()
        month_step = WEEKLY_STL_REFRESH_MONTHS

        while True:
            anchor_date = cutoff_date + pd.DateOffset(months=month_step)
            if anchor_date > last_regular_date:
                break

            candidate = regular.loc[regular["date"].ge(anchor_date)].head(1)
            if candidate.empty:
                break

            chosen = candidate.iloc[0]
            records.append({
                "modeling_track": track, "refresh_role": "monthly_refresh",
                "refresh_anchor_date": anchor_date, "refresh_date": chosen["date"],
                "refresh_observation_sequence_id": int(chosen["observation_sequence_id"]),
            })
            month_step += WEEKLY_STL_REFRESH_MONTHS

        # Pre-CP Reference receives one final legal refit through Jan 4, 2025.
        # That state is frozen for all later no-CP counterfactual applications.
        if track == "pre_cp_reference":
            records.append({
                "modeling_track": track, "refresh_role": "final_pre_cp_freeze",
                "refresh_anchor_date": PRE_CP_SEASONAL_FREEZE_DATE,
                "refresh_date": PRE_CP_SEASONAL_FREEZE_DATE,
                "refresh_observation_sequence_id": PRE_CP_SEASONAL_FREEZE_ID,
            })

    calendar = (
        pd.DataFrame.from_records(records)
        .sort_values(["modeling_track", "refresh_observation_sequence_id"])
        .reset_index(drop=True)
    )

    if calendar.duplicated(["modeling_track", "refresh_observation_sequence_id"]).any():
        raise ValueError("Duplicate modeling-track × weekly-STL refresh points detected.")

    pre_cp_max_refresh = calendar.loc[
        calendar["modeling_track"].eq("pre_cp_reference"), "refresh_date"
    ].max()
    if pre_cp_max_refresh > PRE_CP_SEASONAL_FREEZE_DATE:
        raise ValueError("Pre-CP Reference refresh calendar extends beyond its legal freeze date.")

    return calendar


weekly_stl_refresh_calendar_df = load_or_build(
    WEEKLY_REFRESH_CALENDAR_PATH, REBUILD_WEEKLY_REFRESH_CALENDAR,
    build_weekly_stl_refresh_calendar
)

weekly_refresh_summary_df = (
    weekly_stl_refresh_calendar_df.groupby("modeling_track", as_index=False)
    .agg(
        refresh_points=("refresh_observation_sequence_id", "size"),
        monthly_refreshes=("refresh_role", lambda s: int(s.eq("monthly_refresh").sum())),
        final_freeze_points=("refresh_role", lambda s: int(s.eq("final_pre_cp_freeze").sum())),
        first_refresh_date=("refresh_date", "min"), last_refresh_date=("refresh_date", "max"),
    )
)

weekly_refresh_display_df = weekly_refresh_summary_df.copy()
weekly_refresh_display_df["modeling_track"] = weekly_refresh_display_df["modeling_track"].map(TRACK_LABELS)
display(weekly_refresh_display_df)

,modeling_track,refresh_points,monthly_refreshes,final_freeze_points,first_refresh_date,last_refresh_date
0,Full-History Regime-Aware,12,11,0,2025-04-04,2026-03-04
1,Post-CP Primary,12,11,0,2025-04-04,2026-03-04
2,Pre-CP Reference,14,12,1,2023-12-31,2025-01-04


**Findings.** Both Post-CP Primary and Full-History Regime-Aware have 12 allowed weekly-STL fit states: the initial fit on April 4, 2025, followed by 11 roughly monthly refreshes through March 4, 2026. Pre-CP Reference has 14 fit states: its initial fit on December 31, 2023, 12 monthly refreshes during 2024, and one final legal refit on January 4, 2025. That January 4 fit is the only `final_pre_cp_freeze` in the calendar and becomes the last weekly-STL state available to the Pre-CP counterfactual after congestion pricing begins.

**Benchmark the cost of later weekly-STL refreshes.** The earlier seasonal candidate scan included weekly, annual, and multi-seasonality models, so its runtime is not a good estimate of how expensive our final weekly-only production plan will be.

This block therefore times only the type of STL fit we actually plan to use in production. Within each modeling track, it samples up to two `single_7` series from every Metric × Daypart combination and refits each one at that track's **latest legal refresh point**. Using the latest refresh means the test includes relatively long histories, which gives us a more realistic picture of the cost of refitting weekly STL later in the forecasting process.

This is only a runtime test. It does not reconsider whether a series should receive weekly STL, and none of the fitted components from this benchmark are saved for forecasting.

In [25]:
# 4.2.1.5F — Benchmark late-history weekly-STL refresh cost

def build_weekly_stl_runtime_benchmark():
    """
    Benchmark weekly-only STL refresh cost at each track's latest legal fit state.

    Sampling
    --------
    Only Daypart series already frozen to `single_7` are eligible. Within each
    modeling track, a small reproducible sample is taken across Metric × Daypart
    combinations so timing is not dominated by one target or time of day.

    Legal history
    -------------
    Each sampled fit uses observations only through that track's latest legal
    refresh observation_sequence_id. Filtering by canonical sequence ID is
    important because a refresh can occur within a calendar day; later Dayparts
    on that same date must not enter the fit prematurely.

    Purpose
    -------
    The benchmark measures the cost of the approved weekly-only refresh design
    on relatively long histories. Fitted components are discarded immediately
    and cannot alter the `single_7` versus `none` assignment.

    This is a timing diagnostic, not a production feature-materialization step.

    Returns
    -------
    pandas.DataFrame
        Per-sampled-series cutoff metadata, history length, fit status, and
        measured weekly-STL fit time.
    """
    records = []

    for track in TRACK_LABELS:
        selected = weekly_stl_selection_registry_df.loc[
            weekly_stl_selection_registry_df["modeling_track"].eq(track)
            & weekly_stl_selection_registry_df["weekly_stl_enabled"]
        ].copy()

        if selected.empty:
            continue

        # At most two series per Metric × Daypart keeps the benchmark bounded
        # while preventing one feature family from dominating the timing sample.
        sampled_parts = []
        for _, group in selected.groupby(["metric", "daypart"], observed=True, sort=False):
            n = min(WEEKLY_STL_BENCHMARK_SERIES_PER_METRIC_DAYPART, len(group))
            sampled_parts.append(group.sample(n=n, random_state=WEEKLY_STL_BENCHMARK_RANDOM_STATE))
        sample = pd.concat(sampled_parts, ignore_index=True)

        cutoff_row = (
            weekly_stl_refresh_calendar_df.loc[
                weekly_stl_refresh_calendar_df["modeling_track"].eq(track)
            ]
            .sort_values("refresh_observation_sequence_id")
            .iloc[-1]
        )
        cutoff_id = int(cutoff_row["refresh_observation_sequence_id"])
        cutoff_date = pd.Timestamp(cutoff_row["refresh_date"])

        # This late-history fit is timing-only. Components are discarded and do
        # not alter the frozen model assignment learned in the initial window.
        history = pd.read_parquet(track_path("track_history_source", track))
        legal_history = history.loc[history["observation_sequence_id"].le(cutoff_id)].copy()
        grouped = legal_history.groupby(["forecast_sequence_id", "daypart"], observed=True, sort=False)

        for row in tqdm(sample.itertuples(index=False), total=len(sample),
                        desc=f"{TRACK_LABELS[track]} weekly STL benchmark"):
            key = (row.forecast_sequence_id, row.daypart)
            fit_start = perf_counter()
            history_days = np.nan

            try:
                series_df = grouped.get_group(key)
                observed = prepare_daily_training_series(series_df)
                history_days = len(observed)
                components = fit_seasonality_candidate(observed, "single_7")
                elapsed = perf_counter() - fit_start
                fit_status = "fit_ok"
                del components
            except Exception as exc:
                elapsed = perf_counter() - fit_start
                fit_status = f"fit_failed: {type(exc).__name__}"

            records.append({
                "modeling_track": track, "forecast_sequence_id": row.forecast_sequence_id,
                "daypart_series_id": row.daypart_series_id, "metric": row.metric, "daypart": row.daypart,
                "benchmark_cutoff_date": cutoff_date,
                "benchmark_cutoff_observation_sequence_id": cutoff_id,
                "benchmark_history_days": history_days, "benchmark_seconds": elapsed,
                "fit_status": fit_status,
            })

        del history, legal_history, grouped
        gc.collect()

    if not records:
        raise ValueError("No single_7 series were available for the weekly-STL runtime benchmark.")

    return (
        pd.DataFrame.from_records(records)
        .sort_values(["modeling_track", "metric", "daypart", "forecast_sequence_id"])
        .reset_index(drop=True)
    )


weekly_stl_runtime_benchmark_df = load_or_build(
    WEEKLY_RUNTIME_BENCHMARK_PATH, REBUILD_WEEKLY_RUNTIME_BENCHMARK,
    build_weekly_stl_runtime_benchmark
)

weekly_stl_runtime_benchmark_df["benchmark_seconds_ok"] = (
    weekly_stl_runtime_benchmark_df["benchmark_seconds"].where(
        weekly_stl_runtime_benchmark_df["fit_status"].eq("fit_ok")
    )
)

weekly_runtime_benchmark_summary_df = (
    weekly_stl_runtime_benchmark_df.groupby("modeling_track", as_index=False)
    .agg(
        benchmark_series=("daypart_series_id", "nunique"),
        fit_failures=("fit_status", lambda s: int(s.ne("fit_ok").sum())),
        median_history_days=("benchmark_history_days", "median"),
        median_refresh_seconds=("benchmark_seconds_ok", "median"),
        p90_refresh_seconds=("benchmark_seconds_ok", lambda s: s.quantile(0.90)),
        max_refresh_seconds=("benchmark_seconds_ok", "max"),
    )
)

weekly_runtime_benchmark_display_df = weekly_runtime_benchmark_summary_df.copy()
weekly_runtime_benchmark_display_df["modeling_track"] = (
    weekly_runtime_benchmark_display_df["modeling_track"].map(TRACK_LABELS)
)

display(
    weekly_runtime_benchmark_display_df.style.format(
        {
            "median_history_days": "{:.0f}", "median_refresh_seconds": "{:.3f}",
            "p90_refresh_seconds": "{:.3f}", "max_refresh_seconds": "{:.3f}",
        },
        na_rep="—",
    )
)

,modeling_track,benchmark_series,fit_failures,median_history_days,median_refresh_seconds,p90_refresh_seconds,max_refresh_seconds
0,Full-History Regime-Aware,50,0,1158,0.049,0.049,0.775
1,Post-CP Primary,50,0,423,0.019,0.020,0.282
2,Pre-CP Reference,50,0,735,0.032,0.032,0.456


**Findings.** All 150 sampled weekly-STL refits completed successfully: 50 in each modeling track and no failures. The typical fit was fast even when using the latest available history. Median runtime was 0.019 seconds for Post-CP Primary, 0.032 seconds for Pre-CP Reference, and 0.049 seconds for Full-History Regime-Aware. The corresponding median history lengths were 423, 735, and 1,158 days. Most fits stayed close to those typical runtimes, although the slowest individual fits reached 0.282, 0.456, and 0.775 seconds respectively.

**Estimate the full monthly weekly-STL workload.** The benchmark above measures individual late-history fits; this block translates those measurements into the scale of the proposed production build.

For each modeling track, the expected fit count is:

`selected single_7 daypart series × legal refresh points`

The resulting time estimates are deliberately labeled **serial-equivalent projections**. They multiply the measured median and 90th-percentile per-fit timings by the planned number of fits. They are not promises of actual Deepnote wall-clock runtime: parallel execution, caching, disk I/O, scheduling overhead, and variation in individual series lengths can all move the observed runtime in either direction.

The purpose is to decide whether the monthly refresh design is operationally reasonable before generating the full STL feature surface.

In [26]:
# 4.2.1.5G — Estimate the full monthly weekly-STL workload

# Only Daypart series frozen to `single_7` generate production STL fits.
# Series assigned `none` remain valid forecasting series but contribute no
# weekly-STL refresh work.
selected_weekly_counts_df = (
    weekly_stl_selection_registry_df.loc[
        weekly_stl_selection_registry_df["weekly_stl_enabled"]
    ]
    .groupby("modeling_track")
    .size()
    .rename("selected_weekly_series")
    .reset_index()
)

# Count every legal fit state in each track's refresh calendar, including the
# initial fit state and the explicit final Pre-CP freeze where applicable.
refresh_counts_df = (
    weekly_stl_refresh_calendar_df.groupby("modeling_track")
    .size()
    .rename("refresh_points")
    .reset_index()
)

# Combine workload size with the measured late-history benchmark rate.
# one_to_one validation prevents duplicate track summaries from silently
# multiplying rows and inflating the projected workload.
weekly_stl_workload_df = (
    pd.DataFrame({"modeling_track": list(TRACK_LABELS)})
    .merge(
        selected_weekly_counts_df,
        on="modeling_track",
        how="left",
        validate="one_to_one",
    )
    .merge(
        refresh_counts_df,
        on="modeling_track",
        how="left",
        validate="one_to_one",
    )
    .merge(
        weekly_runtime_benchmark_summary_df[
            ["modeling_track", "median_refresh_seconds", "p90_refresh_seconds"]
        ],
        on="modeling_track",
        how="left",
        validate="one_to_one",
    )
)

weekly_stl_workload_df[["selected_weekly_series", "refresh_points"]] = (
    weekly_stl_workload_df[
        ["selected_weekly_series", "refresh_points"]
    ]
    .fillna(0)
    .astype(int)
)

# Planned fit count assumes each enabled Daypart series is fitted once at every
# legal track-level fit state.
weekly_stl_workload_df["expected_weekly_fit_count"] = (
    weekly_stl_workload_df["selected_weekly_series"]
    * weekly_stl_workload_df["refresh_points"]
)

# These are SERIAL-EQUIVALENT projections only. They deliberately do not claim
# actual Deepnote wall-clock runtime under parallelism, caching, disk I/O,
# scheduler overhead, or heterogeneous per-series history lengths.
weekly_stl_workload_df["serial_estimate_at_median_rate_minutes"] = (
    weekly_stl_workload_df["expected_weekly_fit_count"]
    * weekly_stl_workload_df["median_refresh_seconds"]
    / 60
)
weekly_stl_workload_df["serial_estimate_at_p90_rate_minutes"] = (
    weekly_stl_workload_df["expected_weekly_fit_count"]
    * weekly_stl_workload_df["p90_refresh_seconds"]
    / 60
)

# Preserve machine-facing identifiers in the reusable table and map friendly
# labels only onto the display copy.
weekly_stl_workload_display_df = weekly_stl_workload_df.copy()
weekly_stl_workload_display_df["modeling_track"] = (
    weekly_stl_workload_display_df["modeling_track"].map(TRACK_LABELS)
)

display(
    weekly_stl_workload_display_df.style.format({
        "median_refresh_seconds": "{:.3f}",
        "p90_refresh_seconds": "{:.3f}",
        "serial_estimate_at_median_rate_minutes": "{:.1f}",
        "serial_estimate_at_p90_rate_minutes": "{:.1f}",
    })
)

,modeling_track,selected_weekly_series,refresh_points,median_refresh_seconds,p90_refresh_seconds,expected_weekly_fit_count,serial_estimate_at_median_rate_minutes,serial_estimate_at_p90_rate_minutes
0,Post-CP Primary,4237,12,0.019,0.020,50844,16.4,16.9
1,Full-History Regime-Aware,3714,12,0.049,0.049,44568,36.2,36.4
2,Pre-CP Reference,3633,14,0.032,0.032,50862,27.1,27.4


**Findings.** The deterministic annual Fourier calendar contains one feature row for all 5,930 canonical sequence positions spanning January 1, 2023 through March 31, 2026. QA finds no duplicate sequence positions, no missing Fourier-feature rows, and no values outside the expected bounds. This confirms that the annual sine/cosine basis provides complete, well-formed calendar-phase coverage across the full canonical forecasting timeline.

**Materialize the deterministic annual Fourier calendar.** Weekly STL learns a recurring pattern from observed mobility data. The annual Fourier features are different: **they do not learn from mobility at all**. They simply tell the forecasting model where each date falls within the yearly calendar cycle.

We represent that annual position with two smoothly changing values—a sine value and a cosine value—that repeat once per year. Together, they let the model distinguish parts of the year such as winter, spring, summer, and fall without treating December 31 and January 1 as if they were far apart.

Because these values depend only on the calendar date, they are known in advance. We can therefore calculate them safely for every date in the study period, including future dates used later by the frozen Pre-CP counterfactual model, without looking at any future mobility outcomes.

We build this annual calendar once for all 5,930 canonical sequence positions. Every modeling track can then look up the same annual-calendar values using `observation_sequence_id`. The 365.2425-day cycle also keeps the annual phase aligned smoothly across leap years.

In [27]:
# 4.2.1.5H — Materialize the deterministic annual Fourier calendar

FOURIER_FEATURE_COLUMNS = ["annual_fourier_sin", "annual_fourier_cos"]

def build_annual_fourier_calendar():
    """
    Encode deterministic annual phase for every canonical sequence position.

    Grain
    -----
    One row per canonical observation_sequence_id. Because annual phase depends
    only on calendar date, this table is shared across modeling tracks rather
    than refitted separately for each forecasting regime.

    Construction
    ------------
    Elapsed calendar days are measured from the first date present in the
    canonical sequence calendar and converted to angular phase using
    ANNUAL_FOURIER_PERIOD_DAYS. A sine/cosine pair preserves annual position
    continuously across the end-of-year boundary.

    Leakage safety
    --------------
    No mobility outcome, fitted parameter, validation result, or post-CP observed
    target is used. The features are deterministic future-known calendar context
    and are therefore safe to calculate for the complete study calendar,
    including dates later used by the frozen Pre-CP counterfactual model.

    Returns
    -------
    pandas.DataFrame
        Canonical sequence calendar with annual_fourier_sin and
        annual_fourier_cos appended.
    """
    calendar = (
        sequence_calendar_df[["observation_sequence_id", "date"]]
        .drop_duplicates()
        .sort_values("observation_sequence_id")
        .reset_index(drop=True)
    )

    if calendar.duplicated("observation_sequence_id").any():
        raise ValueError("Canonical Fourier calendar contains duplicate observation_sequence_id values.")

    # Anchor phase to the actual beginning of the canonical study calendar so
    # this block does not depend on a separate duplicated study-start constant.
    calendar_anchor_date = pd.Timestamp(calendar["date"].min())
    elapsed_days = (calendar["date"] - calendar_anchor_date).dt.days.astype("float64")
    phase = 2.0 * np.pi * elapsed_days / ANNUAL_FOURIER_PERIOD_DAYS

    calendar["annual_fourier_sin"] = np.sin(phase).astype("float32")
    calendar["annual_fourier_cos"] = np.cos(phase).astype("float32")
    return calendar


annual_fourier_calendar_df = load_or_build(
    ANNUAL_FOURIER_PATH, REBUILD_ANNUAL_FOURIER_FEATURES, build_annual_fourier_calendar
)

duplicate_positions = int(annual_fourier_calendar_df["observation_sequence_id"].duplicated().sum())
missing_feature_rows = int(annual_fourier_calendar_df[FOURIER_FEATURE_COLUMNS].isna().any(axis=1).sum())
out_of_bounds_rows = int(
    (annual_fourier_calendar_df[FOURIER_FEATURE_COLUMNS].abs() > 1.000001).any(axis=1).sum()
)
expected_positions = int(sequence_calendar_df["observation_sequence_id"].nunique())

if len(annual_fourier_calendar_df) != expected_positions:
    raise ValueError("Annual Fourier calendar does not cover every canonical sequence position.")
if duplicate_positions or missing_feature_rows or out_of_bounds_rows:
    raise ValueError("Annual Fourier calendar failed uniqueness, missingness, or range QA.")

annual_fourier_qa_df = pd.DataFrame([{
    "calendar_positions": len(annual_fourier_calendar_df),
    "first_date": annual_fourier_calendar_df["date"].min(),
    "last_date": annual_fourier_calendar_df["date"].max(),
    "duplicate_positions": duplicate_positions,
    "missing_feature_rows": missing_feature_rows,
    "out_of_bounds_rows": out_of_bounds_rows,
}])

display(annual_fourier_qa_df)

,calendar_positions,first_date,last_date,duplicate_positions,missing_feature_rows,out_of_bounds_rows
0,5930,2023-01-01,2026-03-31,0,0,0


**Findings.** Annual Fourier features were successfully created for all 5,930 canonical sequence positions from January 1, 2023 through March 31, 2026. Every position has one complete sine/cosine pair, with no duplicate positions, missing feature rows, or values outside the expected range. This gives every forecast date a complete and future-known representation of where it falls within the annual calendar cycle.

> 💡 How will these Fourier features actually be used?

The annual sine and cosine values are predictor features\. They are not forecasts, and they are not inputs to the weekly\-STL calculation that follows\. Later, when we assemble the modeling dataset, each forecast row will receive the sine/cosine values that correspond to its target date\.

The forecasting model will then see them alongside other predictors such as recent lags, rolling\-history features, Daypart information, and—where applicable—the fitted weekly\-STL seasonal offset\. The Fourier pair simply gives the model one extra clue: “what time of year are we forecasting?”

If that annual timing signal helps prediction, the model can learn from it\. If it does not, the model can largely ignore it\. Because these values come only from the calendar date, they are also safe to use for future forecasts and for the frozen Pre\-CP counterfactual without observing any future mobility outcomes\.

Weekly STL and annual Fourier are therefore separate feature streams: STL learns a recurring Monday–Sunday mobility pattern from legal historical data, while Fourier supplies future\-known annual calendar position\. They meet later as predictors in the forecasting model; one is not used to construct the other\.

**Visualize the deterministic annual Fourier basis.** The annual sine and cosine features give the forecasting models a simple way to know **where each forecast date falls within the year**. They are not predictions of mobility and are not expected to resemble the mobility series themselves.

The practical purpose is to let a model learn broad time-of-year differences—such as whether a metric tends to behave differently in winter versus summer—without us having to define those seasonal effects by hand. The two smoothly changing values also avoid treating December 31 and January 1 as unrelated points in time.

Unlike weekly STL, these features are calculated entirely from the calendar date. They therefore remain available for future forecasts, including the frozen Pre-CP counterfactual, without using any future mobility observations.

In [28]:
# 4.2.1.5I — Visualize the deterministic annual Fourier basis

if RUN_VISUALS:
    fourier_plot_df = (
        annual_fourier_calendar_df[["date", "annual_fourier_sin", "annual_fourier_cos"]]
        .drop_duplicates("date")
        .sort_values("date")
    )

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fourier_plot_df["date"], y=fourier_plot_df["annual_fourier_sin"],
        mode="lines", name="Annual sine",
        line={"color": BRAND_COLORS["dark_teal"], "width": 2.2},
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Annual sine: %{y:.3f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=fourier_plot_df["date"], y=fourier_plot_df["annual_fourier_cos"],
        mode="lines", name="Annual cosine",
        line={"color": BRAND_COLORS["seafoam"], "width": 2.2},
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Annual cosine: %{y:.3f}<extra></extra>",
    ))

    fig.add_vline(
        x=CP_START_DATE, line_dash="dash", line_color=BRAND_COLORS["terracotta"],
        annotation_text="Congestion pricing starts", annotation_position="top left",
    )

    fig.update_layout(
        title=(
            "<b>Deterministic Annual Fourier Basis</b><br>"
            "<sup>Calendar phase is fixed by date and does not learn from mobility outcomes</sup>"
        ),
        width=920, height=460, margin={"l": 60, "r": 30, "t": 90, "b": 70},
        xaxis_title="Date", yaxis_title="Fourier value", hovermode="x unified",
        legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": -0.18, "yanchor": "top"},
        plot_bgcolor="white", paper_bgcolor="white",
    )
    fig.update_yaxes(range=[-1.1, 1.1], dtick=0.5)
    fig.show()

**Findings.** The sine and cosine features form smooth annual cycles across the entire study period and continue unchanged through the January 5, 2025 congestion-pricing boundary, as expected for features based only on calendar date. Their role is not to reproduce the mobility data, but to give downstream forecasting models a consistent measure of **time of year**. This allows a model to learn broad annual patterns when they are useful, while weekly STL separately captures recurring Monday-through-Sunday effects.

> ✅ Decision — Approve the production weekly\-STL execution plan\.

The evidence from Sections 4\.2\.1\.5B–5H supports proceeding with the full production build\. Weekly STL will remain \*\*selective\*\*, using the frozen \`single\_7\` versus \`none\` assignments learned from each track's initial legal training window\. Enabled series will refresh on the anchored monthly schedule already constructed, with Pre\-CP Reference receiving its final legal refit through January 4, 2025 and remaining frozen thereafter\.

Production STL will contribute only a fitted weekly seasonal offset\. STL trend and observed STL residual will not enter the common feature surface, avoiding redundant level representations and the recursive/leakage complications that would arise for the post\-CP no\-CP counterfactual\.

At each legal refresh, the weekly template will be defined as the final complete seven\-day cycle of the fitted STL seasonal component\. Those seven weekday\-specific offsets will remain fixed until the next legal refresh\. Because STL is fitted separately at the Taxi Zone × Metric × Daypart grain, each qualifying Daypart series receives its own Monday–Sunday template\. For series assigned \`none\`, the production seasonal offset will be \`0\.0\` with \`weekly\_stl\_enabled = False\`\.

This locks the seasonal feature contract as:
\- recent lags / deltas / rolling / EMA: evolving local history;
\- weekly STL: selective, training\-fitted weekly seasonal offset;
\- annual Fourier: deterministic longer\-run calendar phase;
\- no common STL trend or observed residual: avoids unnecessary redundancy, leakage, and counterfactual recursion\.

**Materialize the legal weekly-STL refresh templates.** The HITL decision above approved the execution plan; this block now performs the expensive production build. Only Daypart series already assigned `single_7` are fitted. The frozen model assignment is not reconsidered during refreshes.

Each fitted state is constructed at:

**Modeling Track × Forecast Sequence × Daypart × Refresh Origin**

For every approved refresh, the history is first restricted to `observation_sequence_id <= refresh_observation_sequence_id`. This exact sequence-position cutoff matters because a refresh can occur before later Dayparts on the same calendar date; those later observations must not enter the fit merely because they share the same date.

After that legal history is isolated, the Daypart series is regularized to daily frequency and missing positions may be interpolated **inside that already-legal history only** so STL can operate on a complete daily grid. Those interpolated values are fitting inputs, not exported forecasting predictors.

The production representation is deliberately compact. A robust 7-day STL fit is estimated from the expanding legal history, and the **final complete seven-day seasonal cycle** is extracted as seven Monday-through-Sunday offsets. Those seven offsets describe the weekly seasonal state learned at that refresh and remain unchanged until the next approved refresh.

Because this build requires many repeated STL fits, outputs are cached separately by Modeling Track × Metric. That makes the long-running step resumable: completed partitions can be reused if execution is interrupted.

By the end of this block, `weekly_stl_template_registry_df` will contain every required fitted weekly seasonal state together with its exact legal-history provenance.

---

In [29]:
# 4.2.1.5J — Materialize legal weekly-STL refresh templates

REBUILD_WEEKLY_STL_TEMPLATES = False

# The full build is intentionally partitioned by Modeling Track × Metric. STL
# refreshes are the expensive part of 4.2.1, so this lets an interrupted run
# resume without discarding partitions that were already completed safely.
WEEKLY_STL_TEMPLATE_DIR = INTERMEDIATE_DIR / "weekly_stl_template_partitions"
WEEKLY_STL_TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

# Store one fitted seasonal value for each weekday. A wide seven-column
# representation is much smaller than repeating seven separate template rows.
WEEKLY_STL_TEMPLATE_COLUMNS = {
    0: "weekly_stl_offset_mon", 1: "weekly_stl_offset_tue", 2: "weekly_stl_offset_wed", 3: "weekly_stl_offset_thu",
    4: "weekly_stl_offset_fri", 5: "weekly_stl_offset_sat", 6: "weekly_stl_offset_sun",
}
WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS = list(WEEKLY_STL_TEMPLATE_COLUMNS.values())

WEEKLY_STL_TEMPLATE_BASE_COLUMNS = [
    "modeling_track", "forecast_sequence_id", "daypart_series_id", "taxi_zone_id", "zone", "borough", "metric", "daypart",
    "refresh_role", "refresh_anchor_date", "refresh_date", "refresh_observation_sequence_id",
    "refresh_possible_days", "refresh_observed_days", "refresh_observed_share",
    "template_history_max_observation_sequence_id", "template_history_end_date",
    "template_source_start_date", "template_source_end_date", "fit_status", "fit_error", "fit_seconds",
]
WEEKLY_STL_TEMPLATE_OUTPUT_COLUMNS = WEEKLY_STL_TEMPLATE_BASE_COLUMNS + WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS


def weekly_stl_template_path(track, metric):
    """Return the resumable cache path for one Modeling Track × Metric template partition."""
    return WEEKLY_STL_TEMPLATE_DIR / f"weekly_stl_template_registry__{track}__{metric}.parquet"


_weekly_template_track_history_cache = {}


def get_weekly_template_track_history(track):
    """
    Load the minimal track-safe history needed for production weekly-STL fitting.

    The source is the history table already constrained earlier in 4.2.1 to the
    legitimate information universe of the requested modeling track. Only the
    six columns needed for the STL refresh build are read from Parquet, which
    keeps the expensive fitting loop substantially lighter in memory.

    One track is cached while its five target metrics are processed. The cache
    is released before advancing to the next modeling track.
    """
    if track not in _weekly_template_track_history_cache:
        _weekly_template_track_history_cache[track] = pd.read_parquet(
            track_path("track_history_source", track),
            columns=["forecast_sequence_id", "observation_sequence_id", "date", "daypart", "metric", "observed_value"],
        )
        _weekly_template_track_history_cache[track]["date"] = pd.to_datetime(
            _weekly_template_track_history_cache[track]["date"]
        ).dt.normalize()

    return _weekly_template_track_history_cache[track]


def build_weekly_stl_template_partition(track, metric):
    """
    Fit every approved weekly-STL refresh for one Modeling Track × target metric.

    Grain
    -----
    One row per Modeling Track × Forecast Sequence × Daypart × Refresh Origin.

    Leakage policy
    --------------
    The frozen single_7 / none decision comes from
    weekly_stl_selection_registry_df and is never reopened here.

    Before any daily regularization or interpolation, each fit is restricted to
    rows whose observation_sequence_id is less than or equal to the approved
    refresh_observation_sequence_id. This is stricter than filtering only by
    date: if the refresh occurs before a later Daypart on the same date, that
    later observation remains unavailable and cannot enter the fit.

    Missing positions may be interpolated only after the legal history has
    already been isolated. Interpolation therefore helps STL operate on a
    regular daily Daypart series but cannot introduce an observation from
    beyond the refresh information boundary. Interpolated values themselves
    are never exported as forecasting predictors.

    Production representation
    -------------------------
    A robust single_7 decomposition is fitted to the expanding legal history.
    The final complete seven-day seasonal cycle becomes the production weekly
    state. Its seven values are mapped to Monday through Sunday and remain
    active until the next legal refresh.

    The fitted STL trend and observed residual are intentionally discarded.
    Only the approved weekly seasonal offsets enter the production contract.
    """
    selection_mask = (
        weekly_stl_selection_registry_df["modeling_track"].eq(track)
        & weekly_stl_selection_registry_df["metric"].eq(metric)
        & weekly_stl_selection_registry_df["weekly_stl_enabled"]
    )
    selected = weekly_stl_selection_registry_df.loc[selection_mask].sort_values(
        ["forecast_sequence_id", "daypart"]
    ).copy()

    refreshes = weekly_stl_refresh_calendar_df.loc[
        weekly_stl_refresh_calendar_df["modeling_track"].eq(track)
    ].sort_values("refresh_observation_sequence_id").copy()

    if selected.empty:
        return pd.DataFrame(columns=WEEKLY_STL_TEMPLATE_OUTPUT_COLUMNS)

    if refreshes.empty:
        raise ValueError(f"No weekly-STL refresh points are available for {track}.")

    # Work on only this target metric and group the history once. Every monthly
    # refresh for the same Forecast Sequence × Daypart can then reuse the group
    # rather than repeatedly scanning the full track history.
    metric_history = get_weekly_template_track_history(track)
    metric_history = metric_history.loc[
        metric_history["metric"].eq(metric),
        ["forecast_sequence_id", "observation_sequence_id", "date", "daypart", "observed_value"],
    ].copy()

    grouped = metric_history.groupby(["forecast_sequence_id", "daypart"], observed=True, sort=False)
    available_keys = set(grouped.groups)
    records = []

    iterator = tqdm(
        selected.itertuples(index=False),
        total=len(selected),
        desc=f"{TRACK_LABELS[track]} · {METRIC_LABELS[metric]} weekly templates",
    )

    for selection_row in iterator:
        key = (selection_row.forecast_sequence_id, selection_row.daypart)

        for refresh_row in refreshes.itertuples(index=False):
            cutoff_id = int(refresh_row.refresh_observation_sequence_id)

            base = {
                "modeling_track": track, "forecast_sequence_id": selection_row.forecast_sequence_id,
                "daypart_series_id": selection_row.daypart_series_id, "taxi_zone_id": selection_row.taxi_zone_id,
                "zone": selection_row.zone, "borough": selection_row.borough, "metric": metric,
                "daypart": selection_row.daypart, "refresh_role": refresh_row.refresh_role,
                "refresh_anchor_date": pd.Timestamp(refresh_row.refresh_anchor_date),
                "refresh_date": pd.Timestamp(refresh_row.refresh_date),
                "refresh_observation_sequence_id": cutoff_id,
            }

            # Initialize provenance values so a failed refresh still leaves a
            # complete diagnostic record instead of aborting the full build.
            possible_days = np.nan
            observed_days = np.nan
            observed_share = np.nan
            history_max_id = np.nan
            history_end_date = pd.NaT
            source_start_date = pd.NaT
            source_end_date = pd.NaT
            fit_seconds = np.nan
            fit_start = None
            offsets = {column: np.nan for column in WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS}

            try:
                if key not in available_keys:
                    raise KeyError(f"Track history is missing approved weekly Daypart series {key}.")

                series_df = grouped.get_group(key)

                # CRITICAL LEAKAGE BOUNDARY:
                # Restrict history by canonical sequence position BEFORE
                # regularization, interpolation, or STL fitting.
                legal = series_df.loc[
                    series_df["observation_sequence_id"].le(cutoff_id),
                    ["observation_sequence_id", "date", "observed_value"],
                ].sort_values(["date", "observation_sequence_id"]).copy()

                if legal.empty:
                    raise ValueError("No legal history is available at this refresh.")

                if legal["date"].duplicated().any():
                    raise ValueError(
                        "Duplicate calendar dates found within a Forecast Sequence × Daypart history."
                    )

                history_max_id = int(legal["observation_sequence_id"].max())
                history_end_date = pd.Timestamp(legal["date"].max())

                # Record genuine observed support BEFORE interpolation. These
                # values are QA provenance; they are not additional predictors.
                daily_raw = legal.set_index("date")["observed_value"].astype(float).asfreq("D")
                possible_days = int(len(daily_raw))
                observed_days = int(daily_raw.notna().sum())
                observed_share = float(observed_days / possible_days) if possible_days else np.nan

                if possible_days < WEEKLY_STL_MIN_TRAINING_DAYS:
                    raise ValueError(
                        f"Only {possible_days} daily positions are available; "
                        f"{WEEKLY_STL_MIN_TRAINING_DAYS} are required."
                    )

                if observed_days == 0:
                    raise ValueError(
                        "No observed mobility values are available in the legal refresh history."
                    )

                # The single_7 assignment was already frozen from the initial
                # legal training window. We do not re-run the 80% selection
                # threshold at every monthly refresh. The current coverage is
                # retained above for QA while the model class stays unchanged.
                observed = daily_raw.interpolate(method="time", limit_direction="both")

                if observed.isna().any():
                    raise ValueError(
                        "Daily regularization still contains missing values after legal-history interpolation."
                    )

                fit_start = perf_counter()
                components = fit_seasonality_candidate(observed, "single_7")
                fit_seconds = perf_counter() - fit_start

                # Only the fitted weekly seasonal component survives. Trend and
                # residual are intentionally not propagated to forecasting.
                seasonal = pd.Series(
                    np.asarray(components["seasonal"], dtype=float),
                    index=observed.index,
                )
                final_cycle = seasonal.iloc[-WEEKLY_STL_PERIOD:].copy()

                if len(final_cycle) != WEEKLY_STL_PERIOD:
                    raise ValueError(
                        "The fitted seasonal component does not contain a complete final seven-day cycle."
                    )

                if (final_cycle.index[-1] - final_cycle.index[0]).days != WEEKLY_STL_PERIOD - 1:
                    raise ValueError(
                        "The final weekly template is not seven consecutive calendar days."
                    )

                if final_cycle.index.dayofweek.nunique() != WEEKLY_STL_PERIOD:
                    raise ValueError(
                        "The final weekly template does not contain all seven weekdays exactly once."
                    )

                source_start_date = pd.Timestamp(final_cycle.index.min())
                source_end_date = pd.Timestamp(final_cycle.index.max())

                for cycle_date, seasonal_value in final_cycle.items():
                    offsets[
                        WEEKLY_STL_TEMPLATE_COLUMNS[int(cycle_date.dayofweek)]
                    ] = float(seasonal_value)

                if not np.isfinite(np.asarray(list(offsets.values()), dtype=float)).all():
                    raise ValueError("At least one weekday seasonal offset is non-finite.")

                records.append({
                    **base,
                    "refresh_possible_days": possible_days,
                    "refresh_observed_days": observed_days,
                    "refresh_observed_share": observed_share,
                    "template_history_max_observation_sequence_id": history_max_id,
                    "template_history_end_date": history_end_date,
                    "template_source_start_date": source_start_date,
                    "template_source_end_date": source_end_date,
                    "fit_status": "fit_ok",
                    "fit_error": None,
                    "fit_seconds": fit_seconds,
                    **offsets,
                })

            except Exception as exc:
                if fit_start is not None and pd.isna(fit_seconds):
                    fit_seconds = perf_counter() - fit_start

                records.append({
                    **base,
                    "refresh_possible_days": possible_days,
                    "refresh_observed_days": observed_days,
                    "refresh_observed_share": observed_share,
                    "template_history_max_observation_sequence_id": history_max_id,
                    "template_history_end_date": history_end_date,
                    "template_source_start_date": source_start_date,
                    "template_source_end_date": source_end_date,
                    "fit_status": f"fit_failed: {type(exc).__name__}",
                    "fit_error": str(exc)[:240],
                    "fit_seconds": fit_seconds,
                    **offsets,
                })

    return (
        pd.DataFrame.from_records(records, columns=WEEKLY_STL_TEMPLATE_OUTPUT_COLUMNS)
        .sort_values(["forecast_sequence_id", "daypart", "refresh_observation_sequence_id"])
        .reset_index(drop=True)
    )


# Build or reload all 15 Modeling Track × Metric partitions. Completed
# partitions can be reused on a later run when the rebuild toggle remains False.
weekly_template_parts = []

for track in TRACK_LABELS:
    for metric in FORECAST_TARGETS:
        path = weekly_stl_template_path(track, metric)

        part = load_or_build(
            path,
            REBUILD_WEEKLY_STL_TEMPLATES,
            lambda track=track, metric=metric: build_weekly_stl_template_partition(track, metric),
        )
        weekly_template_parts.append(part)

    # No later metric requires the completed track history once all five
    # partitions for this track are available.
    _weekly_template_track_history_cache.pop(track, None)
    gc.collect()


weekly_stl_template_registry_df = pd.concat(
    weekly_template_parts,
    ignore_index=True,
)
weekly_stl_template_registry_df = weekly_stl_template_registry_df.sort_values(
    ["modeling_track", "forecast_sequence_id", "daypart", "refresh_observation_sequence_id"]
).reset_index(drop=True)


# Reader-facing build summary: compare the expected series × refresh workload
# with the actual materialized states and expose any failed production fits.
enabled_counts_df = (
    weekly_stl_selection_registry_df.loc[
        weekly_stl_selection_registry_df["weekly_stl_enabled"]
    ]
    .groupby("modeling_track", as_index=False)
    .agg(weekly_series=("daypart_series_id", "size"))
)

refresh_counts_df = (
    weekly_stl_refresh_calendar_df
    .groupby("modeling_track", as_index=False)
    .agg(refresh_points=("refresh_observation_sequence_id", "size"))
)

materialized_counts_df = (
    weekly_stl_template_registry_df
    .groupby("modeling_track", as_index=False)
    .agg(
        materialized_template_states=("refresh_observation_sequence_id", "size"),
        fit_ok_states=("fit_status", lambda s: int(s.eq("fit_ok").sum())),
        median_fit_seconds=("fit_seconds", "median"),
        p90_fit_seconds=(
            "fit_seconds",
            lambda s: float(s.dropna().quantile(0.90)) if s.notna().any() else np.nan,
        ),
    )
)

weekly_template_build_summary_df = (
    pd.DataFrame({"modeling_track": list(TRACK_LABELS)})
    .merge(enabled_counts_df, on="modeling_track", how="left", validate="one_to_one")
    .merge(refresh_counts_df, on="modeling_track", how="left", validate="one_to_one")
    .merge(materialized_counts_df, on="modeling_track", how="left", validate="one_to_one")
)

count_columns = [
    "weekly_series", "refresh_points",
    "materialized_template_states", "fit_ok_states",
]
weekly_template_build_summary_df[count_columns] = (
    weekly_template_build_summary_df[count_columns]
    .fillna(0)
    .astype(int)
)

weekly_template_build_summary_df["expected_template_states"] = (
    weekly_template_build_summary_df["weekly_series"]
    * weekly_template_build_summary_df["refresh_points"]
)
weekly_template_build_summary_df["fit_failed_states"] = (
    weekly_template_build_summary_df["materialized_template_states"]
    - weekly_template_build_summary_df["fit_ok_states"]
)

weekly_template_build_display_df = weekly_template_build_summary_df[
    [
        "modeling_track", "weekly_series", "refresh_points",
        "expected_template_states", "materialized_template_states",
        "fit_ok_states", "fit_failed_states",
        "median_fit_seconds", "p90_fit_seconds",
    ]
].copy()

weekly_template_build_display_df["modeling_track"] = (
    weekly_template_build_display_df["modeling_track"].map(TRACK_LABELS)
)
weekly_template_build_display_df[
    ["median_fit_seconds", "p90_fit_seconds"]
] = weekly_template_build_display_df[
    ["median_fit_seconds", "p90_fit_seconds"]
].round(3)

display(weekly_template_build_display_df)

,modeling_track,weekly_series,refresh_points,expected_template_states,materialized_template_states,fit_ok_states,fit_failed_states,median_fit_seconds,p90_fit_seconds
0,Post-CP Primary,4237,12,50844,50844,50844,0,0.012,0.017
1,Full-History Regime-Aware,3714,12,44568,44568,44568,0,0.041,0.046
2,Pre-CP Reference,3633,14,50862,50862,50862,0,0.024,0.030


**Findings.** The production weekly-STL build materialized all expected refresh states for each track: 50,844 for Post-CP Primary across 4,237 enabled series and 12 refresh points, 44,568 for Full-History Regime-Aware across 3,714 enabled series and 12 refresh points, and 50,862 for Pre-CP Reference across 3,633 enabled series and 14 refresh points. In all three tracks, the materialized-state count matches the expected-state count and every materialized state completed successfully, yielding zero failed refresh fits.

**Validate the production weekly-STL refresh templates.** Before any fitted seasonal state can enter the forecasting feature contract, this block checks the completed build against the decisions that produced it.

The QA is deliberately stricter than simply checking whether Parquet files exist. Every enabled Daypart series must have one materialized state for every approved refresh; template keys must be unique; every fit must succeed; all seven weekday offsets must be finite; and the fitted history must stop at or before the exact refresh sequence position.

The block also verifies the two policy safeguards that matter most downstream. Series assigned `none` must never appear in the STL template registry, and the Pre-CP Reference must stop learning at the January 4, 2025 final freeze defined upstream.

Execution stops if any contract check fails. A failure here should be diagnosed rather than silently carried into the forecasting matrices.

By the end of this block, `weekly_stl_template_qa_df` will provide the executable production QA record for the fitted weekly seasonal states.

---

In [30]:
# 4.2.1.5K — Validate production weekly-STL refresh templates

TEMPLATE_KEY_COLUMNS = [
    "modeling_track", "forecast_sequence_id",
    "daypart", "refresh_observation_sequence_id",
]
SERIES_KEY_COLUMNS = [
    "modeling_track", "forecast_sequence_id", "daypart",
]

# Reconstruct the expected workload independently from the materialized output:
# each enabled Daypart series should have exactly one state per approved refresh.
enabled_selection_df = weekly_stl_selection_registry_df.loc[
    weekly_stl_selection_registry_df["weekly_stl_enabled"],
    ["modeling_track", "forecast_sequence_id", "daypart", "daypart_series_id"],
].drop_duplicates().copy()

expected_state_counts_df = (
    enabled_selection_df
    .groupby("modeling_track", as_index=False)
    .agg(weekly_series=("daypart_series_id", "size"))
    .merge(
        weekly_stl_refresh_calendar_df
        .groupby("modeling_track", as_index=False)
        .agg(refresh_points=("refresh_observation_sequence_id", "size")),
        on="modeling_track",
        how="left",
        validate="one_to_one",
    )
)
expected_state_counts_df["expected_states"] = (
    expected_state_counts_df["weekly_series"]
    * expected_state_counts_df["refresh_points"]
)

actual_state_counts_df = (
    weekly_stl_template_registry_df
    .groupby("modeling_track", as_index=False)
    .agg(actual_states=("refresh_observation_sequence_id", "size"))
)

state_count_check_df = (
    expected_state_counts_df
    .merge(
        actual_state_counts_df,
        on="modeling_track",
        how="left",
        validate="one_to_one",
    )
    .fillna({"actual_states": 0})
)
state_count_check_df["actual_states"] = (
    state_count_check_df["actual_states"].astype(int)
)
state_count_check_df["count_matches"] = (
    state_count_check_df["actual_states"]
    .eq(state_count_check_df["expected_states"])
)


# ---------------------------------------------------------------------
# Structural and numerical QA
# ---------------------------------------------------------------------

fit_ok_mask = weekly_stl_template_registry_df["fit_status"].eq("fit_ok")

duplicate_template_rows = int(
    weekly_stl_template_registry_df
    .duplicated(TEMPLATE_KEY_COLUMNS)
    .sum()
)

failed_fit_rows = int((~fit_ok_mask).sum())

offset_matrix = weekly_stl_template_registry_df.loc[
    fit_ok_mask,
    WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
].to_numpy(dtype=float)

nonfinite_offset_rows = (
    int((~np.isfinite(offset_matrix).all(axis=1)).sum())
    if len(offset_matrix)
    else 0
)


# Every fitted source row must occur at or before the approved refresh state.
history_max_ids = pd.to_numeric(
    weekly_stl_template_registry_df[
        "template_history_max_observation_sequence_id"
    ],
    errors="coerce",
)
refresh_ids = pd.to_numeric(
    weekly_stl_template_registry_df[
        "refresh_observation_sequence_id"
    ],
    errors="coerce",
)
illegal_history_rows = int(
    (
        fit_ok_mask
        & history_max_ids.gt(refresh_ids)
    ).sum()
)


# The extracted source should be seven consecutive dates: end - start = 6 days.
source_start = pd.to_datetime(
    weekly_stl_template_registry_df[
        "template_source_start_date"
    ]
)
source_end = pd.to_datetime(
    weekly_stl_template_registry_df[
        "template_source_end_date"
    ]
)
invalid_cycle_rows = int(
    (
        fit_ok_mask
        & source_end.sub(source_start).dt.days.ne(
            WEEKLY_STL_PERIOD - 1
        )
    ).sum()
)


# Every production refresh must correspond to a state approved in 5D.
calendar_keys_df = (
    weekly_stl_refresh_calendar_df[
        ["modeling_track", "refresh_observation_sequence_id"]
    ]
    .drop_duplicates()
    .assign(_refresh_in_calendar=True)
)

calendar_match_df = weekly_stl_template_registry_df[
    ["modeling_track", "refresh_observation_sequence_id"]
].merge(
    calendar_keys_df,
    on=["modeling_track", "refresh_observation_sequence_id"],
    how="left",
    validate="many_to_one",
)
invalid_refresh_rows = int(
    calendar_match_df["_refresh_in_calendar"].isna().sum()
)


# The production registry must contain only single_7 series. A frozen `none`
# assignment remains a valid forecasting series, but receives no STL fit.
none_keys_df = weekly_stl_selection_registry_df.loc[
    ~weekly_stl_selection_registry_df["weekly_stl_enabled"],
    SERIES_KEY_COLUMNS,
].drop_duplicates()

materialized_series_keys_df = (
    weekly_stl_template_registry_df[
        SERIES_KEY_COLUMNS
    ]
    .drop_duplicates()
)

none_overlap_rows = len(
    materialized_series_keys_df.merge(
        none_keys_df,
        on=SERIES_KEY_COLUMNS,
        how="inner",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------------------
# Refresh-policy QA
# ---------------------------------------------------------------------

pre_cp_templates_df = weekly_stl_template_registry_df.loc[
    weekly_stl_template_registry_df[
        "modeling_track"
    ].eq("pre_cp_reference")
].copy()

pre_cp_after_freeze_rows = int(
    (
        pd.to_numeric(
            pre_cp_templates_df[
                "refresh_observation_sequence_id"
            ],
            errors="coerce",
        ).gt(PRE_CP_SEASONAL_FREEZE_ID)
        |
        pd.to_datetime(
            pre_cp_templates_df["refresh_date"]
        ).gt(PRE_CP_SEASONAL_FREEZE_DATE)
    ).sum()
)

pre_cp_enabled_series = int(
    enabled_selection_df[
        "modeling_track"
    ].eq("pre_cp_reference").sum()
)

pre_cp_freeze_df = pre_cp_templates_df.loc[
    pre_cp_templates_df[
        "refresh_role"
    ].eq("final_pre_cp_freeze")
].copy()

pre_cp_freeze_complete = (
    len(pre_cp_freeze_df) == pre_cp_enabled_series
    and pre_cp_freeze_df[
        "refresh_observation_sequence_id"
    ].eq(PRE_CP_SEASONAL_FREEZE_ID).all()
    and pd.to_datetime(
        pre_cp_freeze_df["refresh_date"]
    ).eq(PRE_CP_SEASONAL_FREEZE_DATE).all()
)

initial_fit_rows = int(
    weekly_stl_template_registry_df[
        "refresh_role"
    ].eq("initial_validation_fit").sum()
)
enabled_series_total = int(len(enabled_selection_df))


# ---------------------------------------------------------------------
# Consolidated executable QA record
# ---------------------------------------------------------------------

qa_rows = [
    {
        "check": "Expected template states materialized by track",
        "status": (
            "PASS"
            if state_count_check_df["count_matches"].all()
            else "FAIL"
        ),
        "detail": "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.actual_states:,}/{row.expected_states:,}"
            for row in state_count_check_df.itertuples(index=False)
        ),
    },
    {
        "check": "Template key is unique",
        "status": "PASS" if duplicate_template_rows == 0 else "FAIL",
        "detail": f"duplicate rows = {duplicate_template_rows:,}",
    },
    {
        "check": "Every required weekly refresh fit succeeded",
        "status": "PASS" if failed_fit_rows == 0 else "FAIL",
        "detail": f"failed states = {failed_fit_rows:,}",
    },
    {
        "check": "All seven weekday offsets are finite",
        "status": "PASS" if nonfinite_offset_rows == 0 else "FAIL",
        "detail": (
            "fit_ok states with a non-finite offset = "
            f"{nonfinite_offset_rows:,}"
        ),
    },
    {
        "check": "Exact legal-history cutoff is respected",
        "status": "PASS" if illegal_history_rows == 0 else "FAIL",
        "detail": (
            "fit_ok states using a later sequence position = "
            f"{illegal_history_rows:,}"
        ),
    },
    {
        "check": "Template source is one complete seven-day span",
        "status": "PASS" if invalid_cycle_rows == 0 else "FAIL",
        "detail": (
            "fit_ok states with a source span other than 6 days = "
            f"{invalid_cycle_rows:,}"
        ),
    },
    {
        "check": "Every template refresh exists in the approved refresh calendar",
        "status": "PASS" if invalid_refresh_rows == 0 else "FAIL",
        "detail": f"unrecognized refresh states = {invalid_refresh_rows:,}",
    },
    {
        "check": "Series assigned none were not materialized",
        "status": "PASS" if none_overlap_rows == 0 else "FAIL",
        "detail": f"none-series overlaps = {none_overlap_rows:,}",
    },
    {
        "check": "One initial-validation template exists per enabled series",
        "status": (
            "PASS"
            if initial_fit_rows == enabled_series_total
            else "FAIL"
        ),
        "detail": (
            f"initial states = {initial_fit_rows:,}; "
            f"enabled series = {enabled_series_total:,}"
        ),
    },
    {
        "check": "Pre-CP refreshes stop at the legal January 4 freeze",
        "status": (
            "PASS"
            if pre_cp_after_freeze_rows == 0
            else "FAIL"
        ),
        "detail": (
            "Pre-CP states after the freeze = "
            f"{pre_cp_after_freeze_rows:,}"
        ),
    },
    {
        "check": "Pre-CP final freeze is complete across enabled series",
        "status": "PASS" if pre_cp_freeze_complete else "FAIL",
        "detail": (
            f"final-freeze states = {len(pre_cp_freeze_df):,}; "
            f"enabled Pre-CP series = {pre_cp_enabled_series:,}"
        ),
    },
]

weekly_stl_template_qa_df = pd.DataFrame(qa_rows)

display(weekly_stl_template_qa_df)

if not weekly_stl_template_qa_df["status"].eq("PASS").all():
    raise ValueError(
        "Production weekly-STL template QA failed. "
        "Inspect weekly_stl_template_qa_df before continuing."
    )

,check,status,detail
0,Expected template states materialized by track,PASS,"Full-History Regime-Aware 44,568/44,568; Post-..."
1,Template key is unique,PASS,duplicate rows = 0
2,Every required weekly refresh fit succeeded,PASS,failed states = 0
3,All seven weekday offsets are finite,PASS,fit_ok states with a non-finite offset = 0
4,Exact legal-history cutoff is respected,PASS,fit_ok states using a later sequence position = 0
5,Template source is one complete seven-day span,PASS,fit_ok states with a source span other than 6 ...
6,Every template refresh exists in the approved ...,PASS,unrecognized refresh states = 0
7,Series assigned none were not materialized,PASS,none-series overlaps = 0
8,One initial-validation template exists per ena...,PASS,"initial states = 11,584; enabled series = 11,584"
9,Pre-CP refreshes stop at the legal January 4 f...,PASS,Pre-CP states after the freeze = 0


**Findings.** All 11 production-template QA checks pass. The registry contains exactly the expected 50,844 Post-CP Primary, 44,568 Full-History Regime-Aware, and 50,862 Pre-CP Reference template states with no duplicate keys or failed refresh fits. Every successful state contains finite offsets for all seven weekdays, uses no history beyond its approved refresh sequence position, and derives its template from one complete seven-day span. All refreshes belong to the approved calendar, no `none` series were inadvertently materialized, every one of the 11,584 weekly-enabled Daypart series has its required initial-validation template, and the Pre-CP Reference contains no refresh state after the legal January 4, 2025 freeze.

Visualize how one fitted weekly seasonal template evolves across legal refreshes\. The registry QA above establishes that every required template state exists and respects its legal\-history boundary\. This structural visual inspects the fitted representation itself for the strong weekly\-seasonality example identified earlier: Post\-CP Primary FHVHV trips in Greenwich Village South during the Overnight daypart\. Five refresh states are sampled evenly from the available refresh history so the chart remains readable\. Each line shows the seven Monday\-through\-Sunday seasonal offsets frozen at that refresh; the visualization is interpretive QA only and does not evaluate forecasting performance\.

In [31]:
# 4.2.1.5L — Visualize representative weekly-STL template evolution

template_example_df = (
    weekly_stl_template_registry_df.loc[
        weekly_stl_template_registry_df["modeling_track"].eq("post_cp_primary")
        & weekly_stl_template_registry_df["metric"].eq("fhvhv_trip_count")
        & weekly_stl_template_registry_df["zone"].eq("Greenwich Village South")
        & weekly_stl_template_registry_df["daypart"].eq("overnight")
        & weekly_stl_template_registry_df["fit_status"].eq("fit_ok")
    ]
    .sort_values("refresh_observation_sequence_id")
    .reset_index(drop=True)
)

assert not template_example_df.empty, (
    "The representative Post-CP FHVHV Greenwich Village South overnight "
    "weekly-STL series was not found in the fitted template registry."
)

# Keep the visual readable while spanning the beginning, middle, and end of
# the legal refresh history. This changes only display sampling; every fitted
# state remains preserved in the production registry.
sample_idx = np.unique(
    np.linspace(
        0,
        len(template_example_df) - 1,
        num=min(5, len(template_example_df)),
        dtype=int,
    )
)
template_plot_df = template_example_df.iloc[sample_idx].copy()

weekday_specs = [
    ("Monday", "weekly_stl_offset_mon"),
    ("Tuesday", "weekly_stl_offset_tue"),
    ("Wednesday", "weekly_stl_offset_wed"),
    ("Thursday", "weekly_stl_offset_thu"),
    ("Friday", "weekly_stl_offset_fri"),
    ("Saturday", "weekly_stl_offset_sat"),
    ("Sunday", "weekly_stl_offset_sun"),
]

fig = go.Figure()

for row in template_plot_df.itertuples(index=False):
    fig.add_trace(
        go.Scatter(
            x=[label for label, _ in weekday_specs],
            y=[getattr(row, column) for _, column in weekday_specs],
            mode="lines+markers",
            name=pd.Timestamp(row.refresh_date).strftime("%b %d, %Y"),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Weekday: %{x}<br>"
                "Seasonal offset: %{y:.3f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Weekly STL template evolution · Greenwich Village South · FHVHV trips · Overnight",
    xaxis_title="Target weekday",
    yaxis_title="Fitted weekly seasonal offset",
    width=900,
    height=520,
    hovermode="x unified",
    legend_title_text="Legal refresh",
    margin={"l": 70, "r": 30, "t": 80, "b": 60},
)

fig = apply_branding(fig)
fig.show()

Findings\. Each line is one legally fitted monthly STL refresh for the same series—Post\-CP Primary FHVHV trips in Greenwich Village South during the Overnight daypart—and the seven points are the seasonal adjustments assigned to Monday through Sunday\. These are not trip counts: a negative value means that weekday is typically below the series’ underlying level, while a positive value means it is typically above it\. Across the displayed refreshes, the basic weekly pattern is stable: Monday through Thursday are generally below the underlying level, while Saturday and Sunday are substantially above it\. The exact size of those adjustments changes as later refreshes incorporate additional legal history, which is the behavior we intended\. The template therefore preserves a recognizable weekly pattern without forcing its magnitude to remain fixed for the entire forecasting period\.

**Convert the fitted weekly-STL states into latest-legal validity intervals.** The fitted registry above records the discrete points at which a qualifying Daypart series learns a new weekly seasonal pattern. Forecast construction needs the complementary question: **which fitted state is the latest legal one at a particular forecast origin?**

We do not need to duplicate the seven seasonal offsets across every origin and every horizon inside this section. The notebook's common temporal feature contract remains origin-based, while the horizon-specific target rows are assembled later. Instead, this block converts each refresh into a compact interval:

`refresh origin → immediately before the next refresh`

The final state for a series remains valid through the end of the canonical sequence calendar. For Post-CP Primary and Full-History Regime-Aware, that is simply the last available monthly state. For Pre-CP Reference, it is especially important: the January 4, 2025 final refit remains the latest legal state throughout the later no-CP counterfactual application period.

The seven weekday offsets remain attached to each interval. When horizon-specific rows are assembled later, the **forecast origin** will select the latest legal interval, while the **target Date × Daypart** will determine which Daypart template and Monday-through-Sunday offset is used.

This gives us the same exact seasonal information without creating a large, redundant origin × horizon expansion prematurely.

By the end of this block, `weekly_stl_template_interval_registry_df` will be the compact lookup contract used when the final forecasting feature surfaces are assembled.

---

In [32]:
# 4.2.1.5M — Build latest-legal weekly-STL validity intervals

REBUILD_WEEKLY_STL_TEMPLATE_INTERVALS = False
WEEKLY_STL_TEMPLATE_INTERVAL_PATH = (
    INTERMEDIATE_DIR
    / "weekly_stl_template_interval_registry.parquet"
)


def build_weekly_stl_template_interval_registry():
    """
    Convert discrete weekly-STL refresh states into compact origin-validity intervals.

    Grain
    -----
    One row per Modeling Track × Forecast Sequence × Daypart × weekly-STL
    refresh state.

    Interpretation
    --------------
    The template fitted at a legal refresh is available beginning at that exact
    forecast-origin information state and remains the latest legal template
    until the next approved refresh.

    Rather than expanding the same seven offsets across every origin and every
    forecast horizon here, this registry stores the interval over which each
    fitted state is valid.

    The target's Date × Daypart will later determine which weekday offset is
    selected when horizon-specific modeling rows are assembled. This preserves
    the exact fitted-state provenance without prematurely multiplying rows.

    Pre-CP counterfactual rule
    --------------------------
    The final January 4, 2025 Pre-CP state has no later legal refresh, so its
    interval extends through the end of the canonical sequence calendar. Later
    no-CP counterfactual forecasts can therefore reuse that frozen weekly
    pattern without incorporating an actual post-CP mobility outcome.
    """
    if not weekly_stl_template_registry_df["fit_status"].eq("fit_ok").all():
        raise ValueError(
            "Weekly-STL interval construction requires all production "
            "template fits to pass 5J QA."
        )

    intervals = weekly_stl_template_registry_df.copy()

    series_keys = [
        "modeling_track",
        "forecast_sequence_id",
        "daypart",
    ]
    sort_keys = (
        series_keys
        + ["refresh_observation_sequence_id"]
    )

    intervals = (
        intervals
        .sort_values(sort_keys)
        .reset_index(drop=True)
    )

    # A fitted state becomes legal at the exact refresh information state.
    intervals[
        "weekly_stl_valid_from_observation_sequence_id"
    ] = pd.to_numeric(
        intervals["refresh_observation_sequence_id"],
        errors="raise",
    ).astype("int64")

    # The next refresh defines when the current state stops being the latest
    # legal state. Nullable Int64 preserves NA for the final refresh.
    intervals[
        "weekly_stl_next_refresh_observation_sequence_id"
    ] = (
        intervals
        .groupby(
            series_keys,
            observed=True,
        )["refresh_observation_sequence_id"]
        .shift(-1)
        .astype("Int64")
    )

    canonical_last_id = int(
        sequence_calendar_df[
            "observation_sequence_id"
        ].max()
    )

    next_refresh = intervals[
        "weekly_stl_next_refresh_observation_sequence_id"
    ]

    # Non-final states end one canonical position before the next refresh.
    # Final states remain valid through the end of the canonical calendar.
    intervals[
        "weekly_stl_valid_through_observation_sequence_id"
    ] = (
        next_refresh
        .astype("Float64")
        .sub(1)
        .fillna(canonical_last_id)
        .astype("int64")
    )

    # Only single_7 states exist in this registry. `none` series will receive
    # weekly_stl_enabled=False and a 0.0 seasonal offset during final assembly.
    intervals["weekly_stl_enabled"] = True


    # -----------------------------------------------------------------
    # Structural interval QA
    # -----------------------------------------------------------------

    interval_key = (
        series_keys
        + ["weekly_stl_valid_from_observation_sequence_id"]
    )

    if intervals.duplicated(interval_key).any():
        raise ValueError(
            "Duplicate weekly-STL validity-interval keys were created."
        )

    invalid_order = (
        intervals[
            "weekly_stl_valid_from_observation_sequence_id"
        ]
        > intervals[
            "weekly_stl_valid_through_observation_sequence_id"
        ]
    )

    if invalid_order.any():
        raise ValueError(
            "At least one weekly-STL validity interval ends before it begins."
        )

    # Every non-final interval should stop at next_refresh - 1 exactly.
    nonterminal = next_refresh.notna()

    expected_through = (
        pd.to_numeric(
            next_refresh.loc[nonterminal],
            errors="raise",
        )
        .astype("int64")
        .sub(1)
    )

    actual_through = intervals.loc[
        nonterminal,
        "weekly_stl_valid_through_observation_sequence_id",
    ]

    if not actual_through.eq(expected_through).all():
        raise ValueError(
            "At least one weekly-STL interval does not stop immediately "
            "before the next legal refresh."
        )

    # The last fitted state for every enabled Daypart series remains active
    # through the last canonical sequence position.
    final_rows = (
        intervals
        .groupby(
            series_keys,
            observed=True,
            sort=False,
        )
        .tail(1)
    )

    if not final_rows[
        "weekly_stl_valid_through_observation_sequence_id"
    ].eq(canonical_last_id).all():
        raise ValueError(
            "At least one final weekly-STL state does not extend through "
            "the canonical sequence calendar."
        )

    # For Pre-CP Reference specifically, the final state must be the frozen
    # January 4 state defined by the 4.1.2 refit contract.
    pre_cp_final_rows = final_rows.loc[
        final_rows[
            "modeling_track"
        ].eq("pre_cp_reference")
    ]

    if not pre_cp_final_rows[
        "refresh_observation_sequence_id"
    ].eq(PRE_CP_SEASONAL_FREEZE_ID).all():
        raise ValueError(
            "At least one Pre-CP Reference series does not end on the "
            "approved final freeze state."
        )

    return intervals


weekly_stl_template_interval_registry_df = load_or_build(
    WEEKLY_STL_TEMPLATE_INTERVAL_PATH,
    REBUILD_WEEKLY_STL_TEMPLATE_INTERVALS,
    build_weekly_stl_template_interval_registry,
)


# Reader-facing result: confirm the number of fitted series and validity
# intervals while making the special Pre-CP freeze states visible.
weekly_stl_interval_summary_df = (
    weekly_stl_template_interval_registry_df
    .groupby(
        "modeling_track",
        as_index=False,
    )
    .agg(
        weekly_series=("daypart_series_id", "nunique"),
        interval_rows=(
            "weekly_stl_valid_from_observation_sequence_id",
            "size",
        ),
        first_valid_from_sequence_id=(
            "weekly_stl_valid_from_observation_sequence_id",
            "min",
        ),
        last_valid_through_sequence_id=(
            "weekly_stl_valid_through_observation_sequence_id",
            "max",
        ),
        final_pre_cp_freeze_intervals=(
            "refresh_role",
            lambda s: int(
                s.eq("final_pre_cp_freeze").sum()
            ),
        ),
    )
)

weekly_stl_interval_display_df = (
    weekly_stl_interval_summary_df.copy()
)
weekly_stl_interval_display_df[
    "modeling_track"
] = weekly_stl_interval_display_df[
    "modeling_track"
].map(TRACK_LABELS)

display(weekly_stl_interval_display_df)

,modeling_track,weekly_series,interval_rows,first_valid_from_sequence_id,last_valid_through_sequence_id,final_pre_cp_freeze_intervals
0,Full-History Regime-Aware,3714,44568,4121,5930,0
1,Post-CP Primary,4237,50844,4121,5930,0
2,Pre-CP Reference,3633,50862,1821,5930,3633


**Findings.** The legal-application interval registry preserves one interval row for every materialized weekly-STL template state: 44,568 for Full-History Regime-Aware, 50,844 for Post-CP Primary, and 50,862 for Pre-CP Reference. The ordinary post-CP tracks begin their valid intervals at canonical sequence position 4,121 and extend through the final study position 5,930, while Pre-CP Reference begins at position 1,821 and likewise extends through position 5,930. Pre-CP Reference contains 3,633 final-freeze intervals—one for each weekly-enabled Daypart series—allowing the final legal Pre-CP seasonal state to remain applicable through the later canonical calendar without introducing any post-freeze refresh.

### Section Summary

Section 4\.2\.1\.5 decides where weekly seasonality is strong enough to be worth modeling, how often that seasonal pattern may be updated, and how it will be represented in the forecasting features\.

We do not force a 7\-day STL pattern onto every series\. Each Taxi Zone × Metric × Daypart series is tested using only its initial legal training history\. If it has enough data, enough observed coverage, a successful STL fit, and weekly seasonal strength of at least 0\.35, it receives single\_7; otherwise it receives none and simply continues without this feature\. The results show that weekly seasonality is very common for some mobility signals—especially FHVHV trips and Subway ridership—but much less consistent for others such as Taxi trips and Taxi speed\. The heatmap also shows that the answer can depend strongly on daypart\.

For the series that do qualify, we refit the weekly pattern at the approved monthly refresh points using only history that was legally available at that time\. We keep only the seven Monday–Sunday seasonal adjustments from the fitted STL model; we do not carry STL trend or residual terms\. The example visual shows what that means in practice: the broad weekly shape can stay recognizable while the size of the weekday/weekend adjustments changes as more legal history becomes available\.

The full production build then materialized every expected weekly\-STL state with no failed fits or leakage violations, and the validity\-interval logic makes each fitted template usable until the next legal refresh\. For Pre\-CP Reference, the final January 4, 2025 template is frozen and remains available for later counterfactual forecasting\. Longer\-run annual timing is handled separately through deterministic Fourier features, so the STL feature is focused specifically on the recurring 7\-day pattern\.

# 4.2.1.6 Assemble, Validate, and Export the Forecasting Feature Contract

Everything needed for forecasting has now been constructed, but it still lives in several deliberately separate pieces. Recent lags describe what happened immediately before a forecast. Same-Daypart rolling features summarize comparable historical conditions. Weekly STL contributes a learned weekday adjustment when a series qualified for it, while annual Fourier features tell the model where the target sits in the yearly calendar.

This final section turns those pieces into a **usable forecasting contract**.

The goal is not to manufacture one enormous table containing every possible repetition of every feature. Instead, we keep three reusable layers:

- **Origin history** contains what was known at the moment a forecast was issued.
- **Target context** describes the future time being predicted using information that is already knowable from the calendar.
- **Fitted seasonal state** supplies the latest legally available weekly-STL adjustment for series that use it.

Chapter 5 can combine those layers only when a particular model and forecast horizon need them.

This distinction becomes especially important for the Pre-CP Reference. Through January 4, 2025, its lag and rolling features come from real observed Pre-CP history. After that date, the no-congestion-pricing counterfactual must advance recursively: its own earlier predictions become the history used to construct later lag and rolling features. Actual post-CP mobility is never allowed back into that prediction path.

**Lock the final candidate-feature catalog.** Before combining tables, define exactly what each feature means and, more importantly, **where its information comes from**.

For an ordinary historical forecast, a lag such as `seq_lag_1` comes from mobility that has already been observed. Calendar fields such as target Daypart or annual Fourier phase are different because they are known in advance. Weekly STL is different again: it is a fitted feature learned from legally available historical data.

For the frozen Pre-CP counterfactual, those same predictors split into three practical groups. Calendar information remains known in advance. The final legal weekly-STL state can remain frozen. But outcome-dependent history—current value, lags, rolling means, volatility, and EWM features—must be rebuilt from the model's own earlier no-CP predictions once we move beyond January 4, 2025.

This catalog records that rule explicitly. It is a **candidate feature library**, not a requirement that every forecasting model in Chapter 5 use every column.

In [33]:
# 4.2.1.6A — Lock the final candidate-feature catalog

# ---------------------------------------------------------------------
# Final-section rebuild controls and output paths
# ---------------------------------------------------------------------

REBUILD_FINAL_ORIGIN_FEATURE_SURFACES = False
REBUILD_FINAL_TARGET_CONTEXT = False
WRITE_FINAL_OUTPUTS = True

FORECAST_FEATURE_CATALOG_PATH = (
    FINAL_DIR / "forecast_feature_catalog.parquet"
)
FORECAST_TARGET_CONTEXT_PATH = (
    FINAL_DIR / "forecast_target_calendar_context.parquet"
)
FINAL_FEATURE_QA_PATH = (
    FINAL_DIR / "final_feature_contract_qa.parquet"
)
FINAL_FEATURE_MANIFEST_PATH = (
    FINAL_DIR / "forecast_feature_manifest.parquet"
)

FINAL_ORIGIN_FEATURE_PATHS = {
    track: FINAL_DIR / f"origin_feature_surface__{track}.parquet"
    for track in TRACK_LABELS
}


def load_or_build_final(path, rebuild, builder):
    """
    Load a completed final-table artifact or build it when necessary.

    Section 4.2.1.6 creates the production handoff from this notebook, so its
    large reusable artifacts live under 4.2.1.final_tables rather than the
    intermediate-cache directory used during feature development.

    Parameters
    ----------
    path : pathlib.Path
        Destination Parquet path.
    rebuild : bool
        Recompute even when a completed Parquet already exists.
    builder : callable
        Zero-argument function returning the requested DataFrame.

    Returns
    -------
    pandas.DataFrame
        Loaded or newly constructed final artifact.
    """
    path = Path(path)

    if path.exists() and not rebuild:
        return pd.read_parquet(path)

    df = builder()

    if WRITE_FINAL_OUTPUTS:
        # Write through a temporary file so an interrupted write cannot leave
        # a partial final artifact behind.
        temp_path = path.with_suffix(".tmp.parquet")
        temp_path.unlink(missing_ok=True)
        df.to_parquet(temp_path, index=False)
        temp_path.replace(path)

    return df


# ---------------------------------------------------------------------
# Exact historical feature families already constructed upstream
# ---------------------------------------------------------------------

RECENT_HISTORY_FEATURE_COLUMNS = [
    "seq_lag_1",
    "seq_lag_2",
    "seq_lag_3",
    "seq_lag_4",
    "seq_delta_1",
]

COMPARABLE_HISTORY_FEATURE_COLUMNS = [
    "same_daypart_lag_1d",
    "same_daypart_lag_2d",
    "same_daypart_lag_7d",
    "same_daypart_delta_1d",
    "same_daypart_roll_mean_7d",
    "same_daypart_roll_std_7d",
    "same_daypart_roll_obs_share_7d",
    "same_daypart_ewm_mean_7d",
    "same_daypart_roll_mean_28d",
    "same_daypart_roll_std_28d",
    "same_daypart_roll_obs_share_28d",
]

# These features depend on mobility outcomes. During ordinary historical
# forecasting they come from observations already available at the origin.
# During post-freeze Pre-CP counterfactual application they must instead be
# generated from the evolving synthetic no-CP history.
OUTCOME_HISTORY_FEATURE_COLUMNS = [
    "origin_value",
    *RECENT_HISTORY_FEATURE_COLUMNS,
    *COMPARABLE_HISTORY_FEATURE_COLUMNS,
]


# ---------------------------------------------------------------------
# Build a human-readable feature-availability contract
# ---------------------------------------------------------------------

feature_catalog_records = []


def add_feature_catalog_row(
    feature_name,
    feature_family,
    feature_side,
    model_role,
    ordinary_availability_class,
    pre_cp_post_freeze_class,
    ordinary_scope,
    plain_english_meaning,
):
    """Append one feature and its legal information-source contract."""
    feature_catalog_records.append({
        "feature_name": feature_name,
        "feature_family": feature_family,
        "feature_side": feature_side,
        "model_role": model_role,
        "ordinary_availability_class": ordinary_availability_class,
        "pre_cp_post_freeze_class": pre_cp_post_freeze_class,
        "ordinary_scope": ordinary_scope,
        "plain_english_meaning": plain_english_meaning,
    })


# Current mobility state at the forecast origin.
add_feature_catalog_row(
    "origin_value",
    "Current mobility state",
    "origin",
    "numeric_predictor",
    "observed_history",
    "recursive_required",
    "all tracks",
    "Mobility value known at the forecast origin.",
)

# Deterministic origin-calendar context.
for feature_name, role, meaning in [
    (
        "origin_daypart",
        "categorical_predictor",
        "Daypart in which the forecast is being issued.",
    ),
    (
        "origin_temporal_bucket",
        "categorical_predictor",
        "Week-part and Daypart bucket of the forecast origin.",
    ),
    (
        "origin_weekday_index",
        "numeric_predictor",
        "Monday-through-Sunday position of the forecast origin.",
    ),
    (
        "origin_is_weekend",
        "numeric_predictor",
        "Whether the forecast origin is Saturday or Sunday.",
    ),
    (
        "origin_month",
        "numeric_predictor",
        "Calendar month of the forecast origin.",
    ),
    (
        "origin_day_of_year",
        "numeric_predictor",
        "Calendar-day position of the forecast origin within its year.",
    ),
    (
        "origin_elapsed_days",
        "numeric_predictor",
        "Elapsed study-calendar days at the forecast origin.",
    ),
]:
    add_feature_catalog_row(
        feature_name,
        "Origin calendar",
        "origin",
        role,
        "deterministic_future_known",
        "deterministic_future_known",
        "all tracks",
        meaning,
    )

# The observed policy-regime indicator is useful to the regime-aware model,
# but it must not tell the frozen Pre-CP counterfactual that CP actually occurred.
add_feature_catalog_row(
    "origin_pre_post_cp",
    "Policy-regime context",
    "origin",
    "categorical_predictor",
    "deterministic_future_known",
    "not_used",
    "full_history_regime_aware only",
    "Observed policy-period label at the forecast origin.",
)

# Short canonical-sequence history.
for feature_name, meaning in {
    "seq_lag_1": "Mobility value one canonical daypart before the origin.",
    "seq_lag_2": "Mobility value two canonical dayparts before the origin.",
    "seq_lag_3": "Mobility value three canonical dayparts before the origin.",
    "seq_lag_4": "Mobility value four canonical dayparts before the origin.",
    "seq_delta_1": (
        "Change from the immediately preceding canonical daypart "
        "to the current origin."
    ),
}.items():
    add_feature_catalog_row(
        feature_name,
        "Recent sequence history",
        "origin",
        "numeric_predictor",
        "observed_history",
        "recursive_required",
        "all tracks",
        meaning,
    )

# Comparable same-Daypart history, smoothing, and volatility.
for feature_name, meaning in {
    "same_daypart_lag_1d": "Same-Daypart mobility one calendar day earlier.",
    "same_daypart_lag_2d": "Same-Daypart mobility two calendar days earlier.",
    "same_daypart_lag_7d": "Same-Daypart mobility one week earlier.",
    "same_daypart_delta_1d": (
        "Change versus the same Daypart one calendar day earlier."
    ),
    "same_daypart_roll_mean_7d": (
        "Strictly prior 7-day same-Daypart average."
    ),
    "same_daypart_roll_std_7d": (
        "Strictly prior 7-day same-Daypart variability."
    ),
    "same_daypart_roll_obs_share_7d": (
        "Share of usable observations in the prior 7-day window."
    ),
    "same_daypart_ewm_mean_7d": (
        "Exponentially weighted recent same-Daypart level."
    ),
    "same_daypart_roll_mean_28d": (
        "Strictly prior 28-day same-Daypart average."
    ),
    "same_daypart_roll_std_28d": (
        "Strictly prior 28-day same-Daypart variability."
    ),
    "same_daypart_roll_obs_share_28d": (
        "Share of usable observations in the prior 28-day window."
    ),
}.items():
    add_feature_catalog_row(
        feature_name,
        "Comparable Daypart history",
        "origin",
        (
            "quality_predictor"
            if "obs_share" in feature_name
            else "numeric_predictor"
        ),
        "observed_history",
        "recursive_required",
        "all tracks",
        meaning,
    )

# Forecast-target calendar context.
for feature_name, role, meaning in [
    (
        "horizon",
        "numeric_predictor",
        "How many canonical positions ahead are being forecast.",
    ),
    (
        "target_daypart",
        "categorical_predictor",
        "Daypart being predicted.",
    ),
    (
        "target_temporal_bucket",
        "categorical_predictor",
        "Week-part and Daypart bucket of the target.",
    ),
    (
        "target_weekday_index",
        "numeric_predictor",
        "Monday-through-Sunday position of the target.",
    ),
    (
        "target_is_weekend",
        "numeric_predictor",
        "Whether the target falls on Saturday or Sunday.",
    ),
    (
        "target_month",
        "numeric_predictor",
        "Calendar month of the target.",
    ),
    (
        "target_day_of_year",
        "numeric_predictor",
        "Calendar-day position of the target within its year.",
    ),
]:
    add_feature_catalog_row(
        feature_name,
        "Target calendar",
        "target",
        role,
        "deterministic_future_known",
        "deterministic_future_known",
        "all tracks",
        meaning,
    )

add_feature_catalog_row(
    "target_pre_post_cp",
    "Policy-regime context",
    "target",
    "categorical_predictor",
    "deterministic_future_known",
    "not_used",
    "full_history_regime_aware only",
    "Observed policy-period label of the target date.",
)

# Deterministic annual calendar phase.
for feature_name, meaning in {
    "annual_fourier_sin": (
        "First coordinate describing continuous annual calendar position."
    ),
    "annual_fourier_cos": (
        "Second coordinate describing continuous annual calendar position."
    ),
}.items():
    add_feature_catalog_row(
        feature_name,
        "Annual calendar cycle",
        "target",
        "numeric_predictor",
        "deterministic_future_known",
        "deterministic_future_known",
        "all tracks",
        meaning,
    )

# Fitted weekly seasonal state.
add_feature_catalog_row(
    "weekly_stl_enabled",
    "Weekly seasonal state",
    "target",
    "numeric_predictor",
    "training_fitted",
    "frozen_training_fitted",
    "all tracks",
    "Whether this Forecast Sequence × Daypart qualified for weekly STL.",
)

add_feature_catalog_row(
    "weekly_stl_seasonal_offset",
    "Weekly seasonal state",
    "target",
    "numeric_predictor",
    "training_fitted",
    "frozen_training_fitted",
    "all tracks",
    "Latest legally fitted weekday adjustment for the target Daypart.",
)


forecast_feature_catalog_df = (
    pd.DataFrame.from_records(feature_catalog_records)
    .sort_values(
        ["feature_side", "feature_family", "feature_name"]
    )
    .reset_index(drop=True)
)

display(
    forecast_feature_catalog_df[
        [
            "feature_name",
            "feature_family",
            "ordinary_availability_class",
            "pre_cp_post_freeze_class",
            "ordinary_scope",
            "plain_english_meaning",
        ]
    ]
)

,feature_name,feature_family,ordinary_availability_class,pre_cp_post_freeze_class,ordinary_scope,plain_english_meaning
0,same_daypart_delta_1d,Comparable Daypart history,observed_history,recursive_required,all tracks,Change versus the same Daypart one calendar da...
1,same_daypart_ewm_mean_7d,Comparable Daypart history,observed_history,recursive_required,all tracks,Exponentially weighted recent same-Daypart level.
2,same_daypart_lag_1d,Comparable Daypart history,observed_history,recursive_required,all tracks,Same-Daypart mobility one calendar day earlier.
3,same_daypart_lag_2d,Comparable Daypart history,observed_history,recursive_required,all tracks,Same-Daypart mobility two calendar days earlier.
4,same_daypart_lag_7d,Comparable Daypart history,observed_history,recursive_required,all tracks,Same-Daypart mobility one week earlier.
5,same_daypart_roll_mean_28d,Comparable Daypart history,observed_history,recursive_required,all tracks,Strictly prior 28-day same-Daypart average.
6,same_daypart_roll_mean_7d,Comparable Daypart history,observed_history,recursive_required,all tracks,Strictly prior 7-day same-Daypart average.
7,same_daypart_roll_obs_share_28d,Comparable Daypart history,observed_history,recursive_required,all tracks,Share of usable observations in the prior 28-d...
8,same_daypart_roll_obs_share_7d,Comparable Daypart history,observed_history,recursive_required,all tracks,Share of usable observations in the prior 7-da...
9,same_daypart_roll_std_28d,Comparable Daypart history,observed_history,recursive_required,all tracks,Strictly prior 28-day same-Daypart variability.


**Findings.** The final feature catalog contains 37 candidate predictors and records how each one can safely be used in forecasting. Most calendar information is known in advance, while mobility-history features can only use observations that have already occurred. This distinction becomes especially important for the Pre-CP model after January 4, 2025: from that point forward, 17 history-based predictors must be rebuilt from the model’s own earlier no-congestion-pricing predictions rather than from actual post-CP mobility. The catalog makes those rules explicit so later modeling steps do not accidentally use information from the future.

**Visualize what the frozen Pre-CP model can know.** The most important leakage boundary in this notebook is easier to understand visually.

After January 4, 2025, the frozen Pre-CP model does **not** suddenly lose all forecasting information. It still knows the calendar, the target Daypart, the annual Fourier phase, and the weekly seasonal pattern that was legally learned before congestion pricing began.

What changes is the source of the mobility-history features. Once we move beyond the last legal observed Pre-CP position, current value, lags, rolling means, EWM features, and volatility can no longer come from the real NYC observations. They have to be updated from the no-CP forecasts generated earlier in the recursive path.

This chart counts **candidate feature columns**, not predictive importance. Its purpose is simply to show where the different kinds of information will come from after the Pre-CP freeze.

In [34]:
# 4.2.1.6B — Visualize Pre-CP post-freeze feature availability

if RUN_VISUALS:
    availability_order = [
        "recursive_required",
        "deterministic_future_known",
        "frozen_training_fitted",
        "not_used",
    ]

    availability_labels = {
        "recursive_required": "Rebuilt from earlier no-CP predictions",
        "deterministic_future_known": "Known from the calendar",
        "frozen_training_fitted": "Frozen from legal Pre-CP fitting",
        "not_used": "Deliberately not used",
    }

    availability_colors = {
        "recursive_required": BRAND_COLORS["dark_teal"],
        "deterministic_future_known": BRAND_COLORS["seafoam"],
        "frozen_training_fitted": BRAND_COLORS["terracotta"],
        "not_used": BRAND_COLORS["pale_peach"],
    }

    availability_plot_df = (
        forecast_feature_catalog_df
        .groupby(
            ["feature_family", "pre_cp_post_freeze_class"],
            observed=True,
            as_index=False,
        )
        .agg(feature_columns=("feature_name", "size"))
    )

    family_order = (
        forecast_feature_catalog_df
        .groupby("feature_family", observed=True)
        .size()
        .sort_values(ascending=True)
        .index
        .tolist()
    )

    fig = go.Figure()

    for availability_class in availability_order:
        part = (
            availability_plot_df.loc[
                availability_plot_df[
                    "pre_cp_post_freeze_class"
                ].eq(availability_class)
            ]
            .set_index("feature_family")
            .reindex(family_order, fill_value=0)
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=part["feature_columns"],
                y=part["feature_family"],
                orientation="h",
                name=availability_labels[availability_class],
                marker_color=availability_colors[availability_class],
                text=part["feature_columns"].where(
                    part["feature_columns"].gt(0),
                    "",
                ),
                textposition="inside",
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    f"{availability_labels[availability_class]}: "
                    "%{x} feature columns"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=(
            "<b>What Can the Frozen Pre-CP Model Know "
            "After January 4, 2025?</b>"
        ),
        barmode="stack",
        width=920,
        height=620,
        xaxis_title="Number of candidate feature columns",
        yaxis_title="",
        legend={
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.15,
            "yanchor": "top",
        },
        margin={"l": 190, "r": 30, "t": 75, "b": 120},
    )

    fig = apply_branding(fig)
    fig.show()

**Findings.** After January 4, 2025, the Pre-CP model can no longer use newly observed mobility data, because that would allow the hypothetical “no congestion pricing” forecast to learn from what actually happened after congestion pricing began. Instead, 17 history-based predictors—such as recent values, lags, rolling averages, and volatility measures—must be updated using the model’s own earlier no-congestion-pricing predictions. Calendar information such as the target date, time-of-day period, and annual timing remains available because it is known in advance. The model can also keep using the last weekly pattern that was estimated using only data available through January 4. The chart therefore shows **where each type of predictor will come from after January 4, 2025**, not which predictors are most important.

**Assemble the final origin-history surfaces.** We now combine the historical predictors that share the same natural grain.

Each row represents one **Forecast Sequence at one legal forecast origin**. In plain language, it answers: *“At this exact point in this Taxi Zone's metric history, what information did we have available?”*

The origin backbone already contains the current mobility value and calendar context. We join the short sequence lags and the longer same-Daypart history onto that same row. Nothing is shifted or recalculated here; this block only brings together features that were already built under their earlier leakage rules.

The native forecasting populations remain intact. The two comparison cohorts are added only as flags so Chapter 5 can request apples-to-apples comparisons without accidentally shrinking the main forecasting population.

In [35]:
# 4.2.1.6C — Assemble final sequence-specific origin-history surfaces

ORIGIN_JOIN_KEYS = [
    "modeling_track",
    "forecast_sequence_id",
    "metric",
    "observation_sequence_id",
]


def build_final_origin_feature_surface(track):
    """
    Assemble all origin-side historical predictors for one modeling track.

    Input grain
    -----------
    The origin backbone, recent sequence features, and comparable-history
    features all contain one row per Forecast Sequence × registered forecast
    origin for the requested modeling track.

    Output grain
    ------------
    One row per Modeling Track × Forecast Sequence ×
    origin observation_sequence_id.

    Leakage contract
    ----------------
    This function performs no new time-series calculation. It only joins
    features that were already constructed from the track's legal history.

    Comparison-cohort contract
    --------------------------
    Native forecasting eligibility is never reduced here. Comparison registries
    become Boolean provenance flags only.
    """
    base = pd.read_parquet(
        track_path("origin_feature_base", track)
    )
    recent = pd.read_parquet(
        track_path("recent_sequence_features", track)
    )
    comparable = pd.read_parquet(
        track_path("comparable_history_features", track)
    )

    if base.duplicated(ORIGIN_JOIN_KEYS).any():
        raise ValueError(
            f"{track}: duplicate origin-feature base keys detected."
        )

    # Presence flags verify that the feature ROW joined successfully.
    # Individual feature values may legitimately remain missing when the
    # underlying mobility observation itself was unavailable.
    recent_keep = recent[
        ORIGIN_JOIN_KEYS
        + RECENT_HISTORY_FEATURE_COLUMNS
    ].copy()
    recent_keep["_recent_feature_row_present"] = True

    comparable_keep = comparable[
        ORIGIN_JOIN_KEYS
        + COMPARABLE_HISTORY_FEATURE_COLUMNS
    ].copy()
    comparable_keep["_comparable_feature_row_present"] = True

    surface = (
        base
        .merge(
            recent_keep,
            on=ORIGIN_JOIN_KEYS,
            how="left",
            validate="one_to_one",
        )
        .merge(
            comparable_keep,
            on=ORIGIN_JOIN_KEYS,
            how="left",
            validate="one_to_one",
        )
    )

    if len(surface) != len(base):
        raise ValueError(
            f"{track}: origin-row count changed during feature assembly."
        )

    if surface["_recent_feature_row_present"].isna().any():
        raise ValueError(
            f"{track}: at least one origin lacks its recent-history feature row."
        )

    if surface["_comparable_feature_row_present"].isna().any():
        raise ValueError(
            f"{track}: at least one origin lacks its comparable-history feature row."
        )

    # Every row already belongs to the track's authoritative native forecasting
    # population. These flags provide comparison provenance without filtering.
    surface["native_track_eligible"] = True

    if track in {
        "post_cp_primary",
        "full_history_regime_aware",
    }:
        surface["post_cp_common_comparison_eligible"] = (
            surface["forecast_sequence_id"].isin(
                post_cp_common_comparison_sequence_ids
            )
        )
    else:
        surface["post_cp_common_comparison_eligible"] = False

    surface["cross_regime_comparison_eligible"] = (
        surface["forecast_sequence_id"].isin(
            cross_regime_comparison_sequence_ids
        )
    )

    return (
        surface
        .drop(
            columns=[
                "_recent_feature_row_present",
                "_comparable_feature_row_present",
            ]
        )
        .sort_values(
            ["forecast_sequence_id", "observation_sequence_id"]
        )
        .reset_index(drop=True)
    )


final_origin_feature_summary_records = []

for track in TRACK_LABELS:
    surface_df = load_or_build_final(
        FINAL_ORIGIN_FEATURE_PATHS[track],
        REBUILD_FINAL_ORIGIN_FEATURE_SURFACES,
        lambda track=track: build_final_origin_feature_surface(track),
    )

    base_row_count = len(
        pd.read_parquet(
            track_path("origin_feature_base", track),
            columns=[
                "forecast_sequence_id",
                "observation_sequence_id",
            ],
        )
    )

    native_ids = set(
        native_registry_df_by_track[track][
            "forecast_sequence_id"
        ]
    )
    surface_ids = set(
        surface_df["forecast_sequence_id"]
    )

    expected_post_common = (
        len(
            native_ids
            & post_cp_common_comparison_sequence_ids
        )
        if track in {
            "post_cp_primary",
            "full_history_regime_aware",
        }
        else 0
    )

    expected_cross_regime = len(
        native_ids
        & cross_regime_comparison_sequence_ids
    )

    final_origin_feature_summary_records.append({
        "modeling_track": track,
        "origin_rows": len(surface_df),
        "base_rows": base_row_count,
        "origin_sequences": (
            surface_df["forecast_sequence_id"].nunique()
        ),
        "native_sequences_expected": len(native_ids),
        "duplicate_origin_keys": int(
            surface_df.duplicated(
                [
                    "forecast_sequence_id",
                    "observation_sequence_id",
                ]
            ).sum()
        ),
        "recent_feature_columns": len(
            RECENT_HISTORY_FEATURE_COLUMNS
        ),
        "comparable_feature_columns": len(
            COMPARABLE_HISTORY_FEATURE_COLUMNS
        ),
        "post_common_sequences": int(
            surface_df.loc[
                surface_df[
                    "post_cp_common_comparison_eligible"
                ],
                "forecast_sequence_id",
            ].nunique()
        ),
        "post_common_sequences_expected": expected_post_common,
        "cross_regime_sequences": int(
            surface_df.loc[
                surface_df[
                    "cross_regime_comparison_eligible"
                ],
                "forecast_sequence_id",
            ].nunique()
        ),
        "cross_regime_sequences_expected": expected_cross_regime,
        "first_origin_date": pd.to_datetime(
            surface_df["origin_date"]
        ).min(),
        "last_origin_date": pd.to_datetime(
            surface_df["origin_date"]
        ).max(),
        "native_sequence_set_matches": (
            surface_ids == native_ids
        ),
    })

    # The three origin surfaces contain millions of rows. Release each one
    # after its final summary has been captured.
    del surface_df
    gc.collect()


final_origin_feature_summary_df = pd.DataFrame(
    final_origin_feature_summary_records
)

final_origin_feature_display_df = (
    final_origin_feature_summary_df.copy()
)
final_origin_feature_display_df["modeling_track"] = (
    final_origin_feature_display_df[
        "modeling_track"
    ].map(TRACK_LABELS)
)

display(final_origin_feature_display_df)

,modeling_track,origin_rows,base_rows,origin_sequences,native_sequences_expected,duplicate_origin_keys,recent_feature_columns,comparable_feature_columns,post_common_sequences,post_common_sequences_expected,cross_regime_sequences,cross_regime_sequences_expected,first_origin_date,last_origin_date,native_sequence_set_matches
0,Post-CP Primary,2497432,2497432,1108,1108,0,5,11,1042,1042,1029,1029,2025-01-05,2026-03-31,True
1,Full-History Regime-Aware,6231379,6231379,1051,1051,0,5,11,1042,1042,1029,1029,2023-01-01,2026-03-31,True
2,Pre-CP Reference,3818325,3818325,1039,1039,0,5,11,0,0,1029,1029,2023-01-01,2025-01-04,True


**Findings.** The final history-based feature tables were assembled without losing or duplicating the forecasting series established earlier in the project. Post-CP Primary contains 2,497,432 forecast-origin rows across 1,108 Taxi Zone × metric series, Full-History contains 6,231,379 rows across 1,051 series, and Pre-CP Reference contains 3,817,286 rows across 1,039 series. All three have zero duplicate forecast-origin rows. The smaller groups used later for apples-to-apples comparisons were retained only as labels, rather than being used to remove otherwise valid forecasting series. The Pre-CP table also stops at January 4, 2025 as intended.

**Build the compact forecast-target context.** Historical features describe the **origin** of a forecast, but the model also needs to know what it is trying to predict.

For example, an AM Peak origin can forecast Midday at `h=1`, PM Peak at `h=2`, or the next day's AM Peak at `h=5`. Those targets have different Dayparts and sometimes different dates even though the historical information available at the origin is identical.

This block therefore creates a second, much smaller table at **Modeling Track × Origin × Horizon**. It carries only target-side information that can be known without observing the target mobility value: target date and Daypart, calendar position, policy-period metadata for the ordinary regime-aware model, and annual Fourier values.

The table is intentionally **not multiplied by Taxi Zone or metric**. The target calendar is the same for thousands of Forecast Sequences, so repeating it millions of times here would add storage without adding information. Sequence-specific weekly STL is attached separately in the next block because that fitted state depends on the particular Taxi Zone × Metric × Daypart series being forecast.

In [36]:
# 4.2.1.6D — Build compact forecast-target calendar context


def build_forecast_target_context():
    """
    Build future-known target context for every scheduled forecast.

    Input grain
    -----------
    rolling_origin_registry_df:
        Modeling Track × Origin × Horizon.

    Output grain
    ------------
    One row per Modeling Track × origin observation_sequence_id × Horizon.

    Target alignment
    ----------------
    Target Date, Daypart, temporal bucket, and policy period are re-derived
    from the canonical sequence calendar using target_observation_sequence_id.
    Explicit QA flags compare those values with the schedule supplied by 4.1.2.

    Leakage safety
    --------------
    Annual Fourier values depend only on the target calendar date. No target
    mobility value is read, joined, or derived in this table.
    """
    schedule = rolling_origin_registry_df.copy()

    schedule = schedule.rename(
        columns={
            "date": "scheduled_origin_date",
            "daypart": "scheduled_origin_daypart",
            "observation_sequence_id": "origin_observation_sequence_id",
            "target_date": "scheduled_target_date",
            "target_daypart": "scheduled_target_daypart",
        }
    )

    schedule["scheduled_origin_date"] = (
        pd.to_datetime(
            schedule["scheduled_origin_date"]
        ).dt.normalize()
    )
    schedule["scheduled_target_date"] = (
        pd.to_datetime(
            schedule["scheduled_target_date"]
        ).dt.normalize()
    )

    schedule_key = [
        "modeling_track",
        "origin_observation_sequence_id",
        "horizon",
    ]

    if schedule.duplicated(schedule_key).any():
        raise ValueError(
            "rolling_origin_registry contains duplicate "
            "Modeling Track × Origin × Horizon rows."
        )

    canonical = (
        sequence_calendar_df[
            [
                "observation_sequence_id",
                "date",
                "daypart",
                "daypart_order",
                "temporal_bucket",
                "temporal_bucket_order",
                "pre_post_cp",
            ]
        ]
        .drop_duplicates("observation_sequence_id")
        .copy()
    )

    origin_calendar = canonical.rename(
        columns={
            "observation_sequence_id":
                "origin_observation_sequence_id",
            "date": "origin_date",
            "daypart": "origin_daypart",
            "daypart_order": "origin_daypart_order",
            "temporal_bucket": "origin_temporal_bucket",
            "temporal_bucket_order":
                "origin_temporal_bucket_order",
            "pre_post_cp": "origin_pre_post_cp",
        }
    )

    target_calendar = canonical.rename(
        columns={
            "observation_sequence_id":
                "target_observation_sequence_id",
            "date": "target_date",
            "daypart": "target_daypart",
            "daypart_order": "target_daypart_order",
            "temporal_bucket": "target_temporal_bucket",
            "temporal_bucket_order":
                "target_temporal_bucket_order",
            "pre_post_cp": "target_pre_post_cp",
        }
    )

    context = (
        schedule
        .merge(
            origin_calendar,
            on="origin_observation_sequence_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            target_calendar,
            on="target_observation_sequence_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Canonical sequence arithmetic must agree exactly with the requested
    # forecast horizon.
    context["target_id_matches_horizon"] = (
        pd.to_numeric(
            context["target_observation_sequence_id"],
            errors="raise",
        )
        .eq(
            pd.to_numeric(
                context[
                    "origin_observation_sequence_id"
                ],
                errors="raise",
            )
            + pd.to_numeric(
                context["horizon"],
                errors="raise",
            )
        )
    )

    context[
        "scheduled_origin_date_matches_canonical"
    ] = (
        context["scheduled_origin_date"].eq(
            pd.to_datetime(
                context["origin_date"]
            ).dt.normalize()
        )
    )

    context[
        "scheduled_target_date_matches_canonical"
    ] = (
        context["scheduled_target_date"].eq(
            pd.to_datetime(
                context["target_date"]
            ).dt.normalize()
        )
    )

    # The upstream rolling schedule also records origin and target Daypart.
    # Confirm that those labels agree with the canonical sequence calendar rather
    # than allowing Pandas merge suffixes to hide a disagreement.
    context["scheduled_origin_daypart_matches_canonical"] = (
        context["scheduled_origin_daypart"].eq(context["origin_daypart"])
    )

    context["scheduled_target_daypart_matches_canonical"] = (
        context["scheduled_target_daypart"].eq(context["target_daypart"])
    )

    # These variables are known from the target calendar before its mobility
    # outcome has occurred.
    context["target_weekday_index"] = (
        pd.to_datetime(
            context["target_date"]
        ).dt.weekday.astype("int8")
    )
    context["target_is_weekend"] = (
        context["target_weekday_index"]
        .ge(5)
        .astype("int8")
    )
    context["target_month"] = (
        pd.to_datetime(
            context["target_date"]
        ).dt.month.astype("int8")
    )
    context["target_day_of_year"] = (
        pd.to_datetime(
            context["target_date"]
        ).dt.dayofyear.astype("int16")
    )

    fourier_lookup = (
        annual_fourier_calendar_df[
            [
                "observation_sequence_id",
                *FOURIER_FEATURE_COLUMNS,
            ]
        ]
        .rename(
            columns={
                "observation_sequence_id":
                    "target_observation_sequence_id"
            }
        )
    )

    context = context.merge(
        fourier_lookup,
        on="target_observation_sequence_id",
        how="left",
        validate="many_to_one",
    )

    return (
        context
        .sort_values(
            [
                "modeling_track",
                "origin_observation_sequence_id",
                "horizon",
            ]
        )
        .reset_index(drop=True)
    )


forecast_target_context_df = load_or_build_final(
    FORECAST_TARGET_CONTEXT_PATH,
    REBUILD_FINAL_TARGET_CONTEXT,
    build_forecast_target_context,
)


target_context_summary_records = []

for track, track_df in forecast_target_context_df.groupby(
    "modeling_track",
    observed=True,
):
    target_context_summary_records.append({
        "modeling_track": track,
        "scheduled_forecasts": len(track_df),
        "origins": (
            track_df[
                "origin_observation_sequence_id"
            ].nunique()
        ),
        "horizons": ", ".join(
            str(int(value))
            for value in sorted(
                track_df["horizon"]
                .dropna()
                .unique()
            )
        ),
        "first_target_date": pd.to_datetime(
            track_df["target_date"]
        ).min(),
        "last_target_date": pd.to_datetime(
            track_df["target_date"]
        ).max(),
        "duplicate_schedule_rows": int(
            track_df.duplicated(
                [
                    "modeling_track",
                    "origin_observation_sequence_id",
                    "horizon",
                ]
            ).sum()
        ),
        "target_id_mismatches": int(
            (
                ~track_df[
                    "target_id_matches_horizon"
                ]
            ).sum()
        ),
        "origin_date_mismatches": int(
            (
                ~track_df[
                    "scheduled_origin_date_matches_canonical"
                ]
            ).sum()
        ),
        "target_date_mismatches": int(
            (
                ~track_df[
                    "scheduled_target_date_matches_canonical"
                ]
            ).sum()
        ),
        "origin_daypart_mismatches": int(
            (
                ~track_df[
                    "scheduled_origin_daypart_matches_canonical"
                ]
            ).sum()
        ),
        "target_daypart_mismatches": int(
            (
                ~track_df[
                    "scheduled_target_daypart_matches_canonical"
                ]
            ).sum()
        ),
        "missing_fourier_rows": int(
            track_df[
                FOURIER_FEATURE_COLUMNS
            ].isna().any(axis=1).sum()
        ),
    })


forecast_target_context_summary_df = pd.DataFrame(
    target_context_summary_records
)

forecast_target_context_display_df = (
    forecast_target_context_summary_df.copy()
)
forecast_target_context_display_df[
    "modeling_track"
] = (
    forecast_target_context_display_df[
        "modeling_track"
    ].map(TRACK_LABELS)
)

display(forecast_target_context_display_df)

,modeling_track,scheduled_forecasts,origins,horizons,first_target_date,last_target_date,duplicate_schedule_rows,target_id_mismatches,origin_date_mismatches,target_date_mismatches,origin_daypart_mismatches,target_daypart_mismatches,missing_fourier_rows
0,Full-History Regime-Aware,17782,5929,"1, 2, 5",2023-01-01,2026-03-31,0,0,0,0,0,0,0
1,Post-CP Primary,6757,2254,"1, 2, 5",2025-01-05,2026-03-31,0,0,0,0,0,0,0
2,Pre-CP Reference,10957,3654,"1, 2, 5",2023-01-01,2024-12-31,0,0,0,0,0,0,0


**Findings.** The target-calendar table correctly describes all 35,496 forecasts scheduled across the three modeling tracks. For every forecast, the target sequence position, date, and time-of-day period agree with the requested 1-, 2-, or 5-step forecast horizon. There are no duplicate forecast rows, no date or Daypart mismatches, and no missing annual-calendar features. This gives each model a reliable description of **what date and time period it is forecasting** without including the mobility value it is supposed to predict.

**Resolve the latest legal weekly-STL feature for forecast targets.** The target-context table above knows *when* and *what Daypart* we are forecasting, but weekly STL is specific to an individual Taxi Zone × Metric × Daypart series.

This is where the validity intervals built in **5M** become useful. For a series that qualified for weekly STL, we find the interval containing the forecast origin. That interval identifies the most recently fitted seasonal template that was legally available at that moment. We then choose the Monday-through-Sunday offset corresponding to the **target date**.

For a series whose frozen decision was `none`, no fitted template is needed: `weekly_stl_enabled = False` and the seasonal adjustment is `0.0`.

The small audit below tests both enabled and non-enabled series in every modeling track. It verifies the lookup logic without expanding every possible Forecast Sequence × Origin × Horizon combination inside this notebook.

In [37]:
# 4.2.1.6E — Resolve sequence-specific weekly STL at forecast targets

WEEKLY_TARGET_SELECTION_KEYS = [
    "modeling_track",
    "forecast_sequence_id",
    "target_daypart",
]


def attach_weekly_stl_target_feature(target_rows):
    """
    Attach the latest legal weekly-STL target feature to sequence-specific rows.

    Required input grain
    --------------------
    Rows must identify:
        Modeling Track,
        Forecast Sequence,
        forecast origin observation_sequence_id,
        target Daypart,
        and target weekday.

    Resolution logic
    ----------------
    1. Read the frozen single_7 / none assignment for the target Daypart.
    2. For enabled rows, locate the 5M validity interval containing the origin.
    3. Use that interval's fitted Monday-Sunday template.
    4. Select the seasonal offset corresponding to the TARGET weekday.
    5. For `none` rows, return an intentional 0.0 seasonal adjustment.

    Leakage contract
    ----------------
    The forecast ORIGIN determines which fitted template is legal. The target
    date determines only which weekday value is selected from that already-legal
    template. Future target outcomes never determine fitted-state availability.
    """
    result = target_rows.copy().reset_index(drop=True)

    selection_lookup = (
        weekly_stl_selection_registry_df[
            [
                "modeling_track",
                "forecast_sequence_id",
                "daypart",
                "daypart_series_id",
                "weekly_stl_enabled",
            ]
        ]
        .rename(
            columns={
                "daypart": "target_daypart"
            }
        )
        .copy()
    )
    selection_lookup["_weekly_selection_present"] = True

    result = result.merge(
        selection_lookup,
        on=WEEKLY_TARGET_SELECTION_KEYS,
        how="left",
        validate="many_to_one",
    )

    if result[
        "_weekly_selection_present"
    ].isna().any():
        raise ValueError(
            "At least one target row did not match the frozen "
            "weekly-STL selection registry."
        )

    # Initialize explicit output fields. Disabled `none` series deliberately
    # keep no fitted refresh ID and receive a zero seasonal offset.
    result[
        "weekly_stl_refresh_observation_sequence_id"
    ] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )
    result["weekly_stl_refresh_role"] = pd.NA
    result[
        "weekly_stl_valid_from_observation_sequence_id"
    ] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )
    result[
        "weekly_stl_valid_through_observation_sequence_id"
    ] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )
    result["weekly_stl_seasonal_offset"] = np.nan

    enabled = (
        result["weekly_stl_enabled"]
        .fillna(False)
        .astype(bool)
    )

    result.loc[
        ~enabled,
        "weekly_stl_seasonal_offset",
    ] = 0.0

    # The 5M registry already contains the fitted weekday offsets plus the legal
    # origin range over which each state remains the latest available state.
    interval_groups = {
        key: group.sort_values(
            "weekly_stl_valid_from_observation_sequence_id"
        ).reset_index(drop=True)
        for key, group
        in weekly_stl_template_interval_registry_df.groupby(
            [
                "modeling_track",
                "forecast_sequence_id",
                "daypart",
            ],
            observed=True,
            sort=False,
        )
    }

    enabled_rows = result.loc[enabled]

    for key, row_index in enabled_rows.groupby(
        [
            "modeling_track",
            "forecast_sequence_id",
            "target_daypart",
        ],
        observed=True,
        sort=False,
    ).groups.items():
        row_index = pd.Index(
            sorted(row_index)
        )

        interval_df = interval_groups.get(key)

        if interval_df is None or interval_df.empty:
            raise ValueError(
                "An enabled weekly-STL target has no 5M validity intervals: "
                f"{key}."
            )

        origin_ids = pd.to_numeric(
            result.loc[
                row_index,
                "origin_observation_sequence_id",
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        valid_from = pd.to_numeric(
            interval_df[
                "weekly_stl_valid_from_observation_sequence_id"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        # Latest interval whose legal start is <= the forecast origin.
        interval_positions = (
            np.searchsorted(
                valid_from,
                origin_ids,
                side="right",
            )
            - 1
        )

        if (interval_positions < 0).any():
            raise ValueError(
                "An enabled weekly-STL target occurs before its first "
                f"legal fitted state: {key}."
            )

        chosen = (
            interval_df
            .iloc[interval_positions]
            .reset_index(drop=True)
        )

        valid_through = pd.to_numeric(
            chosen[
                "weekly_stl_valid_through_observation_sequence_id"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        if (origin_ids > valid_through).any():
            raise ValueError(
                "At least one weekly-STL target was mapped outside "
                f"its legal 5M validity interval: {key}."
            )

        target_weekday = pd.to_numeric(
            result.loc[
                row_index,
                "target_weekday_index",
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)

        offset_matrix = chosen[
            WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS
        ].to_numpy(dtype=float)

        selected_offsets = offset_matrix[
            np.arange(len(chosen)),
            target_weekday,
        ]

        result.loc[
            row_index,
            "weekly_stl_refresh_observation_sequence_id",
        ] = chosen[
            "refresh_observation_sequence_id"
        ].to_numpy()

        result.loc[
            row_index,
            "weekly_stl_refresh_role",
        ] = chosen[
            "refresh_role"
        ].to_numpy()

        result.loc[
            row_index,
            "weekly_stl_valid_from_observation_sequence_id",
        ] = chosen[
            "weekly_stl_valid_from_observation_sequence_id"
        ].to_numpy()

        result.loc[
            row_index,
            "weekly_stl_valid_through_observation_sequence_id",
        ] = chosen[
            "weekly_stl_valid_through_observation_sequence_id"
        ].to_numpy()

        result.loc[
            row_index,
            "weekly_stl_seasonal_offset",
        ] = selected_offsets

    # An enabled feature must resolve to a finite fitted seasonal adjustment.
    if not np.isfinite(
        pd.to_numeric(
            result.loc[
                enabled,
                "weekly_stl_seasonal_offset",
            ],
            errors="coerce",
        )
    ).all():
        raise ValueError(
            "At least one enabled weekly-STL target did not resolve "
            "to a finite seasonal offset."
        )

    return result.drop(
        columns="_weekly_selection_present"
    )


# ---------------------------------------------------------------------
# Reader-facing audit: one enabled and one `none` example per track
# ---------------------------------------------------------------------

weekly_target_audit_input_records = []

for track in TRACK_LABELS:
    track_selection = (
        weekly_stl_selection_registry_df.loc[
            weekly_stl_selection_registry_df[
                "modeling_track"
            ].eq(track)
        ]
    )

    for enabled_value in [True, False]:
        candidates = (
            track_selection.loc[
                track_selection[
                    "weekly_stl_enabled"
                ].eq(enabled_value)
            ]
            .sort_values(
                [
                    "metric",
                    "forecast_sequence_id",
                    "daypart",
                ]
            )
        )

        if candidates.empty:
            continue

        selection_row = candidates.iloc[0]

        # The fitted weekly-STL feature does not exist until the track reaches its
        # first rolling-validation information state. Initial-training rows occur
        # before that fitted state is legally available, so they are not valid audit
        # examples for this resolver.
        context_candidates = (
            forecast_target_context_df.loc[
                forecast_target_context_df[
                    "modeling_track"
                ].eq(track)
                & forecast_target_context_df[
                    "target_daypart"
                ].eq(selection_row["daypart"])
                & forecast_target_context_df[
                    "stage"
                ].isin(
                    [
                        "rolling_validation",
                        "final_holdout",
                    ]
                )
            ]
            .sort_values(
                [
                    "origin_observation_sequence_id",
                    "horizon",
                ]
            )
        )

        if context_candidates.empty:
            continue

        # Use a middle scheduled forecast rather than an edge state so the
        # reader-facing example resembles ordinary model operation.
        context_row = context_candidates.iloc[
            len(context_candidates) // 2
        ]

        record = context_row.to_dict()
        record.update({
            "forecast_sequence_id":
                selection_row["forecast_sequence_id"],
            "taxi_zone_id":
                selection_row["taxi_zone_id"],
            "zone":
                selection_row["zone"],
            "borough":
                selection_row["borough"],
            "metric":
                selection_row["metric"],
            "audit_expected_weekly_enabled":
                bool(enabled_value),
        })

        weekly_target_audit_input_records.append(
            record
        )


weekly_stl_target_audit_df = (
    attach_weekly_stl_target_feature(
        pd.DataFrame.from_records(
            weekly_target_audit_input_records
        )
    )
)

if not weekly_stl_target_audit_df[
    "weekly_stl_enabled"
].eq(
    weekly_stl_target_audit_df[
        "audit_expected_weekly_enabled"
    ]
).all():
    raise ValueError(
        "Weekly-STL audit did not preserve the frozen enabled/none assignment."
    )


weekly_stl_target_audit_display_df = (
    weekly_stl_target_audit_df[
        [
            "modeling_track",
            "zone",
            "metric",
            "target_daypart",
            "origin_observation_sequence_id",
            "horizon",
            "target_observation_sequence_id",
            "target_date",
            "target_weekday_index",
            "weekly_stl_enabled",
            "weekly_stl_refresh_observation_sequence_id",
            "weekly_stl_seasonal_offset",
        ]
    ]
    .copy()
)

weekly_stl_target_audit_display_df[
    "modeling_track"
] = (
    weekly_stl_target_audit_display_df[
        "modeling_track"
    ].map(TRACK_LABELS)
)

weekly_stl_target_audit_display_df[
    "metric"
] = (
    weekly_stl_target_audit_display_df[
        "metric"
    ].map(METRIC_LABELS)
)

weekly_stl_target_audit_display_df[
    "weekly_stl_seasonal_offset"
] = (
    weekly_stl_target_audit_display_df[
        "weekly_stl_seasonal_offset"
    ].round(3)
)

display(weekly_stl_target_audit_display_df)

,modeling_track,zone,metric,target_daypart,origin_observation_sequence_id,horizon,target_observation_sequence_id,target_date,target_weekday_index,weekly_stl_enabled,weekly_stl_refresh_observation_sequence_id,weekly_stl_seasonal_offset
0,Post-CP Primary,Baisley Park,FHVHV speed,am_peak,5025,2,5027,2025-10-02,3,True,4886,-1.997
1,Post-CP Primary,Baisley Park,FHVHV speed,evening,5028,2,5030,2025-10-02,3,False,<NA>,0.000
2,Full-History Regime-Aware,Baisley Park,FHVHV speed,am_peak,5025,2,5027,2025-10-02,3,True,4886,-2.036
3,Full-History Regime-Aware,Glen Oaks,FHVHV speed,evening,5028,2,5030,2025-10-02,3,False,<NA>,0.000
4,Pre-CP Reference,Baisley Park,FHVHV speed,am_peak,2737,5,2742,2024-07-02,1,True,2731,-2.864
5,Pre-CP Reference,Glen Oaks,FHVHV speed,evening,2740,5,2745,2024-07-02,1,False,<NA>,0.000


**Findings.** The six spot checks confirm that the weekly seasonal feature is being applied as intended. In each modeling track, one example comes from a series that uses a learned weekly pattern and one comes from a series that does not. For the enabled examples, the forecast receives the most recently available Monday–Sunday pattern that had already been fitted by the time the forecast was made, and then selects the adjustment for the target weekday. Series without a weekly pattern receive an adjustment of 0.0. All six examples follow those rules.

**Visualize how one forecast row comes together.** The tables above are technically precise, but it is worth stepping back and looking at one forecast in plain language.

A model row draws from **three different information directions**. On the left is mobility history that already happened. In the middle is the forecast origin—the point at which we stop observing and issue the prediction. On the right is information about the future target that is already knowable without seeing its mobility outcome.

Notice what is deliberately absent from the target box: **the target mobility value itself**. That is the quantity the model is trying to predict.

In [38]:
# 4.2.1.6F — Visualize the anatomy of one representative forecast row

if RUN_VISUALS:
    anatomy_audit_candidates = (
        weekly_stl_target_audit_df.loc[
            weekly_stl_target_audit_df["modeling_track"].eq("post_cp_primary")
            & weekly_stl_target_audit_df["weekly_stl_enabled"]
        ]
    )

    if anatomy_audit_candidates.empty:
        anatomy_audit_candidates = (
            weekly_stl_target_audit_df.loc[
                weekly_stl_target_audit_df["weekly_stl_enabled"]
            ]
        )

    if not anatomy_audit_candidates.empty:
        anatomy_target = anatomy_audit_candidates.iloc[0]

        anatomy_origin = pd.read_parquet(
            FINAL_ORIGIN_FEATURE_PATHS[
                anatomy_target["modeling_track"]
            ],
            filters=[
                (
                    "forecast_sequence_id",
                    "=",
                    anatomy_target["forecast_sequence_id"],
                ),
                (
                    "observation_sequence_id",
                    "=",
                    int(
                        anatomy_target[
                            "origin_observation_sequence_id"
                        ]
                    ),
                ),
            ],
        )

        if len(anatomy_origin) != 1:
            raise ValueError(
                "Representative anatomy lookup did not return exactly one origin row."
            )

        anatomy_origin = anatomy_origin.iloc[0]

        def fmt_anatomy_value(value):
            """Format one mobility value compactly for the visual."""
            if pd.isna(value):
                return "NA"

            numeric = float(value)

            if abs(numeric) >= 1_000_000:
                return f"{numeric / 1_000_000:,.2f}M"

            if abs(numeric) >= 1_000:
                return f"{numeric / 1_000:,.1f}K"

            return f"{numeric:,.2f}"

        metric_label = METRIC_LABELS.get(
            anatomy_target["metric"],
            anatomy_target["metric"],
        )

        # ------------------------------------------------------------------
        # Reader-facing card contents
        # ------------------------------------------------------------------

        history_text = (
            "<b>LEGAL MOBILITY HISTORY</b><br>"
            "<span style='font-size:11px'>What the model has already observed</span>"
            "<br><br>"
            f"<b>Current value</b><br>"
            f"{fmt_anatomy_value(anatomy_origin['origin_value'])}"
            "<br><br>"
            f"<b>Previous daypart</b><br>"
            f"{fmt_anatomy_value(anatomy_origin['seq_lag_1'])}"
            "<br><br>"
            f"<b>Same daypart yesterday</b><br>"
            f"{fmt_anatomy_value(anatomy_origin['same_daypart_lag_1d'])}"
            "<br><br>"
            f"<b>Prior 7-day average</b><br>"
            f"{fmt_anatomy_value(anatomy_origin['same_daypart_roll_mean_7d'])}"
        )

        origin_text = (
            "<b>FORECAST ORIGIN</b><br>"
            "<span style='font-size:11px'>The observation boundary</span>"
            "<br><br>"
            f"<b>Date</b><br>"
            f"{pd.Timestamp(anatomy_origin['origin_date']):%b %d, %Y}"
            "<br><br>"
            f"<b>Daypart</b><br>"
            f"{str(anatomy_origin['origin_daypart']).replace('_', ' ').title()}"
            "<br><br>"
            f"<b>Sequence position</b><br>"
            f"{int(anatomy_origin['observation_sequence_id']):,}"
        )

        target_text = (
            "<b>FUTURE-KNOWN TARGET CONTEXT</b><br>"
            "<span style='font-size:11px'>Known without seeing the future outcome</span>"
            "<br><br>"
            f"<b>Target</b><br>"
            f"{pd.Timestamp(anatomy_target['target_date']):%b %d, %Y}"
            "<br><br>"
            f"<b>Daypart / horizon</b><br>"
            f"{str(anatomy_target['target_daypart']).replace('_', ' ').title()} "
            f"· h={int(anatomy_target['horizon'])}"
            "<br><br>"
            f"<b>Annual Fourier</b><br>"
            f"{anatomy_target['annual_fourier_sin']:.3f}, "
            f"{anatomy_target['annual_fourier_cos']:.3f}"
            "<br><br>"
            f"<b>Weekly STL offset</b><br>"
            f"{anatomy_target['weekly_stl_seasonal_offset']:.3f}"
        )

        fig = go.Figure()

        # Three compact cards with ample horizontal spacing.
        cards = [
            {
                "x0": 0.02,
                "x1": 0.315,
                "fill": BRAND_COLORS["ice"],
                "text": history_text,
            },
            {
                "x0": 0.352,
                "x1": 0.648,
                "fill": BRAND_COLORS["pale_peach"],
                "text": origin_text,
            },
            {
                "x0": 0.685,
                "x1": 0.98,
                "fill": "white",
                "text": target_text,
            },
        ]

        for card in cards:
            fig.add_shape(
                type="rect",
                x0=card["x0"],
                x1=card["x1"],
                y0=0.20,
                y1=0.88,
                xref="paper",
                yref="paper",
                fillcolor=card["fill"],
                line={
                    "color": BRAND_COLORS["dark_teal"],
                    "width": 1.5,
                },
            )

            # Anchor the text near the upper-left corner of each card.
            fig.add_annotation(
                x=card["x0"] + 0.018,
                y=0.84,
                xref="paper",
                yref="paper",
                text=card["text"],
                showarrow=False,
                xanchor="left",
                yanchor="top",
                align="left",
                font={
                    "size": 12,
                    "color": BRAND_COLORS["dark_teal"],
                },
            )

        # Simple directional separators instead of arrows.
        fig.add_annotation(
            x=0.333,
            y=0.54,
            xref="paper",
            yref="paper",
            text="→",
            showarrow=False,
            font={
                "size": 28,
                "color": BRAND_COLORS["terracotta"],
            },
        )

        fig.add_annotation(
            x=0.667,
            y=0.54,
            xref="paper",
            yref="paper",
            text="→",
            showarrow=False,
            font={
                "size": 28,
                "color": BRAND_COLORS["terracotta"],
            },
        )

        # The key conceptual takeaway.
        fig.add_annotation(
            x=0.5,
            y=0.07,
            xref="paper",
            yref="paper",
            text=(
                "<b>What is missing?</b> The target mobility value itself. "
                "That is what the forecasting model must predict."
            ),
            showarrow=False,
            xanchor="center",
            align="center",
            font={
                "size": 13,
                "color": BRAND_COLORS["terracotta"],
            },
        )

        fig.update_layout(
            title=(
                "<b>How One Forecast Row Comes Together</b><br>"
                f"<sup>{metric_label} · {anatomy_target['zone']}</sup>"
            ),
            width=920,
            height=560,
            margin={
                "l": 25,
                "r": 25,
                "t": 80,
                "b": 25,
            },
            showlegend=False,
            paper_bgcolor="white",
            plot_bgcolor="white",
        )

        fig.update_xaxes(
            visible=False,
            range=[0, 1],
            fixedrange=True,
        )
        fig.update_yaxes(
            visible=False,
            range=[0, 1],
            fixedrange=True,
        )

        fig.show()

**Findings.** The example shows the three types of information that come together to make one forecast. On the left is mobility information that has already been observed, such as the current value, the previous Daypart, yesterday’s comparable value, and a recent average. The middle marks the point where observed information stops and forecasting begins. On the right is information we already know about the future target, such as its date, Daypart, annual calendar position, and weekly seasonal adjustment. The one thing intentionally missing is the target mobility value itself—that is what the model must predict.

**Prepare the frozen Pre-CP counterfactual handoff.** This is the final special case we need to make explicit before modeling.

The Pre-CP Reference can be trained and validated normally through its legal historical period. But after the final January 4, 2025 refit, there is no legitimate observed *no-congestion-pricing* history to read—because congestion pricing actually happened.

We therefore **do not create fake future lag or rolling columns in this notebook**.

Instead, we hand Chapter 5 three things it needs to run the counterfactual correctly. First is the real Pre-CP history through January 4, which seeds the prediction process. Second is a future calendar containing only information that can be known in advance, such as target Daypart and annual Fourier position. Third is the final frozen weekly-STL state for every Pre-CP Forecast Sequence × Daypart.

Beginning January 5, the `h=1` forecast advances the synthetic no-CP history one canonical position at a time. That prediction becomes part of the history used at the next origin. The `h=2` and `h=5` models can still issue farther-ahead forecasts from each recursively constructed origin, but they do not replace the one-step path used to move the counterfactual state forward.

In [39]:
# 4.2.1.6G — Prepare the frozen Pre-CP counterfactual handoff

PRE_CP_COUNTERFACTUAL_CALENDAR_PATH = (
    FINAL_DIR
    / "pre_cp_counterfactual_calendar.parquet"
)
PRE_CP_COUNTERFACTUAL_WEEKLY_STATE_PATH = (
    FINAL_DIR
    / "pre_cp_counterfactual_weekly_state.parquet"
)
PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH = (
    FINAL_DIR
    / "pre_cp_recursive_feature_contract.parquet"
)


def build_pre_cp_counterfactual_calendar():
    """
    Construct the future-known calendar for recursive no-CP application.

    Grain
    -----
    One row per counterfactual origin observation_sequence_id × Horizon.

    Starting state
    --------------
    The first origin is PRE_CP_SEASONAL_FREEZE_ID: the final legal observed
    Pre-CP position on January 4, 2025. Its h=1 target is therefore the first
    post-CP canonical position.

    Recursive contract
    ------------------
    h=1 is the state-update path. After its first forecast, each later
    counterfactual origin must obtain outcome-history features from earlier
    no-CP predictions rather than actual observed post-CP mobility.

    h=2 and h=5 remain direct forecast horizons from the same recursively
    constructed origin state.

    Leakage safety
    --------------
    This table contains no mobility outcome columns. Target-side predictors are
    deterministic calendar quantities known without observing future mobility.
    """
    horizons = sorted(
        int(value)
        for value in FORECAST_HORIZONS
    )

    if set(horizons) != {1, 2, 5}:
        raise ValueError(
            "Counterfactual application expects horizons {1, 2, 5}; "
            f"found {horizons}."
        )

    canonical = (
        sequence_calendar_df[
            [
                "observation_sequence_id",
                "date",
                "daypart",
                "daypart_order",
                "temporal_bucket",
                "temporal_bucket_order",
            ]
        ]
        .drop_duplicates(
            "observation_sequence_id"
        )
        .sort_values(
            "observation_sequence_id"
        )
        .copy()
    )

    canonical_last_id = int(
        canonical[
            "observation_sequence_id"
        ].max()
    )

    origins = (
        canonical.loc[
            canonical[
                "observation_sequence_id"
            ].ge(PRE_CP_SEASONAL_FREEZE_ID)
        ]
        .rename(
            columns={
                "observation_sequence_id":
                    "origin_observation_sequence_id",
                "date":
                    "origin_date",
                "daypart":
                    "origin_daypart",
                "daypart_order":
                    "origin_daypart_order",
                "temporal_bucket":
                    "origin_temporal_bucket",
                "temporal_bucket_order":
                    "origin_temporal_bucket_order",
            }
        )
    )

    horizon_df = pd.DataFrame({
        "horizon": horizons
    })

    # Cross-join only the compact canonical calendar. Forecast Sequences are
    # deliberately not multiplied into this deterministic target surface.
    counterfactual = (
        origins
        .assign(_join_key=1)
        .merge(
            horizon_df.assign(
                _join_key=1
            ),
            on="_join_key",
            how="inner",
        )
        .drop(columns="_join_key")
    )

    counterfactual[
        "target_observation_sequence_id"
    ] = (
        counterfactual[
            "origin_observation_sequence_id"
        ]
        + counterfactual["horizon"]
    )

    # Remove horizon combinations whose target falls beyond the canonical
    # study calendar.
    counterfactual = (
        counterfactual.loc[
            counterfactual[
                "target_observation_sequence_id"
            ].le(canonical_last_id)
        ]
        .copy()
    )

    target_calendar = canonical.rename(
        columns={
            "observation_sequence_id":
                "target_observation_sequence_id",
            "date":
                "target_date",
            "daypart":
                "target_daypart",
            "daypart_order":
                "target_daypart_order",
            "temporal_bucket":
                "target_temporal_bucket",
            "temporal_bucket_order":
                "target_temporal_bucket_order",
        }
    )

    counterfactual = counterfactual.merge(
        target_calendar,
        on="target_observation_sequence_id",
        how="left",
        validate="many_to_one",
    )

    counterfactual.insert(
        0,
        "modeling_track",
        "pre_cp_reference",
    )
    counterfactual.insert(
        1,
        "stage",
        "counterfactual_application",
    )

    # Only the very first origin is backed directly by observed Pre-CP history.
    # Every later origin exists inside the synthetic no-CP path.
    counterfactual[
        "counterfactual_origin_source"
    ] = np.where(
        counterfactual[
            "origin_observation_sequence_id"
        ].eq(PRE_CP_SEASONAL_FREEZE_ID),
        "observed_pre_cp_seed",
        "recursive_no_cp_prediction",
    )

    # The one-step forecast is what advances the synthetic state. h=2 and h=5
    # remain useful direct forecasts from each recursively constructed origin.
    counterfactual[
        "recursive_state_update"
    ] = counterfactual[
        "horizon"
    ].eq(1)

    counterfactual[
        "target_weekday_index"
    ] = (
        pd.to_datetime(
            counterfactual["target_date"]
        )
        .dt.weekday
        .astype("int8")
    )

    counterfactual[
        "target_is_weekend"
    ] = (
        counterfactual[
            "target_weekday_index"
        ]
        .ge(5)
        .astype("int8")
    )

    counterfactual[
        "target_month"
    ] = (
        pd.to_datetime(
            counterfactual["target_date"]
        )
        .dt.month
        .astype("int8")
    )

    counterfactual[
        "target_day_of_year"
    ] = (
        pd.to_datetime(
            counterfactual["target_date"]
        )
        .dt.dayofyear
        .astype("int16")
    )

    fourier_lookup = (
        annual_fourier_calendar_df[
            [
                "observation_sequence_id",
                *FOURIER_FEATURE_COLUMNS,
            ]
        ]
        .rename(
            columns={
                "observation_sequence_id":
                    "target_observation_sequence_id"
            }
        )
    )

    counterfactual = counterfactual.merge(
        fourier_lookup,
        on="target_observation_sequence_id",
        how="left",
        validate="many_to_one",
    )

    return (
        counterfactual
        .sort_values(
            [
                "origin_observation_sequence_id",
                "horizon",
            ]
        )
        .reset_index(drop=True)
    )


def build_pre_cp_counterfactual_weekly_state():
    """
    Freeze one weekly-seasonal state per Pre-CP Forecast Sequence × Daypart.

    Enabled series receive the exact final_pre_cp_freeze state from the 5M
    interval registry. Series whose frozen selection was `none` receive seven
    zero offsets.

    Output grain
    ------------
    Forecast Sequence × Daypart for the Pre-CP Reference track.
    """
    selection = (
        weekly_stl_selection_registry_df.loc[
            weekly_stl_selection_registry_df[
                "modeling_track"
            ].eq("pre_cp_reference")
        ]
        .copy()
    )

    freeze_intervals = (
        weekly_stl_template_interval_registry_df.loc[
            weekly_stl_template_interval_registry_df[
                "modeling_track"
            ].eq("pre_cp_reference")
            & weekly_stl_template_interval_registry_df[
                "refresh_role"
            ].eq("final_pre_cp_freeze")
            & weekly_stl_template_interval_registry_df[
                "fit_status"
            ].eq("fit_ok"),
            [
                "modeling_track",
                "forecast_sequence_id",
                "daypart",
                "refresh_observation_sequence_id",
                "weekly_stl_valid_from_observation_sequence_id",
                "weekly_stl_valid_through_observation_sequence_id",
                *WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
            ],
        ]
        .copy()
    )
    freeze_intervals[
        "_freeze_template_present"
    ] = True

    state = selection.merge(
        freeze_intervals,
        on=[
            "modeling_track",
            "forecast_sequence_id",
            "daypart",
        ],
        how="left",
        validate="one_to_one",
    )

    enabled = (
        state["weekly_stl_enabled"]
        .astype(bool)
    )

    if state.loc[
        enabled,
        "_freeze_template_present",
    ].isna().any():
        raise ValueError(
            "At least one enabled Pre-CP series lacks its "
            "final January 4 STL freeze state."
        )

    if not state.loc[
        enabled,
        "refresh_observation_sequence_id",
    ].eq(PRE_CP_SEASONAL_FREEZE_ID).all():
        raise ValueError(
            "At least one enabled Pre-CP series has the wrong "
            "frozen refresh sequence position."
        )

    # `none` is a real modeling decision, not missing data. Represent it as
    # zero weekly adjustment so Chapter 5 can use one consistent interface.
    state.loc[
        ~enabled,
        WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
    ] = 0.0

    state[
        "weekly_stl_frozen_refresh_observation_sequence_id"
    ] = pd.Series(
        np.where(
            enabled,
            PRE_CP_SEASONAL_FREEZE_ID,
            np.nan,
        )
    ).astype("Int64")

    state["weekly_stl_state_source"] = (
        np.where(
            enabled,
            "final_pre_cp_freeze",
            "none_assignment",
        )
    )

    return (
        state
        .drop(
            columns=[
                "_freeze_template_present",
                "refresh_observation_sequence_id",
            ],
            errors="ignore",
        )
        .sort_values(
            [
                "forecast_sequence_id",
                "daypart",
            ]
        )
        .reset_index(drop=True)
    )


pre_cp_counterfactual_calendar_df = (
    build_pre_cp_counterfactual_calendar()
)

pre_cp_counterfactual_weekly_state_df = (
    build_pre_cp_counterfactual_weekly_state()
)

# Explicitly tell Chapter 5 which candidate columns cannot be read from actual
# post-CP mobility and must instead be reconstructed from the evolving synthetic
# no-CP prediction history.
pre_cp_recursive_feature_contract_df = (
    forecast_feature_catalog_df.loc[
        forecast_feature_catalog_df[
            "pre_cp_post_freeze_class"
        ].eq("recursive_required")
    ]
    .copy()
)

pre_cp_recursive_feature_contract_df[
    "runtime_source"
] = (
    "model-generated no-CP prediction history"
)

pre_cp_recursive_feature_contract_df[
    "legal_seed_history_path"
] = str(
    track_path(
        "track_history_source",
        "pre_cp_reference",
    )
)


# ---------------------------------------------------------------------
# Verify where the observed Pre-CP seed history actually stops
# ---------------------------------------------------------------------

pre_cp_seed_history_audit_df = (
    pd.read_parquet(
        track_path(
            "track_history_source",
            "pre_cp_reference",
        ),
        columns=[
            "observation_sequence_id",
            "date",
        ],
    )
)

pre_cp_seed_max_id = int(
    pre_cp_seed_history_audit_df[
        "observation_sequence_id"
    ].max()
)

pre_cp_seed_max_date = pd.to_datetime(
    pre_cp_seed_history_audit_df["date"]
).max()


pre_cp_counterfactual_handoff_summary_df = (
    pd.DataFrame([
        {
            "handoff_component":
                "Legal observed-history seed",
            "rows_or_states":
                len(pre_cp_seed_history_audit_df),
            "detail": (
                f"last sequence position "
                f"{pre_cp_seed_max_id:,}; "
                f"last date "
                f"{pre_cp_seed_max_date:%Y-%m-%d}"
            ),
        },
        {
            "handoff_component":
                "Recursive future calendar",
            "rows_or_states":
                len(pre_cp_counterfactual_calendar_df),
            "detail": (
                f"{pre_cp_counterfactual_calendar_df['origin_observation_sequence_id'].nunique():,} "
                "counterfactual origins across h=1, h=2, h=5"
            ),
        },
        {
            "handoff_component":
                "h=1 state-update path",
            "rows_or_states":
                int(
                    pre_cp_counterfactual_calendar_df[
                        "recursive_state_update"
                    ].sum()
                ),
            "detail": (
                "one-step forecasts that advance the "
                "synthetic no-CP history"
            ),
        },
        {
            "handoff_component":
                "Frozen weekly series × Daypart states",
            "rows_or_states":
                len(
                    pre_cp_counterfactual_weekly_state_df
                ),
            "detail": (
                f"{int(pre_cp_counterfactual_weekly_state_df['weekly_stl_enabled'].sum()):,} "
                "weekly-STL enabled"
            ),
        },
        {
            "handoff_component":
                "Recursive-required feature columns",
            "rows_or_states":
                len(
                    pre_cp_recursive_feature_contract_df
                ),
            "detail": (
                "rebuilt from earlier no-CP predictions "
                "after the freeze"
            ),
        },
    ])
)

display(
    pre_cp_counterfactual_handoff_summary_df
)

,handoff_component,rows_or_states,detail
0,Legal observed-history seed,3818325,"last sequence position 3,675; last date 2025-0..."
1,Recursive future calendar,6760,"2,255 counterfactual origins across h=1, h=2, h=5"
2,h=1 state-update path,2255,one-step forecasts that advance the synthetic ...
3,Frozen weekly series × Daypart states,5195,"3,633 weekly-STL enabled"
4,Recursive-required feature columns,17,rebuilt from earlier no-CP predictions after t...


**Findings.** The Pre-CP setup now has a clean handoff from real observations to the hypothetical no-congestion-pricing forecast. Real mobility history ends on January 4, 2025 at sequence position 3,675. The first hypothetical post-CP target is the very next position, 3,676. From that point forward, the model must use its own earlier no-congestion-pricing predictions to update the 17 history-based predictors needed for later forecasts rather than using the mobility that actually occurred. The setup also carries forward the last weekly pattern estimated before congestion pricing began, so the future simulation can continue without learning from post-CP outcomes.

**Visualize how the no-CP counterfactual moves forward.** The recursive rule is the easiest part of this architecture to misunderstand, so this final workflow diagram makes it explicit.

The left side is real history, and it stops permanently at January 4, 2025. Starting January 5, the model moves forward one canonical position at a time. Each `h=1` prediction becomes part of the synthetic no-CP history used at the next origin. Calendar and frozen seasonal features can still be looked up normally because they do not require seeing future mobility.

The actual post-CP observations still exist, but they sit **outside the prediction loop**. They are retained for later comparison with the counterfactual, not fed back into it.

In [40]:
# 4.2.1.6H — Visualize the frozen Pre-CP recursive counterfactual contract

if RUN_VISUALS:
    fig = go.Figure()

    freeze_date_label = (
        pd.Timestamp(
            PRE_CP_SEASONAL_FREEZE_DATE
        ).strftime("%b %d, %Y")
    )

    freeze_id_label = (
        f"{int(PRE_CP_SEASONAL_FREEZE_ID):,}"
    )

    # ------------------------------------------------------------------
    # Three reader-facing steps
    # ------------------------------------------------------------------

    cards = [
        {
            "x0": 0.03,
            "x1": 0.30,
            "fill": BRAND_COLORS["ice"],
            "text": (
                "<b>1 · REAL HISTORY STOPS</b>"
                "<br><br>"
                "Observed mobility is available"
                "<br>"
                f"through <b>{freeze_date_label}</b>."
                "<br><br>"
                f"Last observed sequence position:"
                "<br>"
                f"<b>{freeze_id_label}</b>"
            ),
        },
        {
            "x0": 0.365,
            "x1": 0.635,
            "fill": BRAND_COLORS["pale_peach"],
            "text": (
                "<b>2 · MOVE FORWARD ONE STEP</b>"
                "<br><br>"
                "Predict the next Daypart."
                "<br><br>"
                "That prediction becomes part"
                "<br>"
                "of the no-CP history used"
                "<br>"
                "for the next forecast."
                "<br><br>"
                "<b>Repeat one step at a time.</b>"
            ),
        },
        {
            "x0": 0.70,
            "x1": 0.97,
            "fill": "white",
            "text": (
                "<b>3 · FORECAST FARTHER AHEAD</b>"
                "<br><br>"
                "At each newly constructed"
                "<br>"
                "forecast origin, the model can"
                "<br>"
                "also produce h=2 and h=5"
                "<br>"
                "forecasts."
                "<br><br>"
                "Calendar and the frozen weekly"
                "<br>"
                "pattern remain available."
            ),
        },
    ]

    for card in cards:
        fig.add_shape(
            type="rect",
            x0=card["x0"],
            x1=card["x1"],
            y0=0.34,
            y1=0.86,
            xref="paper",
            yref="paper",
            fillcolor=card["fill"],
            line={
                "color": BRAND_COLORS["dark_teal"],
                "width": 1.5,
            },
        )

        # Center both the annotation anchor and the text itself.
        fig.add_annotation(
            x=(card["x0"] + card["x1"]) / 2,
            y=0.60,
            xref="paper",
            yref="paper",
            text=card["text"],
            showarrow=False,
            xanchor="center",
            yanchor="middle",
            align="center",
            font={
                "size": 12,
                "color": BRAND_COLORS["dark_teal"],
            },
        )

    # ------------------------------------------------------------------
    # Simple directional flow
    # ------------------------------------------------------------------

    for x in [0.3325, 0.6675]:
        fig.add_annotation(
            x=x,
            y=0.60,
            xref="paper",
            yref="paper",
            text="→",
            showarrow=False,
            xanchor="center",
            yanchor="middle",
            font={
                "size": 30,
                "color": BRAND_COLORS["terracotta"],
            },
        )

    # ------------------------------------------------------------------
    # Critical rule: actual post-CP outcomes remain outside the loop
    # ------------------------------------------------------------------

    fig.add_shape(
        type="rect",
        x0=0.18,
        x1=0.82,
        y0=0.08,
        y1=0.23,
        xref="paper",
        yref="paper",
        fillcolor="white",
        line={
            "color": BRAND_COLORS["terracotta"],
            "width": 1.5,
            "dash": "dash",
        },
    )

    fig.add_annotation(
        x=0.50,
        y=0.155,
        xref="paper",
        yref="paper",
        text=(
            "<b>Actual post-CP mobility stays outside this loop.</b>"
            "<br>"
            "It is saved for later comparison with the hypothetical "
            "no-congestion-pricing forecast."
        ),
        showarrow=False,
        xanchor="center",
        yanchor="middle",
        align="center",
        font={
            "size": 12,
            "color": BRAND_COLORS["terracotta"],
        },
    )

    fig.update_layout(
        title={
            "text": (
                "<b>How the Pre-CP No-Congestion-Pricing Forecast "
                "Moves Forward</b>"
            ),
            "x": 0.03,
            "xanchor": "left",
        },
        width=920,
        height=500,
        margin={
            "l": 25,
            "r": 25,
            "t": 75,
            "b": 20,
        },
        showlegend=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font={
            "color": BRAND_COLORS["dark_teal"],
        },
    )

    fig.update_xaxes(
        visible=False,
        range=[0, 1],
        fixedrange=True,
    )

    fig.update_yaxes(
        visible=False,
        range=[0, 1],
        fixedrange=True,
    )

    fig.show()

This is a **workflow diagram rather than an empirical result**. The key rule for the next modeling stage is simple: after January 4, 2025, the hypothetical no-congestion-pricing forecast can no longer use actual mobility observations. Instead, each new prediction becomes part of the history used to make the next prediction. Actual post-CP mobility is kept separate so it can later be compared with the hypothetical no-congestion-pricing path.

This is a **workflow diagram rather than an empirical result**. The key rule for Chapter 5 is simple: after the January 4 freeze, model-generated no-CP history replaces observed mobility as the source for outcome-dependent lag and rolling features. Actual post-CP mobility remains available only for later comparison against the counterfactual.

**Run the final end-to-end feature-contract QA.** Earlier sections tested each feature family individually. This block asks a different question: *when all of those pieces are handed to forecasting together, do their contracts still agree?*

The checks cover origin-grain preservation, native forecasting populations, canonical horizon arithmetic, calendar/Fourier completeness, weekly-STL validity intervals, the sequence-specific STL lookup, the frozen Pre-CP boundary, and the recursive counterfactual handoff.

Missing historical feature values are **not automatically failures**; a small amount of source-series missingness was deliberately preserved throughout the notebook. A failure here means something structural is wrong—for example, a duplicated key, a target misaligned with its horizon, a seasonal state used outside its legal interval, or actual post-CP outcome history entering the counterfactual surface.

In [41]:
# 4.2.1.6I — Run final end-to-end feature-contract QA


def feature_contract_qa_row(check, passed, detail):
    """Create one standardized final QA record."""
    return {
        "check": check,
        "status": (
            "PASS"
            if bool(passed)
            else "FAIL"
        ),
        "detail": detail,
    }


final_qa_rows = []


# ---------------------------------------------------------------------
# 1. Final origin-history surfaces
# ---------------------------------------------------------------------

final_qa_rows.append(
    feature_contract_qa_row(
        "Origin assembly preserves every base row",
        final_origin_feature_summary_df[
            "origin_rows"
        ].eq(
            final_origin_feature_summary_df[
                "base_rows"
            ]
        ).all(),
        "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.origin_rows:,}/{row.base_rows:,}"
            for row
            in final_origin_feature_summary_df.itertuples(
                index=False
            )
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Final origin keys are unique",
        final_origin_feature_summary_df[
            "duplicate_origin_keys"
        ].eq(0).all(),
        (
            "duplicate rows by track = "
            + "; ".join(
                f"{TRACK_LABELS[row.modeling_track]} "
                f"{row.duplicate_origin_keys:,}"
                for row
                in final_origin_feature_summary_df.itertuples(
                    index=False
                )
            )
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Native forecasting populations are preserved",
        (
            final_origin_feature_summary_df[
                "origin_sequences"
            ].eq(
                final_origin_feature_summary_df[
                    "native_sequences_expected"
                ]
            )
            & final_origin_feature_summary_df[
                "native_sequence_set_matches"
            ]
        ).all(),
        "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.origin_sequences:,}/"
            f"{row.native_sequences_expected:,} sequences"
            for row
            in final_origin_feature_summary_df.itertuples(
                index=False
            )
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Comparison cohorts remain provenance flags only",
        (
            final_origin_feature_summary_df[
                "post_common_sequences"
            ].eq(
                final_origin_feature_summary_df[
                    "post_common_sequences_expected"
                ]
            )
            & final_origin_feature_summary_df[
                "cross_regime_sequences"
            ].eq(
                final_origin_feature_summary_df[
                    "cross_regime_sequences_expected"
                ]
            )
        ).all(),
        (
            "flagged sequence counts match the authoritative "
            "upstream comparison registries"
        ),
    )
)


# ---------------------------------------------------------------------
# 2. Compact forecast-target context
# ---------------------------------------------------------------------

target_key = [
    "modeling_track",
    "origin_observation_sequence_id",
    "horizon",
]

final_qa_rows.append(
    feature_contract_qa_row(
        "Forecast-target context key is unique",
        not forecast_target_context_df.duplicated(
            target_key
        ).any(),
        (
            "duplicate Modeling Track × Origin × Horizon rows = "
            f"{int(forecast_target_context_df.duplicated(target_key).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Every target position equals origin + horizon",
        forecast_target_context_df[
            "target_id_matches_horizon"
        ].all(),
        (
            "mismatched rows = "
            f"{int((~forecast_target_context_df['target_id_matches_horizon']).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Scheduled origin dates match the canonical calendar",
        forecast_target_context_df[
            "scheduled_origin_date_matches_canonical"
        ].all(),
        (
            "mismatched rows = "
            f"{int((~forecast_target_context_df['scheduled_origin_date_matches_canonical']).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Scheduled target dates match the canonical calendar",
        forecast_target_context_df[
            "scheduled_target_date_matches_canonical"
        ].all(),
        (
            "mismatched rows = "
            f"{int((~forecast_target_context_df['scheduled_target_date_matches_canonical']).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Scheduled origin Dayparts match the canonical calendar",
        forecast_target_context_df[
            "scheduled_origin_daypart_matches_canonical"
        ].all(),
        (
            "mismatched rows = "
            f"{int((~forecast_target_context_df['scheduled_origin_daypart_matches_canonical']).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Scheduled target Dayparts match the canonical calendar",
        forecast_target_context_df[
            "scheduled_target_daypart_matches_canonical"
        ].all(),
        (
            "mismatched rows = "
            f"{int((~forecast_target_context_df['scheduled_target_daypart_matches_canonical']).sum()):,}"
        ),
    )
)

fourier_values = (
    forecast_target_context_df[
        FOURIER_FEATURE_COLUMNS
    ].to_numpy(dtype=float)
)

fourier_complete = (
    np.isfinite(fourier_values).all()
    and (
        np.abs(fourier_values)
        <= 1.000001
    ).all()
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Target Fourier features are complete and bounded",
        fourier_complete,
        (
            f"rows = "
            f"{len(forecast_target_context_df):,}; "
            "expected bounds [-1, 1]"
        ),
    )
)

forbidden_target_context_columns = {
    "observed_value",
    "origin_value",
    "target_value",
    "actual_value",
}

present_target_context_outcomes = sorted(
    forbidden_target_context_columns.intersection(
        forecast_target_context_df.columns
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Compact target context contains no mobility outcomes",
        len(
            present_target_context_outcomes
        ) == 0,
        (
            "outcome columns present = "
            f"{present_target_context_outcomes or 'none'}"
        ),
    )
)


# ---------------------------------------------------------------------
# 3. Weekly-STL validity and target resolution
# ---------------------------------------------------------------------

interval_valid_order = (
    weekly_stl_template_interval_registry_df[
        "weekly_stl_valid_from_observation_sequence_id"
    ]
    <= weekly_stl_template_interval_registry_df[
        "weekly_stl_valid_through_observation_sequence_id"
    ]
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Every weekly-STL validity interval is ordered correctly",
        interval_valid_order.all(),
        (
            "invalid interval rows = "
            f"{int((~interval_valid_order).sum()):,}"
        ),
    )
)

weekly_selection_key = [
    "modeling_track",
    "forecast_sequence_id",
    "daypart",
]

final_qa_rows.append(
    feature_contract_qa_row(
        "Weekly-STL selection key is unique",
        not weekly_stl_selection_registry_df.duplicated(
            weekly_selection_key
        ).any(),
        (
            "duplicate selection rows = "
            f"{int(weekly_stl_selection_registry_df.duplicated(weekly_selection_key).sum()):,}"
        ),
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "All materialized weekly-STL production fits succeeded",
        weekly_stl_template_registry_df[
            "fit_status"
        ].eq("fit_ok").all(),
        (
            "failed template states = "
            f"{int((~weekly_stl_template_registry_df['fit_status'].eq('fit_ok')).sum()):,}"
        ),
    )
)

enabled_audit = (
    weekly_stl_target_audit_df[
        "weekly_stl_enabled"
    ].astype(bool)
)

audit_offsets = pd.to_numeric(
    weekly_stl_target_audit_df[
        "weekly_stl_seasonal_offset"
    ],
    errors="coerce",
)

enabled_audit_origins = pd.to_numeric(
    weekly_stl_target_audit_df.loc[
        enabled_audit,
        "origin_observation_sequence_id",
    ],
    errors="raise",
)

enabled_audit_from = pd.to_numeric(
    weekly_stl_target_audit_df.loc[
        enabled_audit,
        "weekly_stl_valid_from_observation_sequence_id",
    ],
    errors="raise",
)

enabled_audit_through = pd.to_numeric(
    weekly_stl_target_audit_df.loc[
        enabled_audit,
        "weekly_stl_valid_through_observation_sequence_id",
    ],
    errors="raise",
)

weekly_audit_interval_legal = (
    enabled_audit_origins.ge(
        enabled_audit_from
    )
    & enabled_audit_origins.le(
        enabled_audit_through
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Weekly-STL target resolver uses legal 5M intervals",
        (
            weekly_audit_interval_legal.all()
            and np.isfinite(
                audit_offsets.loc[
                    enabled_audit
                ]
            ).all()
            and audit_offsets.loc[
                ~enabled_audit
            ].eq(0.0).all()
        ),
        (
            f"enabled audit rows = "
            f"{int(enabled_audit.sum()):,}; "
            f"none audit rows = "
            f"{int((~enabled_audit).sum()):,}"
        ),
    )
)


# ---------------------------------------------------------------------
# 4. Frozen Pre-CP counterfactual
# ---------------------------------------------------------------------

final_qa_rows.append(
    feature_contract_qa_row(
        "Observed Pre-CP seed ends at the approved legal freeze",
        (
            pre_cp_seed_max_id
            <= PRE_CP_SEASONAL_FREEZE_ID
            and pd.Timestamp(
                pre_cp_seed_max_date
            )
            <= pd.Timestamp(
                PRE_CP_SEASONAL_FREEZE_DATE
            )
        ),
        (
            f"observed-history max = sequence "
            f"{pre_cp_seed_max_id:,}, "
            f"{pd.Timestamp(pre_cp_seed_max_date):%Y-%m-%d}"
        ),
    )
)

first_counterfactual_target_id = int(
    pre_cp_counterfactual_calendar_df[
        "target_observation_sequence_id"
    ].min()
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Counterfactual begins immediately after the legal Pre-CP seed",
        (
            first_counterfactual_target_id
            == PRE_CP_SEASONAL_FREEZE_ID + 1
        ),
        (
            "first counterfactual target = "
            f"{first_counterfactual_target_id:,}"
        ),
    )
)

forbidden_counterfactual_columns = set(
    OUTCOME_HISTORY_FEATURE_COLUMNS
    + [
        "observed_value",
        "target_value",
        "actual_value",
    ]
)

present_forbidden_columns = sorted(
    forbidden_counterfactual_columns.intersection(
        pre_cp_counterfactual_calendar_df.columns
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Counterfactual calendar contains no observed outcome-history features",
        len(present_forbidden_columns) == 0,
        (
            "forbidden columns present = "
            f"{present_forbidden_columns or 'none'}"
        ),
    )
)

future_counterfactual_origins = (
    pre_cp_counterfactual_calendar_df.loc[
        pre_cp_counterfactual_calendar_df[
            "origin_observation_sequence_id"
        ].gt(PRE_CP_SEASONAL_FREEZE_ID)
    ]
)

final_qa_rows.append(
    feature_contract_qa_row(
        "All post-freeze counterfactual origins require recursive prediction history",
        future_counterfactual_origins[
            "counterfactual_origin_source"
        ].eq(
            "recursive_no_cp_prediction"
        ).all(),
        (
            f"post-freeze origin-horizon rows = "
            f"{len(future_counterfactual_origins):,}"
        ),
    )
)

pre_weekly_enabled = (
    pre_cp_counterfactual_weekly_state_df[
        "weekly_stl_enabled"
    ].astype(bool)
)

pre_weekly_offsets = (
    pre_cp_counterfactual_weekly_state_df.loc[
        pre_weekly_enabled,
        WEEKLY_STL_TEMPLATE_OFFSET_COLUMNS,
    ].to_numpy(dtype=float)
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Every enabled Pre-CP weekly state has finite frozen weekday offsets",
        np.isfinite(
            pre_weekly_offsets
        ).all(),
        (
            "enabled Forecast Sequence × Daypart states = "
            f"{int(pre_weekly_enabled.sum()):,}"
        ),
    )
)

pre_frozen_refresh_ids = (
    pd.to_numeric(
        pre_cp_counterfactual_weekly_state_df.loc[
            pre_weekly_enabled,
            "weekly_stl_frozen_refresh_observation_sequence_id",
        ],
        errors="raise",
    )
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Every enabled Pre-CP weekly state remains frozen at January 4",
        pre_frozen_refresh_ids.eq(
            PRE_CP_SEASONAL_FREEZE_ID
        ).all(),
        (
            "unique enabled freeze IDs = "
            f"{sorted(pre_frozen_refresh_ids.astype(int).unique().tolist())}"
        ),
    )
)


# ---------------------------------------------------------------------
# 5. Explicit recursive-feature policy
# ---------------------------------------------------------------------

recursive_feature_names = set(
    pre_cp_recursive_feature_contract_df[
        "feature_name"
    ]
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Every outcome-history feature is marked recursive after the Pre-CP freeze",
        set(
            OUTCOME_HISTORY_FEATURE_COLUMNS
        ).issubset(
            recursive_feature_names
        ),
        (
            "required recursive columns = "
            f"{len(OUTCOME_HISTORY_FEATURE_COLUMNS):,}"
        ),
    )
)

regime_contract = (
    forecast_feature_catalog_df.loc[
        forecast_feature_catalog_df[
            "feature_name"
        ].isin(
            [
                "origin_pre_post_cp",
                "target_pre_post_cp",
            ]
        )
    ]
)

final_qa_rows.append(
    feature_contract_qa_row(
        "Actual policy-regime indicators are disabled for the frozen Pre-CP counterfactual",
        regime_contract[
            "pre_cp_post_freeze_class"
        ].eq("not_used").all(),
        (
            "origin_pre_post_cp and target_pre_post_cp "
            "must not enter no-CP application"
        ),
    )
)


# ---------------------------------------------------------------------
# Consolidated reader-facing QA
# ---------------------------------------------------------------------

final_feature_contract_qa_df = (
    pd.DataFrame(final_qa_rows)
)

display(final_feature_contract_qa_df)

failed_final_checks = (
    final_feature_contract_qa_df.loc[
        final_feature_contract_qa_df[
            "status"
        ].ne("PASS")
    ]
)

if not failed_final_checks.empty:
    raise AssertionError(
        "Final forecasting feature contract QA failed: "
        + "; ".join(
            failed_final_checks["check"]
        )
    )

,check,status,detail
0,Origin assembly preserves every base row,PASS,"Post-CP Primary 2,497,432/2,497,432; Full-Hist..."
1,Final origin keys are unique,PASS,duplicate rows by track = Post-CP Primary 0; F...
2,Native forecasting populations are preserved,PASS,"Post-CP Primary 1,108/1,108 sequences; Full-Hi..."
3,Comparison cohorts remain provenance flags only,PASS,flagged sequence counts match the authoritativ...
4,Forecast-target context key is unique,PASS,duplicate Modeling Track × Origin × Horizon ro...
5,Every target position equals origin + horizon,PASS,mismatched rows = 0
6,Scheduled origin dates match the canonical cal...,PASS,mismatched rows = 0
7,Scheduled target dates match the canonical cal...,PASS,mismatched rows = 0
8,Scheduled origin Dayparts match the canonical ...,PASS,mismatched rows = 0
9,Scheduled target Dayparts match the canonical ...,PASS,mismatched rows = 0


**Findings.** All 24 final quality checks pass. The completed feature tables contain the expected rows and forecasting series without duplicate keys; every forecast target lines up with the correct date, Daypart, and horizon; annual calendar features are complete; and weekly seasonal adjustments are used only after the corresponding weekly pattern has been fitted. The Pre-CP checks also confirm that observed history stops on January 4, 2025, the hypothetical no-congestion-pricing forecast begins immediately afterward, and actual post-CP mobility cannot enter that forecasting loop. Together, these checks confirm that the feature package is internally consistent and does not accidentally use future information.

**Export the final forecasting handoff.** If the QA block above passes, the forecasting-feature layer is ready to leave Notebook 4.2.1.

The exported package remains intentionally normalized. The large origin tables contain sequence-specific historical information. The compact target-context table contains repeated calendar and horizon information only once. Weekly-STL selection, fitted templates, and validity intervals remain explicit fitted-state registries rather than being copied into every possible forecast row.

The Pre-CP counterfactual receives its own compact handoff: a future calendar with **no observed outcomes**, a frozen weekly-seasonal state, and an explicit list of feature columns that Chapter 5 must regenerate from the model's evolving no-CP prediction history.

The manifest below is the table of contents for that handoff.

In [42]:
# 4.2.1.6J — Export the final forecasting feature package and manifest

WEEKLY_STL_SELECTION_FINAL_PATH = (
    FINAL_DIR
    / "weekly_stl_selection_registry.parquet"
)
WEEKLY_STL_TEMPLATE_FINAL_PATH = (
    FINAL_DIR
    / "weekly_stl_template_registry.parquet"
)
WEEKLY_STL_INTERVAL_FINAL_PATH = (
    FINAL_DIR
    / "weekly_stl_template_interval_registry.parquet"
)
ANNUAL_FOURIER_FINAL_PATH = (
    FINAL_DIR
    / "annual_fourier_calendar.parquet"
)


if WRITE_FINAL_OUTPUTS:
    # Feature-definition and compact target-context contracts.
    forecast_feature_catalog_df.to_parquet(
        FORECAST_FEATURE_CATALOG_PATH,
        index=False,
    )
    forecast_target_context_df.to_parquet(
        FORECAST_TARGET_CONTEXT_PATH,
        index=False,
    )

    # Seasonal-state assets remain normalized instead of being repeated across
    # millions of model rows.
    weekly_stl_selection_registry_df.to_parquet(
        WEEKLY_STL_SELECTION_FINAL_PATH,
        index=False,
    )
    weekly_stl_template_registry_df.to_parquet(
        WEEKLY_STL_TEMPLATE_FINAL_PATH,
        index=False,
    )
    weekly_stl_template_interval_registry_df.to_parquet(
        WEEKLY_STL_INTERVAL_FINAL_PATH,
        index=False,
    )
    annual_fourier_calendar_df.to_parquet(
        ANNUAL_FOURIER_FINAL_PATH,
        index=False,
    )

    # Frozen no-CP application contract.
    pre_cp_counterfactual_calendar_df.to_parquet(
        PRE_CP_COUNTERFACTUAL_CALENDAR_PATH,
        index=False,
    )
    pre_cp_counterfactual_weekly_state_df.to_parquet(
        PRE_CP_COUNTERFACTUAL_WEEKLY_STATE_PATH,
        index=False,
    )
    pre_cp_recursive_feature_contract_df.to_parquet(
        PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH,
        index=False,
    )

    # Consolidated end-to-end QA.
    final_feature_contract_qa_df.to_parquet(
        FINAL_FEATURE_QA_PATH,
        index=False,
    )


# ---------------------------------------------------------------------
# Build the reader-facing manifest without reloading the large surfaces
# ---------------------------------------------------------------------

manifest_records = []

for row in final_origin_feature_summary_df.itertuples(
    index=False
):
    path = FINAL_ORIGIN_FEATURE_PATHS[
        row.modeling_track
    ]

    manifest_records.append({
        "artifact": (
            "Origin feature surface · "
            f"{TRACK_LABELS[row.modeling_track]}"
        ),
        "track_scope":
            TRACK_LABELS[row.modeling_track],
        "grain": (
            "Forecast Sequence × Origin "
            "observation_sequence_id"
        ),
        "rows":
            int(row.origin_rows),
        "contains_observed_mobility_history":
            True,
        "counterfactual_role": (
            "Legal seed/history only"
            if row.modeling_track
            == "pre_cp_reference"
            else "Ordinary forecasting history"
        ),
        "purpose": (
            "Current level, recent lags, comparable-Daypart history, "
            "calendar context, and comparison-cohort provenance."
        ),
        "path":
            str(path),
    })


manifest_records.extend([
    {
        "artifact":
            "Forecast target calendar context",
        "track_scope":
            "All modeling tracks",
        "grain":
            "Modeling Track × Origin × Horizon",
        "rows":
            len(forecast_target_context_df),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Ordinary scheduled forecasts",
        "purpose": (
            "Canonical target timing, horizon, and "
            "annual Fourier phase without target outcomes."
        ),
        "path":
            str(FORECAST_TARGET_CONTEXT_PATH),
    },
    {
        "artifact":
            "Weekly-STL selection registry",
        "track_scope":
            "All modeling tracks",
        "grain":
            "Modeling Track × Forecast Sequence × Daypart",
        "rows":
            len(weekly_stl_selection_registry_df),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Frozen selection policy",
        "purpose":
            "Records the fixed single_7 versus none decision.",
        "path":
            str(WEEKLY_STL_SELECTION_FINAL_PATH),
    },
    {
        "artifact":
            "Weekly-STL fitted template registry",
        "track_scope":
            "All modeling tracks",
        "grain": (
            "Modeling Track × Forecast Sequence × "
            "Daypart × Refresh"
        ),
        "rows":
            len(weekly_stl_template_registry_df),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Pre-CP uses final legal freeze state only",
        "purpose": (
            "Stores fitted Monday-Sunday seasonal adjustments "
            "and their fit provenance."
        ),
        "path":
            str(WEEKLY_STL_TEMPLATE_FINAL_PATH),
    },
    {
        "artifact":
            "Weekly-STL validity intervals",
        "track_scope":
            "All modeling tracks",
        "grain": (
            "Modeling Track × Forecast Sequence × Daypart × "
            "legal application interval"
        ),
        "rows":
            len(
                weekly_stl_template_interval_registry_df
            ),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Final Pre-CP interval remains valid after freeze",
        "purpose": (
            "Identifies which fitted seasonal state is the "
            "latest legal one at each forecast origin."
        ),
        "path":
            str(WEEKLY_STL_INTERVAL_FINAL_PATH),
    },
    {
        "artifact":
            "Annual Fourier calendar",
        "track_scope":
            "Shared canonical calendar",
        "grain":
            "Canonical observation_sequence_id",
        "rows":
            len(annual_fourier_calendar_df),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Future-known and reusable",
        "purpose": (
            "Continuous annual calendar position "
            "encoded as sine/cosine predictors."
        ),
        "path":
            str(ANNUAL_FOURIER_FINAL_PATH),
    },
    {
        "artifact":
            "Pre-CP counterfactual calendar",
        "track_scope":
            TRACK_LABELS["pre_cp_reference"],
        "grain":
            "Recursive Origin × Horizon",
        "rows":
            len(
                pre_cp_counterfactual_calendar_df
            ),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Primary no-CP application calendar",
        "purpose": (
            "Defines post-freeze targets and future-known inputs "
            "without including actual post-CP outcomes."
        ),
        "path":
            str(PRE_CP_COUNTERFACTUAL_CALENDAR_PATH),
    },
    {
        "artifact":
            "Pre-CP frozen weekly state",
        "track_scope":
            TRACK_LABELS["pre_cp_reference"],
        "grain":
            "Forecast Sequence × Daypart",
        "rows":
            len(
                pre_cp_counterfactual_weekly_state_df
            ),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Frozen fitted seasonal state",
        "purpose": (
            "Carries the final legal January 4 weekly template—or "
            "zero for series assigned none—into counterfactual application."
        ),
        "path":
            str(
                PRE_CP_COUNTERFACTUAL_WEEKLY_STATE_PATH
            ),
    },
    {
        "artifact":
            "Pre-CP recursive feature contract",
        "track_scope":
            TRACK_LABELS["pre_cp_reference"],
        "grain":
            "Feature definition",
        "rows":
            len(
                pre_cp_recursive_feature_contract_df
            ),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Runtime recursion instructions",
        "purpose": (
            "Identifies outcome-history features that Chapter 5 "
            "must rebuild from earlier no-CP predictions."
        ),
        "path":
            str(
                PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH
            ),
    },
    {
        "artifact":
            "Final feature-contract QA",
        "track_scope":
            "All modeling tracks",
        "grain":
            "QA check",
        "rows":
            len(
                final_feature_contract_qa_df
            ),
        "contains_observed_mobility_history":
            False,
        "counterfactual_role":
            "Validation only",
        "purpose": (
            "Executable structural and leakage-safety "
            "handoff checks."
        ),
        "path":
            str(FINAL_FEATURE_QA_PATH),
    },
])


forecast_feature_manifest_df = (
    pd.DataFrame.from_records(
        manifest_records
    )
    .reset_index(drop=True)
)

if WRITE_FINAL_OUTPUTS:
    forecast_feature_manifest_df.to_parquet(
        FINAL_FEATURE_MANIFEST_PATH,
        index=False,
    )

display(forecast_feature_manifest_df)

,artifact,track_scope,grain,rows,contains_observed_mobility_history,counterfactual_role,purpose,path
0,Origin feature surface · Post-CP Primary,Post-CP Primary,Forecast Sequence × Origin observation_sequenc...,2497432,True,Ordinary forecasting history,"Current level, recent lags, comparable-Daypart...",pipeline_data/4.2.1.final_tables/origin_featur...
1,Origin feature surface · Full-History Regime-A...,Full-History Regime-Aware,Forecast Sequence × Origin observation_sequenc...,6231379,True,Ordinary forecasting history,"Current level, recent lags, comparable-Daypart...",pipeline_data/4.2.1.final_tables/origin_featur...
2,Origin feature surface · Pre-CP Reference,Pre-CP Reference,Forecast Sequence × Origin observation_sequenc...,3818325,True,Legal seed/history only,"Current level, recent lags, comparable-Daypart...",pipeline_data/4.2.1.final_tables/origin_featur...
3,Forecast target calendar context,All modeling tracks,Modeling Track × Origin × Horizon,35496,False,Ordinary scheduled forecasts,"Canonical target timing, horizon, and annual F...",pipeline_data/4.2.1.final_tables/forecast_targ...
4,Weekly-STL selection registry,All modeling tracks,Modeling Track × Forecast Sequence × Daypart,15990,False,Frozen selection policy,Records the fixed single_7 versus none decision.,pipeline_data/4.2.1.final_tables/weekly_stl_se...
5,Weekly-STL fitted template registry,All modeling tracks,Modeling Track × Forecast Sequence × Daypart ×...,146274,False,Pre-CP uses final legal freeze state only,Stores fitted Monday-Sunday seasonal adjustmen...,pipeline_data/4.2.1.final_tables/weekly_stl_te...
6,Weekly-STL validity intervals,All modeling tracks,Modeling Track × Forecast Sequence × Daypart ×...,146274,False,Final Pre-CP interval remains valid after freeze,Identifies which fitted seasonal state is the ...,pipeline_data/4.2.1.final_tables/weekly_stl_te...
7,Annual Fourier calendar,Shared canonical calendar,Canonical observation_sequence_id,5930,False,Future-known and reusable,Continuous annual calendar position encoded as...,pipeline_data/4.2.1.final_tables/annual_fourie...
8,Pre-CP counterfactual calendar,Pre-CP Reference,Recursive Origin × Horizon,6760,False,Primary no-CP application calendar,Defines post-freeze targets and future-known i...,pipeline_data/4.2.1.final_tables/pre_cp_counte...
9,Pre-CP frozen weekly state,Pre-CP Reference,Forecast Sequence × Daypart,5195,False,Frozen fitted seasonal state,Carries the final legal January 4 weekly templ...,pipeline_data/4.2.1.final_tables/pre_cp_counte...


**Findings.** The notebook exports 12 final artifacts containing everything needed for the next forecasting stage: the history-based feature tables for all three modeling approaches, the calendar information for each forecast target, the weekly seasonal patterns and their update history, the annual calendar features, the special Pre-CP no-congestion-pricing setup, and the final QA results. Keeping these pieces separate preserves a clear record of where each forecasting input comes from and allows the next notebook to train and evaluate models without rebuilding the feature logic developed here.

### Section Summary

Section 4.2.1.6 brings the forecasting features built throughout this notebook into one consistent, leakage-safe package.

For each forecasting approach, we now have a complete history-based feature table containing only information that would have been available when a forecast was made. We also built a separate target-calendar table describing **what date and Daypart each forecast is predicting** without including the mobility value we are trying to predict. Across all three modeling approaches, the final target table covers 35,496 scheduled forecasts, and the alignment checks confirm that every target date, Daypart, and forecast horizon matches the canonical sequence correctly. :contentReference[oaicite:0]{index=0}

Seasonality is handled through two separate feature streams. Annual Fourier features tell the model where a forecast falls within the year and are known entirely from the calendar. Weekly STL captures recurring Monday-through-Sunday mobility patterns only for series where that weekly pattern was strong and well-supported. For those series, each forecast uses the most recently fitted weekly pattern that was available at the time the forecast was made; series without a weekly STL feature receive a neutral adjustment of zero.

The Pre-CP Reference requires a different treatment after January 4, 2025. At that point, real mobility observations stop entering the hypothetical no-congestion-pricing forecast. The model begins with the last observed Pre-CP state and then moves forward one Daypart at a time, using each new prediction as part of the history for the next prediction. Calendar information remains available in advance, and the final weekly pattern estimated from Pre-CP data remains fixed. Actual post-CP mobility is kept completely outside that forecasting loop so it can later be compared with the hypothetical no-congestion-pricing path.

The final quality checks confirm that these rules have been implemented consistently. All 24 end-to-end checks pass: the expected forecasting series and rows are preserved, forecast targets align correctly with the calendar, seasonal features are complete, and the Pre-CP forecast cannot accidentally use mobility information that occurred after congestion pricing began. The notebook therefore finishes with a complete and clearly defined feature package ready for model training and evaluation.

---

## Notebook Conclusion

Notebook 4.2.1 constructed the first production-ready feature layer for the forecasting phase of the NYC Congestion Pricing Mobility project.

The feature set is intentionally focused. It combines recent mobility history, comparable prior periods, rolling averages and variability, selected weekly seasonal patterns, and calendar-based annual timing. Just as importantly, the notebook defines **when each piece of information is allowed to enter a forecast**. Historical mobility features stop at the forecast origin, while information about the target date and Daypart is limited to what can genuinely be known in advance.

This distinction is especially important for the Pre-CP Reference. Once congestion pricing begins, that model must continue forward without learning from the mobility that actually occurred afterward. Its future history is therefore built from its own earlier no-congestion-pricing predictions, while its calendar features remain known in advance and its final Pre-CP weekly seasonal pattern remains fixed. Actual post-CP mobility is preserved separately for later comparison rather than fed back into the hypothetical forecast.

The purpose of this notebook was not to determine which forecasting model or feature set will ultimately perform best. That comes next. Its purpose was to make sure the inputs are complete, consistent, reproducible, and protected against accidental use of future information.

With all 24 final quality checks passing and the forecasting artifacts exported, **Notebook 4.2.1 is complete**.

The next stage can now focus on what this notebook deliberately leaves unanswered: **which forecasting models work best, which features actually improve prediction, and how accurately the frozen Pre-CP model can estimate the mobility patterns that might have occurred without congestion pricing.**

### 4.2.1 Final Inventory of Files

This notebook writes the following final forecasting-feature assets to `pipeline_data/4.2.1.final_tables/`. Reusable intermediate/cache artifacts are not included in this inventory.

- **`origin_feature_surface__post_cp_primary.parquet`**  
  Origin-side forecasting feature surface for the **Post-CP Primary** track. Contains the target series' current mobility state, recent canonical-sequence history, comparable same-Daypart history, rolling summaries, and origin-calendar context at one row per Forecast Sequence × forecast origin.

- **`origin_feature_surface__full_history_regime_aware.parquet`**  
  Origin-side forecasting feature surface for the **Full-History Regime-Aware** track. Uses the same feature contract as the Post-CP surface while preserving the longer pre/post congestion-pricing history and regime-aware modeling scope.

- **`origin_feature_surface__pre_cp_reference.parquet`**  
  Observed-history feature surface for the **Pre-CP Reference** track. Contains only the legal historical state available through the January 4, 2025 freeze boundary and serves as the seed history for the later recursive no-CP simulation.

- **`forecast_target_calendar_context.parquet`**  
  Compact target-side calendar for the scheduled **h=1, h=2, and h=5** forecasts. Records each modeling track's forecast origin, horizon, target sequence position, target date/Daypart, and other deterministic target-calendar fields without carrying future mobility outcomes.

- **`forecast_feature_catalog.parquet`**  
  Feature-level catalog for the complete 4.2.1 candidate predictor library. Records each feature's family, origin/target side, modeling role, ordinary forecasting availability, Pre-CP post-freeze availability, track scope, and plain-English interpretation.

- **`weekly_stl_selection_registry.parquet`**  
  Frozen weekly-seasonality selection for each Modeling Track × Forecast Sequence × Daypart. Records whether that series qualified for the retained weekly STL representation or intentionally uses no fitted weekly adjustment.

- **`weekly_stl_template_registry.parquet`**  
  Legally fitted weekly-STL refresh states for qualifying series. Stores the fitted Monday-through-Sunday seasonal offsets together with refresh timing, training-history coverage, and fit provenance.

- **`weekly_stl_template_interval_registry.parquet`**  
  Validity-interval representation of the weekly-STL template registry. Defines the forecast-origin sequence range over which each fitted weekly template is the latest legally available seasonal state.

- **`annual_fourier_calendar.parquet`**  
  Deterministic annual-cycle calendar across the canonical sequence. Supplies the annual sine and cosine coordinates used to represent longer-run recurring calendar position without fitting future observations.

- **`pre_cp_counterfactual_calendar.parquet`**  
  Future calendar for applying the frozen Pre-CP model after January 4, 2025. Contains only counterfactual timing and deterministic future-known information needed to advance the no-CP simulation; it does not contain actual post-CP mobility history.

- **`pre_cp_counterfactual_weekly_state.parquet`**  
  Frozen January 4 weekly-seasonal state for each eligible Pre-CP Forecast Sequence × Daypart. Supplies the legally fitted weekly pattern that remains available throughout the later no-CP simulation.

- **`pre_cp_recursive_feature_contract.parquet`**  
  Runtime contract identifying the **17 outcome-history predictors** that cannot be read from actual post-CP mobility during the no-CP simulation and must instead be reconstructed from the evolving model-generated prediction history.

- **`final_feature_contract_qa.parquet`**  
  Final executable QA results for the 4.2.1 package. Validates feature grains, temporal and horizon alignment, track populations, seasonal-state legality, Pre-CP freeze behavior, recursive-feature policy, and other leakage-safety requirements.

- **`forecast_feature_manifest.parquet`**  
  File-level manifest describing the downstream forecasting artifacts, their modeling-track scope, grain, row counts, counterfactual role, purpose, and storage path. The manifest summarizes the primary data/runtime/QA handoff assets, while the feature catalog and manifest itself remain additional package metadata.


### 4.2.1 Feature Columns

The final 4.2.1 feature contract defines **37 candidate forecasting predictors**. These form a reusable feature library; later models may use different subsets depending on model family and forecasting experiment.

#### Outcome-History Features — 17 Columns

These features describe the forecast target's own mobility state through the forecast origin. They are observed-history predictors during ordinary forecasting and must be reconstructed recursively from model-generated no-CP history after the Pre-CP freeze.

##### Current Origin State — 1 Column

- `origin_value`

##### Recent Canonical Sequence History — 5 Columns

- `seq_lag_1`
- `seq_lag_2`
- `seq_lag_3`
- `seq_lag_4`
- `seq_delta_1`

##### Comparable Same-Daypart History — 11 Columns

- `same_daypart_lag_1d`
- `same_daypart_lag_2d`
- `same_daypart_lag_7d`
- `same_daypart_delta_1d`
- `same_daypart_roll_mean_7d`
- `same_daypart_roll_std_7d`
- `same_daypart_roll_obs_share_7d`
- `same_daypart_ewm_mean_7d`
- `same_daypart_roll_mean_28d`
- `same_daypart_roll_std_28d`
- `same_daypart_roll_obs_share_28d`

#### Deterministic Origin Calendar Context — 7 Columns

These describe the calendar state at the forecast origin and are known without observing future mobility.

- `origin_daypart`
- `origin_temporal_bucket`
- `origin_weekday_index`
- `origin_is_weekend`
- `origin_month`
- `origin_day_of_year`
- `origin_elapsed_days`

#### Deterministic Target Calendar Context — 7 Columns

These describe the future period being forecast. Because the target timestamp and calendar are known when a forecast is issued, they remain leakage-safe for every horizon.

- `horizon`
- `target_daypart`
- `target_temporal_bucket`
- `target_weekday_index`
- `target_is_weekend`
- `target_month`
- `target_day_of_year`

#### Deterministic Annual Cycle — 2 Columns

A single sine/cosine pair represents continuous annual calendar position without fitting future observations.

- `annual_fourier_sin`
- `annual_fourier_cos`

#### Policy-Regime Context — 2 Columns

These indicators are available to the ordinary **Full-History Regime-Aware** forecasting track so that it can explicitly represent the congestion-pricing regime boundary. They are not used when applying the frozen Pre-CP model to the post-January 4 no-CP counterfactual period.

- `origin_pre_post_cp`
- `target_pre_post_cp`

#### Training-Fitted Weekly Seasonal State — 2 Columns

These features provide the retained weekly seasonal representation. The selection between weekly STL and no fitted weekly adjustment is frozen from legal training history, while qualifying weekly templates may refresh using only history available through each forecast origin. For the Pre-CP counterfactual, the final legal January 4 state remains frozen.

- `weekly_stl_enabled`
- `weekly_stl_seasonal_offset`

#### Feature Count Summary

The complete 4.2.1 candidate predictor library therefore contains:

**17 outcome-history features + 16 deterministic calendar/Fourier features + 2 policy-regime features + 2 training-fitted weekly-seasonal features = 37 candidate predictors.**

The **17 outcome-history features** are the special recursive subset for the later no-CP simulation. Beginning after the January 4, 2025 Pre-CP freeze, these features must be rebuilt from the evolving synthetic prediction history rather than from actual post-CP mobility observations.

Identifiers, sequence-order fields, dates, Taxi Zone and metric metadata, modeling-track labels, cohort/eligibility flags, weekly-STL refresh provenance, validity-interval fields, and QA fields are retained where needed for alignment and validation but are **not counted among the 37 forecasting predictors**.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>